## Master's Research
# *Developing a neural network model that can estimate surface curvatures from function values*
### This research was conducted under Dr. Jacob Hauenstein in the Department of Computer Science at the University of Alabama in Huntsville. This work continues our undergraduate research, where we tried to develop a neural network model that could satisfactorily estimate surface curvatures using generated function value data and unsigned labels, but failed. To address the previous model's inadequate performance, we initially decided to use a deeper and narrower neural network architecture and trained it on a larger dataset. Although some of the experiments produced promising results, the performance varied significantly depending on the expressions used, making it difficult to properly compare different model architectures. We therefore introduced seeded expressions and began checking the model predictions directly in addition to examining the loss and error values.
### Further experimentation revealed that using both principal curvature values as outputs created another problem, particularly because the second curvature value for the univariate polynomial expressions used in our experiments was always zero. This could cause the models to learn an incorrect pattern from the data, resulting in predictions that were often negative, identical across many samples, or significantly different from the actual labels. As a result, we changed our procedure to estimate only the maximum-valued surface curvature and deferred estimating the minimum-valued curvature to a separate model at a later stage. We then experimented with several different neural network architectures and also investigated whether the scale and distribution of the input data were contributing to the poor performance.
### One important observation from these experiments was that the models performed substantially better when trained on expressions producing higher-valued curvatures. In particular, experiments using sine-wave surfaces and other expressions with higher curvature values produced much better results than experiments involving polynomial expressions with very small curvature values. Based on these observations, we formed the hypothesis that the small curvature values associated with some polynomial surfaces could be contributing to the poor behavior of the models. To investigate this further, we began training the models on combinations of different surface types containing both high- and low-valued curvatures. These experiments were also intended to make the training data more representative of real-world surfaces, which can contain different types of surface regions.
### Despite these changes, the fully connected architectures continued to produce inconsistent results across different types of functions. We therefore changed the type of model architecture and experimented with convolutional neural networks using transfer learning, first with VGG16 and later with ResNet50V2. However, although some experiments showed gradual or comparatively better results, neither CNN-based approach produced sufficiently reliable performance for our purpose. Finally, we designed an architecture with selective connectivity in the first hidden layer. This approach was inspired by the mathematical models previously developed by Dr. Hauenstein for surface-curvature estimation, which used partial derivatives. Although this architecture showed some ability to identify patterns in the data, it still failed to provide consistently accurate curvature estimates. Moreover, it was not clear whether the mask was properly being applied on the inputs. As my Master's program came to a close, we were ultimately unable to develop a model that consistently achieved satisfactory surface-curvature estimation performance; nevertheless, we detailed pathways to continue this research at the end of this notebook.

In [1]:
import sympy
import numpy as np
import tensorflow as tf
from tensorflow import keras
from random import uniform, seed
from sklearn.utils import shuffle
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

In [2]:
x = sympy.Symbol('x')
y = sympy.Symbol('y')

In [3]:
# Function for generating input sample and unsigned label for a particular point of interest

def data_generator(exp, center_x, center_y):
    l_sample = []
    
    for i in range(center_x-1, center_x+2):
        for j in range(center_y-1, center_y+2):
            value = exp.evalf(subs={x: i, y: j})
            l_sample.append(value)
            
    fx = exp.diff(x, 1)
    fy = exp.diff(y, 1)
    fxx = exp.diff(x, 2)
    fyy = exp.diff(y, 2)
    fxy = exp.diff(x, y, 1)
    
    v_fx = fx.evalf(subs={x: center_x, y: center_y})
    v_fy = fy.evalf(subs={x: center_x, y: center_y})
    v_fxx = fxx.evalf(subs={x: center_x, y: center_y})
    v_fyy = fyy.evalf(subs={x: center_x, y: center_y})
    v_fxy = fxy.evalf(subs={x: center_x, y: center_y})
    
    K = (v_fxx*v_fyy - v_fxy**2) / (1 + v_fx**2 + v_fy**2)**2
    H = (v_fxx + v_fyy + v_fxx*v_fy**2 + v_fyy*v_fx**2 - 2*v_fx*v_fy*v_fxy) / (2 * (1 + v_fx**2 + v_fy**2)**1.5)
    
    k1 = H + (H**2 - K)**0.5
    k2 = H - (H**2 - K)**0.5
    
    #*********************************************
    # Changes made to create unsigned labels
    
    u_k1 = abs(k1)
    u_k2 = abs(k2)
    
    if(u_k1 < u_k2):
        temp = u_k1
        u_k1 = u_k2
        u_k2 = temp
    
    #*********************************************
    
    l_label = []
    l_label.extend([u_k1, u_k2])
    
    return l_sample, l_label

In [4]:
# Function for generating a list of samples and labels for a range of points

def datalist_generator(exp, x_start, y_start, x_end, y_end):
    l_samples = []
    l_labels = []

    for i in range(x_start, x_end+1):
        for j in range(y_start, y_end+1):
            t_sample, t_label = data_generator(exp, i, j)
            l_samples.append(t_sample)
            l_labels.append(t_label)
        
    samples = np.array(l_samples, dtype = 'float64')
    labels = np.array(l_labels, dtype = 'float64')
    
    return samples, labels

In [3]:
# Function for generating expressions

def exp_generator(num, h_deg):
    
    if (num == 1):
        
        seed(num + h_deg)
        
        a = uniform(-5.0, 5.0)
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-5.0, 5.0)
        e = uniform(-5.0, 5.0)
        
        if(h_deg == 3):
            a = 0.0
        elif(h_deg == 2):
            a = 0.0
            b = 0.0
        
        f = a*x**4 + b*x**3 + c*x**2 + d*x + e
        
        return f
    
    elif (num == 2):
        
        seed(num + h_deg)
        
        a = uniform(-5.0, 5.0)
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-5.0, 5.0)
        e = uniform(-5.0, 5.0)
        f = uniform(-5.0, 5.0)
        g = uniform(-5.0, 5.0)
        h = uniform(-5.0, 5.0)
        i = uniform(-5.0, 5.0)
        j = uniform(-5.0, 5.0)
        k = uniform(-5.0, 5.0)
        l = uniform(-5.0, 5.0)
        m = uniform(-5.0, 5.0)
        n = uniform(-5.0, 5.0)
        o = uniform(-5.0, 5.0)
        
        if(h_deg == 3):
            a = 0.0
            b = 0.0
            c = 0.0
            d = 0.0
            e = 0.0
        elif(h_deg == 2):
            a = 0.0
            b = 0.0
            c = 0.0
            d = 0.0
            e = 0.0
            f = 0.0
            g = 0.0
            h = 0.0
            i = 0.0
        
        f = a*x**4 + b*y**4 + c*x**3*y + d*x*y**3 + e*x**2*y**2 + f*x**3 + g*y**3 + h*x**2*y + i*x*y**2 + j*x**2 + k*y**2 + l*x*y + m*x + n*y + o
        
        return f
    
    elif (num == 3):
        
        seed(num)
        
        a = uniform(-5.0, 5.0)
        
        while(a == 0):
            a = uniform(-5.0, 5.0)
        
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-5.0, 5.0)
        
        f = a*sympy.sin(b*x-c) + d
        
        return f
    
    elif (num == 4):
        
        seed(num)
        
        a = uniform(-5.0, 5.0)
        
        while(a == 0):
            a = uniform(-5.0, 5.0)
        
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-5.0, 5.0)
        
        f = a*sympy.cos(b*x-c) + d
        
        return f
    
    else:
        f = 0
        return f

### Continuing from where we left off our undergraduate research, we had a model that wasn't working well for our purpose. So, we decided to use a deeper and narrower network and check our results. (Deeper and narrower models have been found to work well compared to shallower and wider networks).

## Architecture Changed - Deeper (4 more layers), Narrower (Highest no of nodes - 24, compared to 36)
### Larger dataset (No. of samples (128 x 128)) <br>"Unsigned" Labels<br>Training & Testing from scratch

In [6]:
funct = exp_generator(1, 2)

In [7]:
funct

2.51698522080082*x**2 + 2.1306919240195*x + 1.75472294006063

In [8]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [9]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [10]:
model = Sequential([
    Dense(units = 3, input_shape = (9,), activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 18, activation = 'relu'),
    Dense(units = 21, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 21, activation = 'relu'),
    Dense(units = 18, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 3, activation = 'relu'),
    Dense(units = 2)
])

In [11]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [12]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 6s - loss: 21777.9004 - mean_absolute_error: 49.2652 - val_loss: 0.0684 - val_mean_absolute_error: 0.1694 - 6s/epoch - 14ms/step
Epoch 2/30
417/417 - 2s - loss: 0.0030 - mean_absolute_error: 0.0215 - val_loss: 1.1750e-04 - val_mean_absolute_error: 0.0101 - 2s/epoch - 4ms/step
Epoch 3/30
417/417 - 2s - loss: 1.0926e-04 - mean_absolute_error: 0.0099 - val_loss: 1.0605e-04 - val_mean_absolute_error: 0.0098 - 2s/epoch - 4ms/step
Epoch 4/30
417/417 - 2s - loss: 1.0482e-04 - mean_absolute_error: 0.0098 - val_loss: 1.0406e-04 - val_mean_absolute_error: 0.0098 - 2s/epoch - 4ms/step
Epoch 5/30
417/417 - 2s - loss: 1.0364e-04 - mean_absolute_error: 0.0098 - val_loss: 1.0325e-04 - val_mean_absolute_error: 0.0098 - 2s/epoch - 4ms/step
Epoch 6/30
417/417 - 2s - loss: 1.0297e-04 - mean_absolute_error: 0.0098 - val_loss: 1.0243e-04 - val_mean_absolute_error: 0.0098 - 2s/epoch - 4ms/step
Epoch 7/30
417/417 - 2s - loss: 1.0224e-04 - mean_absolute_error: 0.0098 - val_loss: 1.0169e-0

In [13]:
model.save('Research_main.h5')

In [14]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 6.2202e-06 - mean_absolute_error: 0.0019 - 579ms/epoch - 2ms/step


In [15]:
funct = exp_generator(1, 3)
funct

-2.35739253193974*x**3 + 0.805085069833922*x**2 + 2.78166708758322*x - 3.35235934813067

In [16]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [17]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [18]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 1s - loss: 8688610.0000 - mean_absolute_error: 189.9773 - val_loss: 0.0014 - val_mean_absolute_error: 0.0117 - 1s/epoch - 3ms/step
Epoch 2/30
417/417 - 1s - loss: 0.0012 - mean_absolute_error: 0.0113 - val_loss: 0.0014 - val_mean_absolute_error: 0.0117 - 1s/epoch - 3ms/step
Epoch 3/30
417/417 - 1s - loss: 0.0012 - mean_absolute_error: 0.0113 - val_loss: 0.0014 - val_mean_absolute_error: 0.0117 - 1s/epoch - 3ms/step
Epoch 4/30
417/417 - 1s - loss: 0.0012 - mean_absolute_error: 0.0113 - val_loss: 0.0014 - val_mean_absolute_error: 0.0117 - 1s/epoch - 3ms/step
Epoch 5/30
417/417 - 2s - loss: 0.0012 - mean_absolute_error: 0.0113 - val_loss: 0.0014 - val_mean_absolute_error: 0.0117 - 2s/epoch - 4ms/step
Epoch 6/30
417/417 - 1s - loss: 0.0012 - mean_absolute_error: 0.0113 - val_loss: 0.0014 - val_mean_absolute_error: 0.0117 - 1s/epoch - 4ms/step
Epoch 7/30
417/417 - 2s - loss: 0.0012 - mean_absolute_error: 0.0113 - val_loss: 0.0014 - val_mean_absolute_error: 0.0117 - 2s/e

In [19]:
model.save('Research_main.h5')

##### The loss was stuck at a particular value but the results weren't too worse.

In [20]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 0.0010 - mean_absolute_error: 0.0104 - 435ms/epoch - 2ms/step


In [21]:
funct = exp_generator(1, 4)
funct

4.14419406116664*x**4 + 1.83068032222987*x**3 - 0.522892874886124*x**2 + 0.84518450969701*x - 0.490485027835526

In [22]:
s, l = datalist_generator(funct, 5, 5, 130, 130)

In [23]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [25]:
for i in train_labels:
    print(i)

#### This code is to check the labels for the training set. Output deleted to save memory.

In [24]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 1s - loss: 7.4236e-05 - mean_absolute_error: 0.0085 - val_loss: 7.2329e-05 - val_mean_absolute_error: 0.0084 - 1s/epoch - 3ms/step
Epoch 2/30
417/417 - 1s - loss: 7.0257e-05 - mean_absolute_error: 0.0083 - val_loss: 6.8082e-05 - val_mean_absolute_error: 0.0081 - 1s/epoch - 3ms/step
Epoch 3/30
417/417 - 1s - loss: 6.5743e-05 - mean_absolute_error: 0.0080 - val_loss: 6.3301e-05 - val_mean_absolute_error: 0.0078 - 1s/epoch - 3ms/step
Epoch 4/30
417/417 - 1s - loss: 6.0707e-05 - mean_absolute_error: 0.0076 - val_loss: 5.8018e-05 - val_mean_absolute_error: 0.0074 - 1s/epoch - 3ms/step
Epoch 5/30
417/417 - 1s - loss: 5.5203e-05 - mean_absolute_error: 0.0072 - val_loss: 5.2306e-05 - val_mean_absolute_error: 0.0070 - 1s/epoch - 4ms/step
Epoch 6/30
417/417 - 1s - loss: 4.9323e-05 - mean_absolute_error: 0.0068 - val_loss: 4.6280e-05 - val_mean_absolute_error: 0.0065 - 1s/epoch - 3ms/step
Epoch 7/30
417/417 - 1s - loss: 4.3203e-05 - mean_absolute_error: 0.0063 - val_loss: 4.0

In [26]:
model.save('Research_main.h5')

In [27]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 6.8340e-17 - mean_absolute_error: 1.2504e-09 - 413ms/epoch - 2ms/step


In [28]:
funct = exp_generator(2, 2)
funct

-4.13570930279608*x**2 + 1.03221907478977*x*y - 1.13964657364066*x + 3.58151102898716*y**2 - 0.00427432139239592*y - 4.52590120671638

In [29]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [30]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [31]:
for i in train_labels:
    print(i)

In [32]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 1s - loss: 6.2808e-05 - mean_absolute_error: 0.0029 - val_loss: 5.3030e-05 - val_mean_absolute_error: 0.0028 - 1s/epoch - 3ms/step
Epoch 2/30
417/417 - 1s - loss: 6.1579e-05 - mean_absolute_error: 0.0028 - val_loss: 5.3278e-05 - val_mean_absolute_error: 0.0028 - 1s/epoch - 3ms/step
Epoch 3/30
417/417 - 1s - loss: 6.1575e-05 - mean_absolute_error: 0.0028 - val_loss: 5.2711e-05 - val_mean_absolute_error: 0.0029 - 1s/epoch - 3ms/step
Epoch 4/30
417/417 - 2s - loss: 6.1524e-05 - mean_absolute_error: 0.0028 - val_loss: 5.2702e-05 - val_mean_absolute_error: 0.0029 - 2s/epoch - 4ms/step
Epoch 5/30
417/417 - 1s - loss: 6.1563e-05 - mean_absolute_error: 0.0028 - val_loss: 5.2893e-05 - val_mean_absolute_error: 0.0028 - 925ms/epoch - 2ms/step
Epoch 6/30
417/417 - 1s - loss: 6.1529e-05 - mean_absolute_error: 0.0028 - val_loss: 5.2814e-05 - val_mean_absolute_error: 0.0029 - 1s/epoch - 3ms/step
Epoch 7/30
417/417 - 1s - loss: 6.1552e-05 - mean_absolute_error: 0.0028 - val_loss: 

In [33]:
model.save('Research_main.h5')

In [34]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 5.7319e-05 - mean_absolute_error: 0.0028 - 379ms/epoch - 2ms/step


In [6]:
funct = exp_generator(2, 3)
funct

1.21359755725873*x**3 + 4.66473450769645*x**2*y + 0.914979777465742*x**2 + 0.969489772615303*x*y**2 - 3.46564436182193*x*y - 3.66212422398862*x - 2.86035542097839*y**3 + 2.03936641334441*y**2 + 1.30171485258465*y + 1.15451529700597

In [7]:
s, l = datalist_generator(funct, 15, 15, 140, 140)

In [8]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [9]:
for i in train_labels:
    print(i)

In [10]:
from tensorflow.keras.models import load_model
model = load_model('Research_main.h5')

In [11]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 3s - loss: 2.0174e-05 - mean_absolute_error: 0.0021 - val_loss: 2.0422e-05 - val_mean_absolute_error: 0.0021 - 3s/epoch - 6ms/step
Epoch 2/30
417/417 - 1s - loss: 2.0107e-05 - mean_absolute_error: 0.0021 - val_loss: 2.0401e-05 - val_mean_absolute_error: 0.0022 - 1s/epoch - 3ms/step
Epoch 3/30
417/417 - 1s - loss: 2.0108e-05 - mean_absolute_error: 0.0021 - val_loss: 2.0457e-05 - val_mean_absolute_error: 0.0022 - 1s/epoch - 3ms/step
Epoch 4/30
417/417 - 1s - loss: 2.0116e-05 - mean_absolute_error: 0.0021 - val_loss: 2.0412e-05 - val_mean_absolute_error: 0.0022 - 1s/epoch - 3ms/step
Epoch 5/30
417/417 - 1s - loss: 2.0123e-05 - mean_absolute_error: 0.0022 - val_loss: 2.0396e-05 - val_mean_absolute_error: 0.0022 - 1s/epoch - 3ms/step
Epoch 6/30
417/417 - 2s - loss: 2.0122e-05 - mean_absolute_error: 0.0022 - val_loss: 2.0400e-05 - val_mean_absolute_error: 0.0022 - 2s/epoch - 4ms/step
Epoch 7/30
417/417 - 1s - loss: 2.0121e-05 - mean_absolute_error: 0.0021 - val_loss: 2.0

In [12]:
model.save('Research_main.h5')

In [13]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 2.2711e-05 - mean_absolute_error: 0.0022 - 452ms/epoch - 2ms/step


In [6]:
funct = exp_generator(2, 4)
funct

-3.32688813355041*x**4 + 0.0441903371778372*x**3*y + 0.803906419321059*x**3 - 2.71168460449816*x**2*y**2 + 0.868856678979647*x**2*y - 1.64298433173466*x**2 - 3.56665582096703*x*y**3 + 4.74144668586045*x*y**2 - 0.275525417807349*x*y - 3.161013788055*x - 3.61628383093444*y**4 + 0.895850965577887*y**3 - 1.07104002071268*y**2 - 4.01327279164397*y + 4.81765708253643

In [7]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [8]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [10]:
from tensorflow.keras.models import load_model
model = load_model('Research_main.h5')

In [11]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 4s - loss: 1.5515e-04 - mean_absolute_error: 0.0039 - val_loss: 1.8420e-04 - val_mean_absolute_error: 0.0039 - 4s/epoch - 10ms/step
Epoch 2/30
417/417 - 2s - loss: 1.5494e-04 - mean_absolute_error: 0.0038 - val_loss: 1.8411e-04 - val_mean_absolute_error: 0.0040 - 2s/epoch - 4ms/step
Epoch 3/30
417/417 - 2s - loss: 1.5485e-04 - mean_absolute_error: 0.0039 - val_loss: 1.8439e-04 - val_mean_absolute_error: 0.0039 - 2s/epoch - 4ms/step
Epoch 4/30
417/417 - 2s - loss: 1.5494e-04 - mean_absolute_error: 0.0038 - val_loss: 1.8416e-04 - val_mean_absolute_error: 0.0039 - 2s/epoch - 4ms/step
Epoch 5/30
417/417 - 2s - loss: 1.5490e-04 - mean_absolute_error: 0.0039 - val_loss: 1.8426e-04 - val_mean_absolute_error: 0.0039 - 2s/epoch - 4ms/step
Epoch 6/30
417/417 - 2s - loss: 1.5490e-04 - mean_absolute_error: 0.0039 - val_loss: 1.8415e-04 - val_mean_absolute_error: 0.0039 - 2s/epoch - 4ms/step
Epoch 7/30
417/417 - 2s - loss: 1.5485e-04 - mean_absolute_error: 0.0039 - val_loss: 1.

In [12]:
model.save('Research_main.h5')

In [13]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 2.6766e-04 - mean_absolute_error: 0.0039 - 612ms/epoch - 3ms/step


In [14]:
funct = exp_generator(3, 1)
funct

1.95890066062232 - 0.143647637621346*sin(3.01094098472383*x + 1.41631003937393)

In [15]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [16]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [18]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 2s - loss: 0.3188 - mean_absolute_error: 0.3459 - val_loss: 0.2919 - val_mean_absolute_error: 0.3290 - 2s/epoch - 4ms/step
Epoch 2/30
417/417 - 2s - loss: 0.2710 - mean_absolute_error: 0.3143 - val_loss: 0.2565 - val_mean_absolute_error: 0.3061 - 2s/epoch - 4ms/step
Epoch 3/30
417/417 - 2s - loss: 0.2411 - mean_absolute_error: 0.2949 - val_loss: 0.2301 - val_mean_absolute_error: 0.2881 - 2s/epoch - 4ms/step
Epoch 4/30
417/417 - 2s - loss: 0.2174 - mean_absolute_error: 0.2785 - val_loss: 0.2081 - val_mean_absolute_error: 0.2724 - 2s/epoch - 4ms/step
Epoch 5/30
417/417 - 2s - loss: 0.1973 - mean_absolute_error: 0.2651 - val_loss: 0.1892 - val_mean_absolute_error: 0.2601 - 2s/epoch - 4ms/step
Epoch 6/30
417/417 - 2s - loss: 0.1800 - mean_absolute_error: 0.2537 - val_loss: 0.1728 - val_mean_absolute_error: 0.2487 - 2s/epoch - 4ms/step
Epoch 7/30
417/417 - 2s - loss: 0.1647 - mean_absolute_error: 0.2429 - val_loss: 0.1582 - val_mean_absolute_error: 0.2377 - 2s/epoch - 4

##### The results were quite high and got stuck to singular values towards the latest epochs. So, we ran the training for a few more epochs just to check whether the results improved.

In [19]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 15, shuffle = True, verbose = 2)

Epoch 1/15
417/417 - 2s - loss: 0.0831 - mean_absolute_error: 0.1786 - val_loss: 0.0808 - val_mean_absolute_error: 0.1766 - 2s/epoch - 4ms/step
Epoch 2/15
417/417 - 2s - loss: 0.0831 - mean_absolute_error: 0.1787 - val_loss: 0.0808 - val_mean_absolute_error: 0.1765 - 2s/epoch - 4ms/step
Epoch 3/15
417/417 - 2s - loss: 0.0831 - mean_absolute_error: 0.1786 - val_loss: 0.0808 - val_mean_absolute_error: 0.1766 - 2s/epoch - 4ms/step
Epoch 4/15
417/417 - 2s - loss: 0.0831 - mean_absolute_error: 0.1786 - val_loss: 0.0808 - val_mean_absolute_error: 0.1766 - 2s/epoch - 4ms/step
Epoch 5/15
417/417 - 2s - loss: 0.0831 - mean_absolute_error: 0.1786 - val_loss: 0.0808 - val_mean_absolute_error: 0.1766 - 2s/epoch - 4ms/step
Epoch 6/15
417/417 - 2s - loss: 0.0831 - mean_absolute_error: 0.1787 - val_loss: 0.0808 - val_mean_absolute_error: 0.1765 - 2s/epoch - 4ms/step
Epoch 7/15
417/417 - 2s - loss: 0.0831 - mean_absolute_error: 0.1786 - val_loss: 0.0808 - val_mean_absolute_error: 0.1765 - 2s/epoch - 4

In [20]:
model.save('Research_main.h5')

##### The results were the same.

In [21]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 0.0828 - mean_absolute_error: 0.1781 - 603ms/epoch - 3ms/step


In [22]:
funct = exp_generator(4, 1)
funct

2.48453270754651 - 3.26175032812469*cos(0.172772350308447*x - 4.4127480838843)

In [23]:
s, l = datalist_generator(funct, 11, 11, 136, 136)

In [24]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [26]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 2s - loss: 0.2144 - mean_absolute_error: 0.3267 - val_loss: 0.1875 - val_mean_absolute_error: 0.3058 - 2s/epoch - 4ms/step
Epoch 2/30
417/417 - 2s - loss: 0.1700 - mean_absolute_error: 0.2910 - val_loss: 0.1545 - val_mean_absolute_error: 0.2775 - 2s/epoch - 4ms/step
Epoch 3/30
417/417 - 2s - loss: 0.1416 - mean_absolute_error: 0.2656 - val_loss: 0.1297 - val_mean_absolute_error: 0.2542 - 2s/epoch - 4ms/step
Epoch 4/30
417/417 - 2s - loss: 0.1193 - mean_absolute_error: 0.2436 - val_loss: 0.1094 - val_mean_absolute_error: 0.2334 - 2s/epoch - 4ms/step
Epoch 5/30
417/417 - 2s - loss: 0.1005 - mean_absolute_error: 0.2235 - val_loss: 0.0920 - val_mean_absolute_error: 0.2139 - 2s/epoch - 4ms/step
Epoch 6/30
417/417 - 2s - loss: 0.0842 - mean_absolute_error: 0.2046 - val_loss: 0.0769 - val_mean_absolute_error: 0.1955 - 2s/epoch - 4ms/step
Epoch 7/30
417/417 - 2s - loss: 0.0701 - mean_absolute_error: 0.1865 - val_loss: 0.0636 - val_mean_absolute_error: 0.1777 - 2s/epoch - 4

In [27]:
model.save('Research_main.h5')

# Deleted, because eventually it was discovered that this architecture fails to deliver a good performance 
# in terms of our purpose.

##### The results for the cosine expression was comparatively better compared to those for the similar sine expression. However, they weren't significantly better.

In [28]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 4.9170e-04 - mean_absolute_error: 0.0139 - 601ms/epoch - 3ms/step


### Looking at the loss and error for different expressions using this particular model, we can conclude that the model is working quite well for our purpose. However, in order to achieve a better accuracy, we decided to work with a slightly different architecture and compare the results of the two models to finally decide on a particular architecture best suited for our purpose.

## Architecture Changed - Further Deeper(16 layers in total, compared to 12), A bit wider (36 nodes highest among the layers)
### No. of samples (128 x 128) <br>"Unsigned" Labels<br>Training & Testing from scratch

In [6]:
funct = exp_generator(1, 2)
funct

3.16607696992371*x**2 - 2.19036159174715*x + 2.80766561086486

In [7]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [8]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [10]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 2)
])

In [11]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [12]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 3s - loss: 2.5078 - mean_absolute_error: 0.2326 - val_loss: 3.1705e-05 - val_mean_absolute_error: 0.0026 - 3s/epoch - 7ms/step
Epoch 2/30
417/417 - 1s - loss: 3.1790e-05 - mean_absolute_error: 0.0025 - val_loss: 3.0048e-05 - val_mean_absolute_error: 0.0024 - 988ms/epoch - 2ms/step
Epoch 3/30
417/417 - 1s - loss: 3.0942e-05 - mean_absolute_error: 0.0024 - val_loss: 2.9240e-05 - val_mean_absolute_error: 0.0023 - 809ms/epoch - 2ms/step
Epoch 4/30
417/417 - 1s - loss: 3.0065e-05 - mean_absolute_error: 0.0022 - val_loss: 2.8324e-05 - val_mean_absolute_error: 0.0021 - 666ms/epoch - 2ms/step
Epoch 5/30
417/417 - 1s - loss: 2.9116e-05 - mean_absolute_error: 0.0019 - val_loss: 2.7369e-05 - val_mean_absolute_error: 0.0018 - 767ms/epoch - 2ms/step
Epoch 6/30
417/417 - 1s - loss: 2.8168e-05 - mean_absolute_error: 0.0017 - val_loss: 2.6457e-05 - val_mean_absolute_error: 0.0015 - 734ms/epoch - 2ms/step
Epoch 7/30
417/417 - 1s - loss: 2.7293e-05 - mean_absolute_error: 0.0014 - va

##### Although, the error is better, loss is higher than the one with the previous architecture, for this type of expression. The reason behind this could be the fact that we had 2 different expressions for the 2 experiments as we didn't seed the random coefficients of the expressions when generating them.

In [13]:
model.save('Research_main2.h5')

In [14]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 3.0084e-05 - mean_absolute_error: 8.7927e-04 - 346ms/epoch - 1ms/step


In [6]:
funct = exp_generator(1, 3)
funct

-3.51290325454141*x**3 + 1.1656637817496*x**2 + 4.80960180106871*x + 1.22653521430332

In [7]:
s, l = datalist_generator(funct, 11, 11, 136, 136)

In [8]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [10]:
from tensorflow.keras.models import load_model
model = load_model('Research_main2.h5')

In [11]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 3s - loss: 3861.1851 - mean_absolute_error: 4.5564 - val_loss: 3.7214e-06 - val_mean_absolute_error: 0.0014 - 3s/epoch - 7ms/step
Epoch 2/30
417/417 - 1s - loss: 3.7124e-06 - mean_absolute_error: 0.0014 - val_loss: 3.7027e-06 - val_mean_absolute_error: 0.0014 - 1s/epoch - 3ms/step
Epoch 3/30
417/417 - 2s - loss: 3.6917e-06 - mean_absolute_error: 0.0014 - val_loss: 3.6798e-06 - val_mean_absolute_error: 0.0014 - 2s/epoch - 5ms/step
Epoch 4/30
417/417 - 1s - loss: 3.6663e-06 - mean_absolute_error: 0.0014 - val_loss: 3.6518e-06 - val_mean_absolute_error: 0.0014 - 1s/epoch - 2ms/step
Epoch 5/30
417/417 - 2s - loss: 3.6354e-06 - mean_absolute_error: 0.0014 - val_loss: 3.6176e-06 - val_mean_absolute_error: 0.0014 - 2s/epoch - 4ms/step
Epoch 6/30
417/417 - 2s - loss: 3.5976e-06 - mean_absolute_error: 0.0014 - val_loss: 3.5760e-06 - val_mean_absolute_error: 0.0014 - 2s/epoch - 4ms/step
Epoch 7/30
417/417 - 2s - loss: 3.5515e-06 - mean_absolute_error: 0.0014 - val_loss: 3.52

##### Results are a lot better in comparison.

In [12]:
model.save('Research_main2.h5')

In [13]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 3.5377e-10 - mean_absolute_error: 1.5027e-05 - 306ms/epoch - 1ms/step


In [6]:
funct = exp_generator(1, 4)
funct

4.81320087413642*x**4 - 3.93729992334191*x**3 + 1.02017726050588*x**2 - 1.95872640757447*x + 3.17650105420917

In [7]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [8]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [10]:
from tensorflow.keras.models import load_model
model = load_model('Research_main2.h5')

In [11]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 2s - loss: 6169.1558 - mean_absolute_error: 1.6740 - val_loss: 5.7598e-05 - val_mean_absolute_error: 0.0055 - 2s/epoch - 5ms/step
Epoch 2/30
417/417 - 1s - loss: 5.1965e-05 - mean_absolute_error: 0.0055 - val_loss: 5.7411e-05 - val_mean_absolute_error: 0.0055 - 729ms/epoch - 2ms/step
Epoch 3/30
417/417 - 1s - loss: 5.1761e-05 - mean_absolute_error: 0.0054 - val_loss: 5.7184e-05 - val_mean_absolute_error: 0.0055 - 730ms/epoch - 2ms/step
Epoch 4/30
417/417 - 1s - loss: 5.1511e-05 - mean_absolute_error: 0.0054 - val_loss: 5.6904e-05 - val_mean_absolute_error: 0.0055 - 735ms/epoch - 2ms/step
Epoch 5/30
417/417 - 1s - loss: 5.1207e-05 - mean_absolute_error: 0.0054 - val_loss: 5.6565e-05 - val_mean_absolute_error: 0.0054 - 861ms/epoch - 2ms/step
Epoch 6/30
417/417 - 1s - loss: 5.0836e-05 - mean_absolute_error: 0.0054 - val_loss: 5.6152e-05 - val_mean_absolute_error: 0.0054 - 894ms/epoch - 2ms/step
Epoch 7/30
417/417 - 1s - loss: 5.0386e-05 - mean_absolute_error: 0.0053 -

##### Results are worse than the ones with the previous architecture, for the same type of expression.

In [14]:
model.save('Research_main2.h5')

In [13]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 3.5656e-05 - mean_absolute_error: 6.8527e-04 - 279ms/epoch - 1ms/step


In [17]:
funct = exp_generator(2, 2)
funct

-2.55097188326331*x**2 + 4.16711889238534*x*y - 0.318283999947169*x - 3.20166730635742*y**2 + 4.75180620629512*y - 1.62907172696319

In [18]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [19]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [21]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 2s - loss: 0.0013 - mean_absolute_error: 0.0096 - val_loss: 7.9236e-04 - val_mean_absolute_error: 0.0095 - 2s/epoch - 4ms/step
Epoch 2/30
417/417 - 2s - loss: 0.0012 - mean_absolute_error: 0.0101 - val_loss: 7.7343e-04 - val_mean_absolute_error: 0.0103 - 2s/epoch - 4ms/step
Epoch 3/30
417/417 - 2s - loss: 0.0012 - mean_absolute_error: 0.0106 - val_loss: 7.7287e-04 - val_mean_absolute_error: 0.0105 - 2s/epoch - 4ms/step
Epoch 4/30
417/417 - 2s - loss: 0.0012 - mean_absolute_error: 0.0107 - val_loss: 7.7292e-04 - val_mean_absolute_error: 0.0104 - 2s/epoch - 4ms/step
Epoch 5/30
417/417 - 2s - loss: 0.0012 - mean_absolute_error: 0.0106 - val_loss: 7.7282e-04 - val_mean_absolute_error: 0.0105 - 2s/epoch - 4ms/step
Epoch 6/30
417/417 - 2s - loss: 0.0012 - mean_absolute_error: 0.0106 - val_loss: 7.7303e-04 - val_mean_absolute_error: 0.0107 - 2s/epoch - 4ms/step
Epoch 7/30
417/417 - 2s - loss: 0.0012 - mean_absolute_error: 0.0107 - val_loss: 7.7335e-04 - val_mean_absolute_

##### Results are worse than the ones with the previous architecture, for this type of expression too.<br>The change of coefficients in the expressions could be the reason behind this, since we didn't use seeded expressions for these experiments.

In [22]:
model.save('Research_main2.h5')

In [23]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 0.0016 - mean_absolute_error: 0.0107 - 573ms/epoch - 2ms/step


In [24]:
funct = exp_generator(2, 3)
funct

3.27766338844073*x**3 - 4.49368070149602*x**2*y - 1.13337742891189*x**2 - 2.19948328698718*x*y**2 - 0.836593895248027*x*y + 4.72896708275793*x - 2.87835857483956*y**3 - 1.85977415547259*y**2 - 4.22595000955948*y + 3.43836707680716

In [25]:
s, l = datalist_generator(funct, 5, 5, 130, 130)

In [26]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [28]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 2s - loss: 4.0993e-05 - mean_absolute_error: 0.0033 - val_loss: 2.4703e-05 - val_mean_absolute_error: 0.0020 - 2s/epoch - 4ms/step
Epoch 2/30
417/417 - 2s - loss: 2.2729e-05 - mean_absolute_error: 0.0019 - val_loss: 2.4702e-05 - val_mean_absolute_error: 0.0020 - 2s/epoch - 4ms/step
Epoch 3/30
417/417 - 2s - loss: 2.2735e-05 - mean_absolute_error: 0.0020 - val_loss: 2.4710e-05 - val_mean_absolute_error: 0.0020 - 2s/epoch - 4ms/step
Epoch 4/30
417/417 - 2s - loss: 2.2731e-05 - mean_absolute_error: 0.0019 - val_loss: 2.4708e-05 - val_mean_absolute_error: 0.0020 - 2s/epoch - 4ms/step
Epoch 5/30
417/417 - 2s - loss: 2.2726e-05 - mean_absolute_error: 0.0019 - val_loss: 2.4703e-05 - val_mean_absolute_error: 0.0020 - 2s/epoch - 4ms/step
Epoch 6/30
417/417 - 2s - loss: 2.2726e-05 - mean_absolute_error: 0.0020 - val_loss: 2.4721e-05 - val_mean_absolute_error: 0.0020 - 2s/epoch - 4ms/step
Epoch 7/30
417/417 - 2s - loss: 2.2735e-05 - mean_absolute_error: 0.0019 - val_loss: 2.4

##### Results are similar in comparison.

In [29]:
model.save('Research_main2.h5')

In [30]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 2.3979e-05 - mean_absolute_error: 0.0020 - 636ms/epoch - 3ms/step


### The change in architecture didn't significantly improve the results. However, the performances of the 2 different architectures aren't properly comparable because of the difference in the expressions used.
### Next, we decided to see the performance of a much narrower architecture, just to gauge the type of architecture that works best for our purpose, but this time we use seeded expressions and also check the predictions by the model on the test data, to properly understand the performance of the said model.

## Architecture Changed - Much Narrower (18 nodes highest among the layers), Depth is the same as last time (16 layers in total)
### No. of samples (128 x 128) <br>"Unsigned" Labels<br>Training & Testing from scratch<br>First seeded expressions used & predictions checked

In [6]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [7]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [8]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [10]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 10, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 14, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 18, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 14, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 10, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 2)
])

In [11]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [12]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 6s - loss: 0.0463 - mean_absolute_error: 0.0982 - val_loss: 0.0024 - val_mean_absolute_error: 0.0319 - 6s/epoch - 14ms/step
Epoch 2/30
417/417 - 2s - loss: 0.0017 - mean_absolute_error: 0.0263 - val_loss: 0.0012 - val_mean_absolute_error: 0.0209 - 2s/epoch - 4ms/step
Epoch 3/30
417/417 - 2s - loss: 9.5178e-04 - mean_absolute_error: 0.0159 - val_loss: 7.2786e-04 - val_mean_absolute_error: 0.0107 - 2s/epoch - 4ms/step
Epoch 4/30
417/417 - 2s - loss: 6.3277e-04 - mean_absolute_error: 0.0080 - val_loss: 5.5663e-04 - val_mean_absolute_error: 0.0064 - 2s/epoch - 4ms/step
Epoch 5/30
417/417 - 2s - loss: 5.3652e-04 - mean_absolute_error: 0.0056 - val_loss: 5.0453e-04 - val_mean_absolute_error: 0.0049 - 2s/epoch - 4ms/step
Epoch 6/30
417/417 - 2s - loss: 5.0011e-04 - mean_absolute_error: 0.0045 - val_loss: 4.7941e-04 - val_mean_absolute_error: 0.0039 - 2s/epoch - 4ms/step
Epoch 7/30
417/417 - 2s - loss: 4.7921e-04 - mean_absolute_error: 0.0035 - val_loss: 4.6250e-04 - val_m

##### Error is slightly lower than that of initial architecture, but loss is higher, which could be the result of using a different expression for the data.

In [13]:
model.save('Research_main.h5')

In [14]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 5.7961e-05 - mean_absolute_error: 0.0011 - 459ms/epoch - 2ms/step


In [15]:
test_labels.shape

(4763, 2)

In [16]:
for i in test_labels:
    print(i)

[1.91186985e-06 0.00000000e+00]
[1.49609873e-07 0.00000000e+00]
[1.78332448e-06 0.00000000e+00]
[1.74607743e-07 0.00000000e+00]
[2.56987636e-06 0.00000000e+00]
[0.0331497 0.       ]
[4.38427582e-07 0.00000000e+00]
[5.74578705e-07 0.00000000e+00]
[4.57880908e-07 0.00000000e+00]
[3.04413e-07 0.00000e+00]
[1.00833824e-07 0.00000000e+00]
[1.28763356e-06 0.00000000e+00]
[6.98197323e-07 0.00000000e+00]
[1.18659709e-07 0.00000000e+00]
[9.59875906e-07 0.00000000e+00]
[1.07586378e-06 0.00000000e+00]
[3.86293747e-07 0.00000000e+00]
[2.78039645e-06 0.00000000e+00]
[0.00083565 0.        ]
[0.00033541 0.        ]
[1.91186985e-06 0.00000000e+00]
[4.02705766e-07 0.00000000e+00]
[8.86110734e-08 0.00000000e+00]
[1.80264026e-07 0.00000000e+00]
[3.16333488e-07 0.00000000e+00]
[1.00833824e-07 0.00000000e+00]
[1.49609873e-07 0.00000000e+00]
[1.74607743e-07 0.00000000e+00]
[5.69835062e-06 0.00000000e+00]
[1.45192632e-07 0.00000000e+00]
[1.40947585e-07 0.00000000e+00]
[1.37049752e-06 0.00000000e+00]
[2.20852

[1.03544225e-07 0.00000000e+00]
[3.28884577e-07 0.00000000e+00]
[2.82310256e-07 0.00000000e+00]
[3.22883034e-05 0.00000000e+00]
[2.52996348e-07 0.00000000e+00]
[2.29564903e-05 0.00000000e+00]
[1.22030467e-07 0.00000000e+00]
[1.21131734e-06 0.00000000e+00]
[2.12536645e-07 0.00000000e+00]
[2.38008347e-06 0.00000000e+00]
[1.07586378e-06 0.00000000e+00]
[1.15411961e-07 0.00000000e+00]
[1.69004043e-05 0.00000000e+00]
[1.00833824e-07 0.00000000e+00]
[4.02705766e-07 0.00000000e+00]
[2.52996348e-07 0.00000000e+00]
[6.98197323e-07 0.00000000e+00]
[1.96202875e-05 0.00000000e+00]
[6.3237817e-07 0.0000000e+00]
[0.00311386 0.        ]
[1.28001277e-05 0.00000000e+00]
[6.0255537e-07 0.0000000e+00]
[6.3237817e-07 0.0000000e+00]
[0.0331497 0.       ]
[1.78332448e-06 0.00000000e+00]
[3.70761559e-07 0.00000000e+00]
[2.35674497e-07 0.00000000e+00]
[4.02705766e-07 0.00000000e+00]
[2.78039645e-06 0.00000000e+00]
[3.01455221e-06 0.00000000e+00]
[8.42753272e-08 0.00000000e+00]
[0.00033541 0.        ]
[1.03544

[2.12536645e-07 0.00000000e+00]
[0.00012386 0.        ]
[7.34552787e-07 0.00000000e+00]
[1.80264026e-07 0.00000000e+00]
[2.05499753e-07 0.00000000e+00]
[2.12536645e-07 0.00000000e+00]
[1.46608558e-05 0.00000000e+00]
[2.35674497e-07 0.00000000e+00]
[3.70761559e-07 0.00000000e+00]
[6.31709921e-06 0.00000000e+00]
[7.46060696e-08 0.00000000e+00]
[4.78502333e-07 0.00000000e+00]
[2.52996348e-07 0.00000000e+00]
[1.74607743e-07 0.00000000e+00]
[3.88997889e-05 0.00000000e+00]
[1.58996768e-07 0.00000000e+00]
[1.06352643e-07 0.00000000e+00]
[8.59978992e-07 0.00000000e+00]
[9.82172013e-08 0.00000000e+00]
[1.86167277e-07 0.00000000e+00]
[1.06352643e-07 0.00000000e+00]
[5.15783836e-06 0.00000000e+00]
[1.29164867e-07 0.00000000e+00]
[1.69185648e-07 0.00000000e+00]
[1.21131734e-06 0.00000000e+00]
[2.56987636e-06 0.00000000e+00]
[5.86913366e-05 0.00000000e+00]
[5.86913366e-05 0.00000000e+00]
[1.1409147e-06 0.0000000e+00]
[9.56903351e-08 0.00000000e+00]
[1.01566558e-06 0.00000000e+00]
[3.89612974e-06 0.

In [17]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 668ms/epoch - 3ms/step


In [ ]:
for i in predictions:
    print(i)

#### Lots of issues: Predicting mostly negative values; the second curvature value is higher than the first in some cases, which should be the opposite, given we are using unsigned labels; the second curvature value should be zero, but the predicted values have significant difference<br>Positives: Predicted values are comparable to the original values (applicable only to the first curvature value).

In [20]:
funct = exp_generator(1, 3)
funct

-3.96833965769284*x**3 - 1.03941757389319*x**2 - 3.4502772919759*x - 4.3348490432041

In [21]:
s, l = datalist_generator(funct, 5, 5, 130, 130)

In [22]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

##### Results are better compared to those of initial architecture<br>(Output deleted to save memory; this architecture failed in giving a better performance, so the output wasn't considered important) 

In [25]:
model.save('Research_main.h5')

In [26]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 3.2825e-07 - mean_absolute_error: 2.4286e-04 - 631ms/epoch - 3ms/step


In [27]:
for i in test_labels:
    print(i)

[7.24777976e-12 0.00000000e+00]
[1.36240588e-10 0.00000000e+00]
[8.06404907e-10 0.00000000e+00]
[6.76536325e-12 0.00000000e+00]
[1.15490688e-12 0.00000000e+00]
[7.76000388e-07 0.00000000e+00]
[2.25396892e-07 0.00000000e+00]
[3.16369844e-12 0.00000000e+00]
[2.14789778e-09 0.00000000e+00]
[4.30576267e-09 0.00000000e+00]
[6.77045077e-10 0.00000000e+00]
[1.05007417e-12 0.00000000e+00]
[1.41178127e-11 0.00000000e+00]
[2.9841767e-12 0.0000000e+00]
[3.35633683e-12 0.00000000e+00]
[1.63575586e-12 0.00000000e+00]
[1.16649595e-09 0.00000000e+00]
[3.16369844e-12 0.00000000e+00]
[3.34569074e-11 0.00000000e+00]
[1.75960394e-10 0.00000000e+00]
[2.9841767e-12 0.0000000e+00]
[4.60765616e-13 0.00000000e+00]
[5.43760158e-08 0.00000000e+00]
[1.4181277e-09 0.0000000e+00]
[1.80116792e-11 0.00000000e+00]
[3.78560373e-12 0.00000000e+00]
[4.02484769e-12 0.00000000e+00]
[8.06404907e-10 0.00000000e+00]
[5.89137239e-13 0.00000000e+00]
[5.55696236e-09 0.00000000e+00]
[4.560128e-12 0.000000e+00]
[4.99435209e-13 0.

[1.30536759e-08 0.00000000e+00]
[2.65167858e-10 0.00000000e+00]
[4.00860387e-06 0.00000000e+00]
[3.78560373e-12 0.00000000e+00]
[6.77045077e-10 0.00000000e+00]
[6.7840644e-11 0.0000000e+00]
[2.81674139e-12 0.00000000e+00]
[8.34099349e-13 0.00000000e+00]
[4.00860387e-06 0.00000000e+00]
[3.37782699e-09 0.00000000e+00]
[4.79633047e-13 0.00000000e+00]
[9.13414222e-13 0.00000000e+00]
[1.20451867e-10 0.00000000e+00]
[6.69296889e-13 0.00000000e+00]
[7.77206958e-12 0.00000000e+00]
[1.80116792e-11 0.00000000e+00]
[1.00195386e-12 0.00000000e+00]
[2.66045373e-12 0.00000000e+00]
[3.0640075e-10 0.0000000e+00]
[2.30417264e-10 0.00000000e+00]
[2.81674139e-12 0.00000000e+00]
[3.78560373e-12 0.00000000e+00]
[9.56455602e-13 0.00000000e+00]
[5.53280262e-12 0.00000000e+00]
[5.20226577e-13 0.00000000e+00]
[8.46862106e-11 0.00000000e+00]
[7.77206958e-12 0.00000000e+00]
[1.05007417e-12 0.00000000e+00]
[1.91369639e-12 0.00000000e+00]
[4.05381386e-11 0.00000000e+00]
[7.27070623e-09 0.00000000e+00]
[5.43760158e

[1.9588151e-11 0.0000000e+00]
[3.55571352e-10 0.00000000e+00]
[1.20451867e-10 0.00000000e+00]
[2.30417264e-10 0.00000000e+00]
[6.98883638e-13 0.00000000e+00]
[7.24777976e-12 0.00000000e+00]
[6.77045077e-10 0.00000000e+00]
[2.2502778e-12 0.0000000e+00]
[5.55696236e-09 0.00000000e+00]
[6.77045077e-10 0.00000000e+00]
[1.27255262e-12 0.00000000e+00]
[3.78768133e-13 0.00000000e+00]
[2.53266711e-08 0.00000000e+00]
[4.09276577e-13 0.00000000e+00]
[8.06404907e-10 0.00000000e+00]
[7.57037469e-11 0.00000000e+00]
[6.14507677e-13 0.00000000e+00]
[9.49784152e-11 0.00000000e+00]
[2.78093516e-11 0.00000000e+00]
[1.52918576e-11 0.00000000e+00]
[2.13331288e-11 0.00000000e+00]
[8.34099349e-13 0.00000000e+00]
[1.52918576e-11 0.00000000e+00]
[4.28246655e-12 0.00000000e+00]
[4.00860387e-06 0.00000000e+00]
[1.27255262e-12 0.00000000e+00]
[1.52918576e-11 0.00000000e+00]
[1.00195386e-12 0.00000000e+00]
[3.37782699e-09 0.00000000e+00]
[1.06808659e-10 0.00000000e+00]
[1.119168e-11 0.000000e+00]
[7.27070623e-09 

In [28]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 554ms/epoch - 2ms/step


In [ ]:
for i in predictions:
    print(i)

#### Positives: Predicting positive values.<br>Issues: Predicted values are not comparable with the original values (neither the first nor the second curvature value); few cases have the first value lower than the second; most of the predicted values are the same, which isn't comparable with the test set.

In [30]:
funct = exp_generator(1, 4)
funct

1.22901694889702*x**4 + 2.41786989260729*x**3 + 2.95193565565697*x**2 + 4.4245028377705*x + 2.39898574739931

In [31]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [32]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

##### Results are extremely bad compared to those of previous architectures.

In [35]:
model.save('Research_main.h5')

# Deleted, as this architecture gives a very poor performance.

In [36]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 0.0203 - mean_absolute_error: 0.1338 - 592ms/epoch - 2ms/step


In [37]:
for i in test_labels:
    print(i)

[6.94972017e-13 0.00000000e+00]
[7.39606847e-15 0.00000000e+00]
[1.07178678e-14 0.00000000e+00]
[3.59449216e-13 0.00000000e+00]
[1.70576964e-13 0.00000000e+00]
[2.34299424e-11 0.00000000e+00]
[1.70713385e-05 0.00000000e+00]
[2.69980307e-15 0.00000000e+00]
[8.27885534e-13 0.00000000e+00]
[2.92073382e-15 0.00000000e+00]
[1.19103367e-12 0.00000000e+00]
[8.27885534e-13 0.00000000e+00]
[1.75507062e-14 0.00000000e+00]
[6.18784666e-15 0.00000000e+00]
[3.16256937e-15 0.00000000e+00]
[8.88152667e-15 0.00000000e+00]
[1.5138421e-09 0.0000000e+00]
[7.39606847e-15 0.00000000e+00]
[2.92073382e-15 0.00000000e+00]
[9.90093805e-14 0.00000000e+00]
[6.57910887e-16 0.00000000e+00]
[2.49776604e-15 0.00000000e+00]
[2.49776604e-15 0.00000000e+00]
[3.33410274e-14 0.00000000e+00]
[1.5138421e-09 0.0000000e+00]
[2.62303354e-12 0.00000000e+00]
[3.33410274e-14 0.00000000e+00]
[3.82045163e-08 0.00000000e+00]
[2.49776604e-15 0.00000000e+00]
[1.74797006e-12 0.00000000e+00]
[2.67434629e-14 0.00000000e+00]
[2.98346745e

[3.17700812e-16 0.00000000e+00]
[1.75507062e-14 0.00000000e+00]
[4.95944865e-13 0.00000000e+00]
[4.03573446e-12 0.00000000e+00]
[1.15710662e-10 0.00000000e+00]
[5.29730086e-14 0.00000000e+00]
[7.82598209e-07 0.00000000e+00]
[9.12553106e-16 0.00000000e+00]
[4.03573446e-12 0.00000000e+00]
[9.18695588e-10 0.00000000e+00]
[5.97684978e-14 0.00000000e+00]
[9.90653318e-13 0.00000000e+00]
[5.76347223e-10 0.00000000e+00]
[8.88152667e-15 0.00000000e+00]
[3.73263038e-14 0.00000000e+00]
[3.13628676e-11 0.00000000e+00]
[1.5138421e-09 0.0000000e+00]
[2.53245682e-16 0.00000000e+00]
[6.75786662e-14 0.00000000e+00]
[5.11788021e-16 0.00000000e+00]
[8.09999053e-15 0.00000000e+00]
[4.25169343e-11 0.00000000e+00]
[5.66939351e-15 0.00000000e+00]
[8.15506877e-11 0.00000000e+00]
[5.66939351e-15 0.00000000e+00]
[2.99984349e-16 0.00000000e+00]
[3.10683984e-06 0.00000000e+00]
[9.90093805e-14 0.00000000e+00]
[1.71419742e-15 0.00000000e+00]
[1.07178678e-14 0.00000000e+00]
[4.18653614e-14 0.00000000e+00]
[4.0357344

[1.5138421e-09 0.0000000e+00]
[3.73263038e-14 0.00000000e+00]
[3.07665062e-13 0.00000000e+00]
[5.85892262e-13 0.00000000e+00]
[2.98346745e-14 0.00000000e+00]
[1.43909055e-12 0.00000000e+00]
[3.16256937e-15 0.00000000e+00]
[1.74797006e-12 0.00000000e+00]
[1.43483261e-14 0.00000000e+00]
[8.69713131e-14 0.00000000e+00]
[1.43483261e-14 0.00000000e+00]
[5.05806196e-12 0.00000000e+00]
[0.00307836 0.        ]
[1.07178678e-14 0.00000000e+00]
[1.96772869e-13 0.00000000e+00]
[3.07665062e-13 0.00000000e+00]
[2.83388011e-16 0.00000000e+00]
[3.78466157e-16 0.00000000e+00]
[1.70576964e-13 0.00000000e+00]
[4.52881896e-16 0.00000000e+00]
[3.07665062e-13 0.00000000e+00]
[1.19926467e-15 0.00000000e+00]
[2.27668288e-13 0.00000000e+00]
[1.15710662e-10 0.00000000e+00]
[1.98795277e-15 0.00000000e+00]
[1.29271995e-13 0.00000000e+00]
[1.35294311e-11 0.00000000e+00]
[1.17965627e-14 0.00000000e+00]
[9.75043776e-15 0.00000000e+00]
[1.04473883e-15 0.00000000e+00]
[4.18653614e-14 0.00000000e+00]
[1.04473883e-15 0.

[2.27668288e-13 0.00000000e+00]
[1.04399329e-11 0.00000000e+00]
[1.30011269e-14 0.00000000e+00]
[7.39606847e-15 0.00000000e+00]
[9.12553106e-16 0.00000000e+00]
[1.48291528e-13 0.00000000e+00]
[8.75636511e-09 0.00000000e+00]
[2.92073382e-15 0.00000000e+00]
[2.15963629e-14 0.00000000e+00]
[6.18784666e-15 0.00000000e+00]
[3.16256937e-15 0.00000000e+00]
[1.75507062e-14 0.00000000e+00]
[4.03573446e-12 0.00000000e+00]
[5.19999396e-15 0.00000000e+00]
[1.71419742e-15 0.00000000e+00]
[6.17346159e-16 0.00000000e+00]
[2.99984349e-16 0.00000000e+00]
[8.69713131e-14 0.00000000e+00]
[2.15963629e-14 0.00000000e+00]
[4.26359794e-16 0.00000000e+00]
[2.53245682e-16 0.00000000e+00]
[4.52881896e-16 0.00000000e+00]
[7.99139086e-16 0.00000000e+00]
[3.16256937e-15 0.00000000e+00]
[5.85892262e-13 0.00000000e+00]
[1.19103367e-12 0.00000000e+00]
[4.81306343e-16 0.00000000e+00]
[8.15506877e-11 0.00000000e+00]
[1.12989704e-13 0.00000000e+00]
[8.27885534e-13 0.00000000e+00]
[9.18695588e-10 0.00000000e+00]
[6.76119

In [38]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 562ms/epoch - 2ms/step


In [ ]:
for i in predictions:
    print(i)

#### Positives: Predicted mostly positive values for first curvature value; predicted mostly the same value for the second one, which is like the original labels where the second curvature value is all zero.<br>Major Issue: The values are not at all comparable, hence the higher loss and error.

### Looking at these results, we can safely conclude that this architecture has a very poor performance.
### Since the performance of the architectures used so far aren't properly comparable because of the use of different expressions, we decided to test the initial architecture again, but this time on the seeded dataset for proper comparison, along with checking its predictions on the same dataset to better understand the performance of the said architecture.

## Initial Architecture Used - 12 Layers Deep; 24 nodes Wide
### No. of samples (128 x 128)<br>"Unsigned" Labels<br>Training & Testing from scratch<br>Seeded expressions used

In [6]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [7]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [8]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [10]:
model = Sequential([
    Dense(units = 3, input_shape = (9,), activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 18, activation = 'relu'),
    Dense(units = 21, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 21, activation = 'relu'),
    Dense(units = 18, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 3, activation = 'relu'),
    Dense(units = 2)
])

In [11]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [12]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 3s - loss: 7.3700e-04 - mean_absolute_error: 0.0034 - val_loss: 7.4555e-04 - val_mean_absolute_error: 0.0040 - 3s/epoch - 7ms/step
Epoch 2/30
417/417 - 1s - loss: 7.3506e-04 - mean_absolute_error: 0.0038 - val_loss: 7.4556e-04 - val_mean_absolute_error: 0.0039 - 827ms/epoch - 2ms/step
Epoch 3/30
417/417 - 1s - loss: 7.3494e-04 - mean_absolute_error: 0.0039 - val_loss: 7.4569e-04 - val_mean_absolute_error: 0.0037 - 1s/epoch - 3ms/step
Epoch 4/30
417/417 - 1s - loss: 7.3505e-04 - mean_absolute_error: 0.0037 - val_loss: 7.4557e-04 - val_mean_absolute_error: 0.0041 - 1s/epoch - 3ms/step
Epoch 5/30
417/417 - 1s - loss: 7.3504e-04 - mean_absolute_error: 0.0038 - val_loss: 7.4557e-04 - val_mean_absolute_error: 0.0041 - 1s/epoch - 3ms/step
Epoch 6/30
417/417 - 1s - loss: 7.3504e-04 - mean_absolute_error: 0.0039 - val_loss: 7.4573e-04 - val_mean_absolute_error: 0.0037 - 1s/epoch - 3ms/step
Epoch 7/30
417/417 - 1s - loss: 7.3494e-04 - mean_absolute_error: 0.0037 - val_loss: 

##### The results are not better than what was obtained with the other architectures

In [13]:
model.save('Research_main3.h5')

In [14]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 4.7312e-04 - mean_absolute_error: 0.0030 - 370ms/epoch - 2ms/step


In [ ]:
for i in test_labels:
    print(i)

In [16]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 0s - 436ms/epoch - 2ms/step


In [ ]:
for i in predictions:
    print(i)

#### Positives: The predicted second curvature value is the same for all samples and close to zero, which is comparable with the test labels. The first curvature value is positive and higher than the second curvature value for all samples, which is also comparable with the test set.<br>Major issues: The first curvature values are the same for all samples, which is not comparable to the test set, and the values themselves are not comparable to the test labels as well.

In [18]:
funct = exp_generator(1, 3)
funct

-3.96833965769284*x**3 - 1.03941757389319*x**2 - 3.4502772919759*x - 4.3348490432041

In [19]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [20]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [22]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 1s - loss: 8.7387e-07 - mean_absolute_error: 4.0578e-04 - val_loss: 1.2223e-07 - val_mean_absolute_error: 4.6165e-05 - 1s/epoch - 3ms/step
Epoch 2/30
417/417 - 1s - loss: 8.1952e-08 - mean_absolute_error: 3.6823e-05 - val_loss: 1.2252e-07 - val_mean_absolute_error: 3.8729e-05 - 1s/epoch - 3ms/step
Epoch 3/30
417/417 - 1s - loss: 8.1994e-08 - mean_absolute_error: 3.4915e-05 - val_loss: 1.2225e-07 - val_mean_absolute_error: 4.5471e-05 - 1s/epoch - 3ms/step
Epoch 4/30
417/417 - 1s - loss: 8.2026e-08 - mean_absolute_error: 3.5938e-05 - val_loss: 1.2231e-07 - val_mean_absolute_error: 4.3446e-05 - 1s/epoch - 4ms/step
Epoch 5/30
417/417 - 1s - loss: 8.2016e-08 - mean_absolute_error: 3.6337e-05 - val_loss: 1.2264e-07 - val_mean_absolute_error: 3.6832e-05 - 1s/epoch - 3ms/step
Epoch 6/30
417/417 - 1s - loss: 8.2003e-08 - mean_absolute_error: 3.5746e-05 - val_loss: 1.2257e-07 - val_mean_absolute_error: 3.7846e-05 - 1s/epoch - 3ms/step
Epoch 7/30
417/417 - 2s - loss: 8.2009e-

##### In general, results are better compared to other architectures.<br>However, the training loss and error got down to a minimum value, then started to slowly increase. On the other hand, the validation error oscillated a lot, and the validation loss was slightly higher than the training loss.

In [23]:
model.save('Research_main3.h5')

In [24]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 9.6256e-08 - mean_absolute_error: 3.8355e-05 - 369ms/epoch - 2ms/step


In [ ]:
for i in test_labels:
    print(i)

In [26]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 0s - 265ms/epoch - 1ms/step


In [ ]:
for i in predictions:
    print(i)

#### Positives: The predicted second curvature value is the same for all samples and close to zero, which is comparable with the test labels. The predicted values are also equal to those obtained for the univariate quadratic samples, which is a good thing, given for both types of expressions, the second curvature value is always equal to zero. The first curvature value is positive and higher than the second curvature value for all samples, which is also comparable with the test set.<br>Major issues: The first curvature values are the same for all samples, which is not comparable to the test set, and the values themselves are not comparable to the test labels as well.

In [28]:
funct = exp_generator(1, 4)
funct

1.22901694889702*x**4 + 2.41786989260729*x**3 + 2.95193565565697*x**2 + 4.4245028377705*x + 2.39898574739931

In [29]:
s, l = datalist_generator(funct, 5, 5, 130, 130)

In [30]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [32]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 2s - loss: 183945887744.0000 - mean_absolute_error: 57599.5977 - val_loss: 2.4943e-04 - val_mean_absolute_error: 0.0158 - 2s/epoch - 4ms/step
Epoch 2/30
417/417 - 1s - loss: 2.4943e-04 - mean_absolute_error: 0.0158 - val_loss: 2.4943e-04 - val_mean_absolute_error: 0.0158 - 1s/epoch - 3ms/step
Epoch 3/30
417/417 - 2s - loss: 2.4943e-04 - mean_absolute_error: 0.0158 - val_loss: 2.4943e-04 - val_mean_absolute_error: 0.0158 - 2s/epoch - 4ms/step
Epoch 4/30
417/417 - 2s - loss: 2.4943e-04 - mean_absolute_error: 0.0158 - val_loss: 2.4943e-04 - val_mean_absolute_error: 0.0158 - 2s/epoch - 4ms/step
Epoch 5/30
417/417 - 2s - loss: 2.4943e-04 - mean_absolute_error: 0.0158 - val_loss: 2.4943e-04 - val_mean_absolute_error: 0.0158 - 2s/epoch - 4ms/step
Epoch 6/30
417/417 - 2s - loss: 2.4943e-04 - mean_absolute_error: 0.0158 - val_loss: 2.4943e-04 - val_mean_absolute_error: 0.0158 - 2s/epoch - 4ms/step
Epoch 7/30
417/417 - 2s - loss: 2.4943e-04 - mean_absolute_error: 0.0158 - va

##### Compared to the 16 layer deep, 18 nodes wide architecture, these results are better; however, the results are worse than what was obtained with the 16 layer deep, 36 nodes wide architecture.<br>One thing to be noted is that, this same architecture gave extremely good results for a different univariate quartic expression

In [33]:
model.save('Research_main3.h5')

In [34]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 2.4914e-04 - mean_absolute_error: 0.0158 - 605ms/epoch - 3ms/step


In [ ]:
for i in test_labels:
    print(i)

In [36]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 537ms/epoch - 2ms/step


In [ ]:
for i in predictions:
    print(i)

#### Positives: The predicted second curvature value is the same for all samples, which is comparable with the test labels. The first curvature value is positive and higher than the second curvature value for all samples, which is also comparable with the test set.<br>Major issues: The first curvature values are the same for all samples, which is not comparable to the test set. The values for both the max and min curvatures are a lot higher than the test labels. The second curvature values are also negative.

In [6]:
funct = exp_generator(2, 3)
funct

4.22324996665417*x**3 - 0.343773456218947*x**2*y + 1.48974553136924*x**2 + 4.43356716998314*x*y**2 - 3.86794035346856*x*y - 0.309309522178363*x - 4.70994771716385*y**3 + 4.00900491750623*y**2 - 2.5342716738017*y + 0.437608592359304

In [7]:
s, l = datalist_generator(funct, 5, 5, 130, 130)

In [8]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [10]:
from tensorflow.keras.models import load_model
model = load_model('Research_main3.h5')

In [11]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 4s - loss: 0.0677 - mean_absolute_error: 0.0242 - val_loss: 0.0124 - val_mean_absolute_error: 0.0176 - 4s/epoch - 10ms/step
Epoch 2/30
417/417 - 2s - loss: 0.0021 - mean_absolute_error: 0.0142 - val_loss: 5.3721e-04 - val_mean_absolute_error: 0.0137 - 2s/epoch - 4ms/step
Epoch 3/30
417/417 - 1s - loss: 5.7724e-04 - mean_absolute_error: 0.0135 - val_loss: 3.4783e-04 - val_mean_absolute_error: 0.0135 - 1s/epoch - 4ms/step
Epoch 4/30
417/417 - 2s - loss: 4.1142e-04 - mean_absolute_error: 0.0134 - val_loss: 2.5588e-04 - val_mean_absolute_error: 0.0134 - 2s/epoch - 5ms/step
Epoch 5/30
417/417 - 2s - loss: 3.2059e-04 - mean_absolute_error: 0.0133 - val_loss: 2.1669e-04 - val_mean_absolute_error: 0.0133 - 2s/epoch - 4ms/step
Epoch 6/30
417/417 - 3s - loss: 2.6057e-04 - mean_absolute_error: 0.0132 - val_loss: 2.0369e-04 - val_mean_absolute_error: 0.0132 - 3s/epoch - 7ms/step
Epoch 7/30
417/417 - 2s - loss: 2.2129e-04 - mean_absolute_error: 0.0132 - val_loss: 1.9676e-04 - v

##### Results are not too worse; good thing about the results is that they decreased gradually.

In [12]:
model.save('Research_main3.h5')

In [13]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 9.8049e-05 - mean_absolute_error: 0.0095 - 605ms/epoch - 3ms/step


In [14]:
for i in test_labels:
    print(i)

[8.19269498e-03 1.72902630e-11]
[1.06319679e-02 1.50428552e-09]
[6.76316390e-03 7.13636846e-10]
[8.06591020e-03 9.38571443e-12]
[3.02635834e-03 1.50252488e-12]
[1.16015438e-03 4.44332982e-12]
[5.73322478e-03 3.75289643e-13]
[1.70923178e-02 7.67981078e-11]
[5.40522984e-03 9.74513872e-13]
[3.55349158e-03 5.98356434e-13]
[3.04619015e-03 8.62608596e-13]
[1.02524410e-02 3.93919582e-10]
[6.75237300e-03 9.21067476e-13]
[5.27419133e-03 2.97205403e-13]
[9.47212450e-04 3.20181273e-13]
[7.86996886e-03 4.69574466e-12]
[6.58291307e-03 6.53417858e-13]
[5.10905093e-03 6.48328179e-13]
[1.87843315e-03 1.14986291e-11]
[1.00721132e-02 5.58455279e-12]
[1.17166166e-02 4.99023947e-11]
[1.25777068e-03 4.15457013e-12]
[1.63598940e-03 7.15781601e-12]
[3.50370318e-02 3.13514169e-09]
[5.78665057e-03 1.12631258e-12]
[2.02975345e-03 3.98182789e-13]
[3.53910199e-03 1.28969058e-12]
[1.58061038e-02 5.45612184e-11]
[3.73440546e-03 5.35705161e-13]
[2.40248760e-02 1.10341266e-09]
[2.18806635e-05 1.41528108e-10]
[7.79119

[4.60894349e-03 1.63549469e-12]
[2.23584410e-02 4.25341403e-09]
[4.7684364e-03 6.0720309e-13]
[3.99454938e-03 3.48088745e-13]
[2.53275954e-03 8.37563569e-12]
[2.90623025e-03 1.74821616e-12]
[3.37015905e-03 7.82423605e-13]
[1.37781609e-02 2.91578124e-11]
[1.97461257e-02 5.14572366e-10]
[2.05919917e-02 2.21957732e-10]
[1.01655364e-03 5.28509571e-12]
[2.85200386e-03 5.46639773e-13]
[1.97718386e-03 1.87051459e-12]
[4.83437397e-03 1.24035591e-12]
[4.87734927e-03 9.28471276e-13]
[7.87165611e-03 4.38568799e-12]
[1.78793749e-03 1.19834045e-10]
[8.34371216e-03 2.14119087e-12]
[3.82724253e-03 2.81271078e-12]
[5.78538992e-03 1.01315835e-11]
[1.30561789e-02 1.83234292e-10]
[9.47540709e-03 1.46341455e-11]
[7.36711703e-03 1.23485900e-12]
[5.66770655e-03 8.34754574e-13]
[5.61385994e-03 3.92325885e-12]
[6.24442031e-03 2.42847400e-11]
[2.72432041e-03 1.05649864e-12]
[4.10605338e-03 9.12209978e-12]
[9.08203709e-03 2.90150643e-09]
[1.77452793e-03 2.25359477e-12]
[9.86949891e-03 4.84747398e-12]
[8.2553881

[6.62963817e-03 2.24427118e-12]
[2.00239906e-03 4.31186337e-13]
[6.05979737e-03 3.78085974e-12]
[5.71960083e-03 3.58105039e-13]
[1.29093075e-03 3.93110674e-13]
[7.33457964e-03 5.54668941e-12]
[2.63724723e-03 8.67567953e-13]
[4.96434327e-03 8.13609163e-13]
[6.89595577e-03 8.26294762e-13]
[6.55645734e-03 1.73494509e-12]
[6.81849966e-03 1.25798634e-12]
[5.99884886e-03 2.54477585e-11]
[1.52074017e-02 4.38548876e-11]
[5.56986961e-03 2.82156676e-13]
[6.15351769e-03 1.99325582e-12]
[8.32640849e-03 2.41115287e-12]
[4.73080215e-04 2.19605988e-12]
[3.36025644e-03 6.88817059e-13]
[3.76749801e-03 1.11477212e-12]
[1.53474462e-03 1.26404375e-10]
[2.19015234e-02 8.35154515e-10]
[1.84982710e-03 4.30116229e-13]
[3.33433076e-03 3.74384117e-13]
[1.05819427e-02 1.85606400e-11]
[5.89892677e-03 5.79169525e-13]
[3.75124463e-03 8.89743812e-12]
[1.05645879e-02 1.58198506e-11]
[5.08423781e-03 2.98164271e-13]
[6.95157267e-03 2.04186798e-11]
[4.37973370e-03 3.90173137e-13]
[2.00814894e-03 1.22169581e-11]
[7.83878

In [15]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 786ms/epoch - 3ms/step


In [ ]:
for i in predictions:
    print(i)

#### Positives: Predicted max curvature is higher than the predicted min.<br>Similar issues: Predicting one single value for all samples. Predicted values are quite higher. Predicted min curvatures are negative.

In [17]:
funct = exp_generator(3, 1)
funct

1.03920038596194 - 2.62035372908109*sin(0.442292252959518*x + 1.30044833451921)

In [18]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [19]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [21]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 2s - loss: 0.0329 - mean_absolute_error: 0.1026 - val_loss: 0.0309 - val_mean_absolute_error: 0.0980 - 2s/epoch - 4ms/step
Epoch 2/30
417/417 - 2s - loss: 0.0306 - mean_absolute_error: 0.0972 - val_loss: 0.0285 - val_mean_absolute_error: 0.0926 - 2s/epoch - 4ms/step
Epoch 3/30
417/417 - 2s - loss: 0.0281 - mean_absolute_error: 0.0920 - val_loss: 0.0261 - val_mean_absolute_error: 0.0876 - 2s/epoch - 4ms/step
Epoch 4/30
417/417 - 2s - loss: 0.0256 - mean_absolute_error: 0.0872 - val_loss: 0.0237 - val_mean_absolute_error: 0.0831 - 2s/epoch - 5ms/step
Epoch 5/30
417/417 - 2s - loss: 0.0233 - mean_absolute_error: 0.0829 - val_loss: 0.0215 - val_mean_absolute_error: 0.0790 - 2s/epoch - 5ms/step
Epoch 6/30
417/417 - 2s - loss: 0.0211 - mean_absolute_error: 0.0791 - val_loss: 0.0195 - val_mean_absolute_error: 0.0756 - 2s/epoch - 4ms/step
Epoch 7/30
417/417 - 2s - loss: 0.0191 - mean_absolute_error: 0.0762 - val_loss: 0.0177 - val_mean_absolute_error: 0.0732 - 2s/epoch - 4

##### The results are a bit higher, but a reason that could be attributed to this is the fact that curvature values for this type of expressions are generally higher.

In [22]:
model.save('Research_main3.h5')

# Deleted, as this architecture doesn't perform well for our purpose.

In [23]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 0.0141 - mean_absolute_error: 0.0744 - 633ms/epoch - 3ms/step


In [24]:
for i in test_labels:
    print(i)

[0.34657273 0.        ]
[0.08794473 0.        ]
[0.14147338 0.        ]
[0.11755703 0.        ]
[0.46821516 0.        ]
[0.09385088 0.        ]
[0.02066544 0.        ]
[0.34657273 0.        ]
[0.05594906 0.        ]
[0.10524055 0.        ]
[0.42361821 0.        ]
[0.08794473 0.        ]
[0.11755703 0.        ]
[0.36535038 0.        ]
[0.07347424 0.        ]
[0.01870925 0.        ]
[0.1685098 0.       ]
[0.02553777 0.        ]
[0.13725839 0.        ]
[0.06430826 0.        ]
[0.09101973 0.        ]
[0.01890055 0.        ]
[0.04783225 0.        ]
[0.00084918 0.        ]
[0.03992343 0.        ]
[0.08038014 0.        ]
[0.04990655 0.        ]
[0.37224158 0.        ]
[0.5109974 0.       ]
[0.11424871 0.        ]
[0.01223045 0.        ]
[0.5109974 0.       ]
[0.13725839 0.        ]
[0.24733617 0.        ]
[0.4964949 0.       ]
[0.39839299 0.        ]
[0.04990655 0.        ]
[0.12692734 0.        ]
[0.46300888 0.        ]
[0.10185455 0.        ]
[0.25326211 0.        ]
[0.04556573 0.        ]


[0.44122667 0.        ]
[0.25326211 0.        ]
[0.42361821 0.        ]
[0.09071508 0.        ]
[0.1685098 0.       ]
[0.50428391 0.        ]
[0.39912513 0.        ]
[0.02573454 0.        ]
[0.09385088 0.        ]
[0.2949714 0.       ]
[0.2698609 0.       ]
[0.03992343 0.        ]
[0.09071508 0.        ]
[0.48595759 0.        ]
[0.36535038 0.        ]
[0.04783225 0.        ]
[0.3729904 0.       ]
[0.32042252 0.        ]
[0.06430826 0.        ]
[0.34582447 0.        ]
[0.48169156 0.        ]
[0.13725839 0.        ]
[0.06201373 0.        ]
[0.46821516 0.        ]
[0.17296726 0.        ]
[0.03797859 0.        ]
[0.46821516 0.        ]
[0.01395824 0.        ]
[0.08794473 0.        ]
[0.02066544 0.        ]
[0.15682561 0.        ]
[0.1021834 0.       ]
[0.03258838 0.        ]
[0.39839299 0.        ]
[0.39163042 0.        ]
[0.24669709 0.        ]
[0.04760835 0.        ]
[0.48121127 0.        ]
[0.40581758 0.        ]
[0.3729904 0.       ]
[0.33895875 0.        ]
[0.39839299 0.        ]
[0.0

[0.31297161 0.        ]
[0.49265315 0.        ]
[0.5109974 0.       ]
[0.00084918 0.        ]
[0.02573454 0.        ]
[0.02573454 0.        ]
[0.27053725 0.        ]
[0.01695666 0.        ]
[0.0535534 0.       ]
[0.33895875 0.        ]
[0.06430826 0.        ]
[0.18609834 0.        ]
[0.13089783 0.        ]
[0.5109974 0.       ]
[0.36535038 0.        ]
[0.31968872 0.        ]
[0.17345772 0.        ]
[0.48169156 0.        ]
[0.1610384 0.       ]
[0.08794473 0.        ]
[0.35345986 0.        ]
[0.31297161 0.        ]
[0.01870925 0.        ]
[0.17345772 0.        ]
[0.24087452 0.        ]
[0.02553777 0.        ]
[0.05814508 0.        ]
[0.1106599 0.       ]
[0.44122667 0.        ]
[0.37986157 0.        ]
[0.01395824 0.        ]
[0.50675069 0.        ]
[0.01870925 0.        ]
[0.37224158 0.        ]
[0.0421077 0.       ]
[0.1909152 0.       ]
[0.08038014 0.        ]
[0.05594906 0.        ]
[0.42361821 0.        ]
[0.30152485 0.        ]
[0.31297161 0.        ]
[0.33895875 0.        ]
[0.512

In [25]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 518ms/epoch - 2ms/step


In [ ]:
for i in predictions:
    print(i)

#### Positives: Predicted min curvature is close to zero, the actual label value. The predicted max curvature is positive, higher than the min curvature, and close to the actual labels.<br>Issues: Same problem as others, predicting only one value.

### So, this architecture fails in properly estimating surface curvatures, as it only outputs a single value for all cases. Hence, we experiment with a deeper network next.

## Changed Architecture - 16 Layers Deep; 24 nodes Wide (Width remains the same)
### No. of samples (128 x 128)<br>"Unsigned" Labels<br>Training & Testing from scratch<br>Seeded expressions used

In [6]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [7]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [8]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [10]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 10, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 10, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 2)
])

In [11]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [12]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 4s - loss: 0.0331 - mean_absolute_error: 0.0884 - val_loss: 0.0014 - val_mean_absolute_error: 0.0219 - 4s/epoch - 8ms/step
Epoch 2/30
417/417 - 1s - loss: 9.8986e-04 - mean_absolute_error: 0.0151 - val_loss: 7.7404e-04 - val_mean_absolute_error: 0.0120 - 1s/epoch - 3ms/step
Epoch 3/30
417/417 - 2s - loss: 8.1101e-04 - mean_absolute_error: 0.0110 - val_loss: 7.2297e-04 - val_mean_absolute_error: 0.0096 - 2s/epoch - 4ms/step
Epoch 4/30
417/417 - 2s - loss: 7.6113e-04 - mean_absolute_error: 0.0083 - val_loss: 6.7266e-04 - val_mean_absolute_error: 0.0066 - 2s/epoch - 4ms/step
Epoch 5/30
417/417 - 1s - loss: 7.2929e-04 - mean_absolute_error: 0.0063 - val_loss: 6.6011e-04 - val_mean_absolute_error: 0.0057 - 1s/epoch - 3ms/step
Epoch 6/30
417/417 - 1s - loss: 7.1599e-04 - mean_absolute_error: 0.0055 - val_loss: 6.3997e-04 - val_mean_absolute_error: 0.0048 - 1s/epoch - 4ms/step
Epoch 7/30
417/417 - 2s - loss: 6.8688e-04 - mean_absolute_error: 0.0040 - val_loss: 6.1323e-04 

##### Results are quite good.

In [13]:
model.save('Research_main4.h5')

In [14]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 1.3139e-05 - mean_absolute_error: 4.7868e-04 - 352ms/epoch - 1ms/step


In [ ]:
for i in test_labels:
    print(i)

In [16]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 0s - 433ms/epoch - 2ms/step


In [ ]:
for i in predictions:
    print(i)

#### Positives: The first values are comparable.<br>Issues: Signed predictions; second values are not comparable

In [18]:
funct = exp_generator(1, 3)
funct

-3.96833965769284*x**3 - 1.03941757389319*x**2 - 3.4502772919759*x - 4.3348490432041

In [19]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [20]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [22]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 1s - loss: 1.3846 - mean_absolute_error: 0.0907 - val_loss: 4.3723e-05 - val_mean_absolute_error: 0.0052 - 1s/epoch - 3ms/step
Epoch 2/30
417/417 - 1s - loss: 3.0848e-05 - mean_absolute_error: 0.0047 - val_loss: 2.2134e-05 - val_mean_absolute_error: 0.0042 - 1s/epoch - 3ms/step
Epoch 3/30
417/417 - 1s - loss: 1.7609e-05 - mean_absolute_error: 0.0038 - val_loss: 1.3150e-05 - val_mean_absolute_error: 0.0032 - 1s/epoch - 3ms/step
Epoch 4/30
417/417 - 3s - loss: 1.0391e-05 - mean_absolute_error: 0.0028 - val_loss: 7.6247e-06 - val_mean_absolute_error: 0.0024 - 3s/epoch - 7ms/step
Epoch 5/30
417/417 - 2s - loss: 6.1215e-06 - mean_absolute_error: 0.0021 - val_loss: 4.4065e-06 - val_mean_absolute_error: 0.0018 - 2s/epoch - 4ms/step
Epoch 6/30
417/417 - 2s - loss: 3.6236e-06 - mean_absolute_error: 0.0015 - val_loss: 2.5304e-06 - val_mean_absolute_error: 0.0012 - 2s/epoch - 6ms/step
Epoch 7/30
417/417 - 1s - loss: 2.2834e-06 - mean_absolute_error: 0.0011 - val_loss: 1.5699e

##### Results are pretty good.

In [23]:
model.save('Research_main4.h5')

# Deleted, because the architecture doesn't perform well, and also, as we change our workflow and goal hereafter, these
# model parameters are no longer important.

In [24]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 3.2695e-07 - mean_absolute_error: 1.9260e-04 - 356ms/epoch - 1ms/step


In [ ]:
for i in test_labels:
    print(i)

In [26]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 0s - 325ms/epoch - 1ms/step


In [ ]:
for i in predictions:
    print(i)

#### Issues: Mostly predicts similar values; values are not comparable.<br>The reason behind the losses and errors being low could be that the curvature values were themselves quite low.

In [6]:
funct = exp_generator(1, 4)
funct

1.22901694889702*x**4 + 2.41786989260729*x**3 + 2.95193565565697*x**2 + 4.4245028377705*x + 2.39898574739931

In [7]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [8]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [10]:
from tensorflow.keras.models import load_model
model = load_model('Research_main4.h5')

In [11]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 2s - loss: 9808657408.0000 - mean_absolute_error: 25827.0098 - val_loss: 398973696.0000 - val_mean_absolute_error: 11842.6006 - 2s/epoch - 6ms/step
Epoch 2/30
417/417 - 1s - loss: 222158864.0000 - mean_absolute_error: 8601.7324 - val_loss: 112692904.0000 - val_mean_absolute_error: 6300.7188 - 865ms/epoch - 2ms/step
Epoch 3/30
417/417 - 1s - loss: 63337344.0000 - mean_absolute_error: 4634.0435 - val_loss: 31274016.0000 - val_mean_absolute_error: 3321.3184 - 969ms/epoch - 2ms/step
Epoch 4/30
417/417 - 1s - loss: 16643549.0000 - mean_absolute_error: 2357.9531 - val_loss: 7266558.0000 - val_mean_absolute_error: 1607.4458 - 1s/epoch - 3ms/step
Epoch 5/30
417/417 - 1s - loss: 3499227.2500 - mean_absolute_error: 1072.4126 - val_loss: 1243757.7500 - val_mean_absolute_error: 667.7463 - 1s/epoch - 3ms/step
Epoch 6/30
417/417 - 1s - loss: 524837.9375 - mean_absolute_error: 410.2877 - val_loss: 141863.1562 - val_mean_absolute_error: 225.5672 - 1s/epoch - 3ms/step
Epoch 7/30
41

##### The worst of all results.

In [12]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 1.3179 - mean_absolute_error: 1.0309 - 417ms/epoch - 2ms/step


In [ ]:
for i in test_labels:
    print(i)

In [14]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 0s - 449ms/epoch - 2ms/step


In [ ]:
for i in predictions:
    print(i)

#### Issues: Values are extremely high; some predictions are negative; 2nd value is higher than the first in some cases.

### This architecture also fails to perform for the work we are trying to do.
### After deliberating, we came to the conclusion that, since the 2nd curvature values for all types of univariate polynomials are always zero, the models could inaccurately learn a pattern from this. Consequently, the predictions are way off, like, sometimes the models are predicting only a single value for all cases, sometimes the predictions are negative, and often the predictions have a huge difference with the actual labels.
### So, we decide to change our experiment procedure to training our models on only the max valued curvatures and focus our goal for the time being to identifying an architecture that can accurately estimate the max valued surface curvature only, putting off the work of discovering another architecture that can estimate the min valued surface curvature to a later stage. Thus, we divide the work of estimating the two different principal surface curvatures to be done separately by two different architectures.

## Architecture - Depth(12 layers in total), Width (24 nodes highest among the layers)
### No. of samples (128 x 128)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked

In [4]:
# Function for generating input sample and unsigned labels for a particular point of interest

def data_generator(exp, center_x, center_y):
    l_sample = []
    
    for i in range(center_x-1, center_x+2):
        for j in range(center_y-1, center_y+2):
            value = exp.evalf(subs={x: i, y: j})
            l_sample.append(value)
            
    fx = exp.diff(x, 1)
    fy = exp.diff(y, 1)
    fxx = exp.diff(x, 2)
    fyy = exp.diff(y, 2)
    fxy = exp.diff(x, y, 1)
    
    v_fx = fx.evalf(subs={x: center_x, y: center_y})
    v_fy = fy.evalf(subs={x: center_x, y: center_y})
    v_fxx = fxx.evalf(subs={x: center_x, y: center_y})
    v_fyy = fyy.evalf(subs={x: center_x, y: center_y})
    v_fxy = fxy.evalf(subs={x: center_x, y: center_y})
    
    K = (v_fxx*v_fyy - v_fxy**2) / (1 + v_fx**2 + v_fy**2)**2
    H = (v_fxx + v_fyy + v_fxx*v_fy**2 + v_fyy*v_fx**2 - 2*v_fx*v_fy*v_fxy) / (2 * (1 + v_fx**2 + v_fy**2)**1.5)
    
    k1 = H + (H**2 - K)**0.5
    k2 = H - (H**2 - K)**0.5
    
    #*********************************************
    # Changes made to create unsigned labels
    
    u_k1 = abs(k1)
    u_k2 = abs(k2)
    
    if(u_k1 < u_k2):
        temp = u_k1
        u_k1 = u_k2
        u_k2 = temp
    
    #*********************************************
    
    #*********************************************
    # Changes made to split labels into two groups
    
    l_label_max = u_k1
    l_label_min = u_k2
    
    return l_sample, l_label_max, l_label_min

    #*********************************************

In [5]:
# Function for generating a list of samples and labels for a range of points

def datalist_generator(exp, x_start, y_start, x_end, y_end):
    l_samples = []
    l_labels_max = []
    l_labels_min = []

    for i in range(x_start, x_end+1):
        for j in range(y_start, y_end+1):
            t_sample, t_label_max, t_label_min = data_generator(exp, i, j)
            l_samples.append(t_sample)
            l_labels_max.append(t_label_max)
            l_labels_min.append(t_label_min)
        
    samples = np.array(l_samples, dtype = 'float64')
    labels_max = np.array(l_labels_max, dtype = 'float64')
    labels_min = np.array(l_labels_min, dtype = 'float64')
    
    return samples, labels_max, labels_min

In [6]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [7]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [8]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l_max, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [10]:
train_samples.shape

(11113, 9)

In [11]:
model = Sequential([
    Dense(units = 3, input_shape = (9,), activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 18, activation = 'relu'),
    Dense(units = 21, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 21, activation = 'relu'),
    Dense(units = 18, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 3, activation = 'relu'),
    Dense(units = 1)
])

In [12]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

##### Results are stuck at a particular point, and they are not that low either.

In [14]:
model.save('Research_split1.h5')

In [15]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 0.0015 - mean_absolute_error: 0.0076 - 550ms/epoch - 2ms/step


In [ ]:
for i in test_labels:
    print(i)

In [17]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 768ms/epoch - 3ms/step


In [ ]:
for i in predictions:
    print(i)

#### Issues: Predicting same value; predicted values are quite higher than the actual labels. Similar type of prediction for this architecture as with the case of considering both curvatures together.

In [19]:
funct = exp_generator(1, 3)
funct

-3.96833965769284*x**3 - 1.03941757389319*x**2 - 3.4502772919759*x - 4.3348490432041

In [20]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [21]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l_max, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [23]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 2s - loss: 1.7361e-06 - mean_absolute_error: 7.6656e-04 - val_loss: 1.8631e-07 - val_mean_absolute_error: 9.4375e-05 - 2s/epoch - 4ms/step
Epoch 2/30
417/417 - 2s - loss: 1.7219e-07 - mean_absolute_error: 8.0057e-05 - val_loss: 1.8660e-07 - val_mean_absolute_error: 5.8993e-05 - 2s/epoch - 4ms/step
Epoch 3/30
417/417 - 2s - loss: 1.7243e-07 - mean_absolute_error: 7.2121e-05 - val_loss: 1.8613e-07 - val_mean_absolute_error: 7.4805e-05 - 2s/epoch - 4ms/step
Epoch 4/30
417/417 - 2s - loss: 1.7224e-07 - mean_absolute_error: 7.4280e-05 - val_loss: 1.8610e-07 - val_mean_absolute_error: 8.2080e-05 - 2s/epoch - 4ms/step
Epoch 5/30
417/417 - 2s - loss: 1.7238e-07 - mean_absolute_error: 7.4276e-05 - val_loss: 1.8663e-07 - val_mean_absolute_error: 1.0258e-04 - 2s/epoch - 4ms/step
Epoch 6/30
417/417 - 2s - loss: 1.7242e-07 - mean_absolute_error: 7.5421e-05 - val_loss: 1.8616e-07 - val_mean_absolute_error: 8.7676e-05 - 2s/epoch - 4ms/step
Epoch 7/30
417/417 - 2s - loss: 1.7245e-

##### Results are low in value; however, they are oscillating quite a bit.

In [ ]:
model.save('Research_split1.h5')

# Deleted, because this architecture isn't suited for our work.

In [25]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 2.1193e-07 - mean_absolute_error: 9.5417e-05 - 624ms/epoch - 3ms/step


In [ ]:
for i in test_labels:
    print(i)

In [27]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 586ms/epoch - 2ms/step


In [ ]:
for i in predictions:
    print(i)

#### Issues: Predicting same value again; not significantly higher, but high enough compared to the actual labels.

### This architecture isn't suited for our purpose, because, even after using only the max valued curvatures, similar issues as earlier persist. So, we experiment with a deeper architecture next.

## Architecture - Depth(16 layers in total), Width (24 nodes highest among the layers)
### No. of samples (128 x 128)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked

In [29]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [30]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [31]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l_max, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [33]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 10, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 10, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [34]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [35]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 7s - loss: 0.0626 - mean_absolute_error: 0.1330 - val_loss: 0.0037 - val_mean_absolute_error: 0.0390 - 7s/epoch - 16ms/step
Epoch 2/30
417/417 - 2s - loss: 0.0022 - mean_absolute_error: 0.0264 - val_loss: 0.0017 - val_mean_absolute_error: 0.0199 - 2s/epoch - 4ms/step
Epoch 3/30
417/417 - 2s - loss: 0.0015 - mean_absolute_error: 0.0177 - val_loss: 0.0016 - val_mean_absolute_error: 0.0160 - 2s/epoch - 4ms/step
Epoch 4/30
417/417 - 2s - loss: 0.0014 - mean_absolute_error: 0.0144 - val_loss: 0.0015 - val_mean_absolute_error: 0.0123 - 2s/epoch - 4ms/step
Epoch 5/30
417/417 - 2s - loss: 0.0013 - mean_absolute_error: 0.0091 - val_loss: 0.0014 - val_mean_absolute_error: 0.0068 - 2s/epoch - 4ms/step
Epoch 6/30
417/417 - 2s - loss: 0.0013 - mean_absolute_error: 0.0061 - val_loss: 0.0014 - val_mean_absolute_error: 0.0058 - 2s/epoch - 4ms/step
Epoch 7/30
417/417 - 2s - loss: 0.0013 - mean_absolute_error: 0.0053 - val_loss: 0.0013 - val_mean_absolute_error: 0.0053 - 2s/epoch - 

##### Results are not too bad. Looks good

In [36]:
model.save('Research_split2.h5')

In [37]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 1.1871e-04 - mean_absolute_error: 0.0011 - 637ms/epoch - 3ms/step


In [ ]:
for i in test_labels:
    print(i)

In [39]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 730ms/epoch - 3ms/step


In [ ]:
for i in predictions:
    print(i)

#### Issues: Predicting a lot of negative values; predicted values are slightly higher than the actual labels.<br>Similar type of prediction for this architecture, as with the case of considering both curvatures together.

In [41]:
funct = exp_generator(1, 3)
funct

-3.96833965769284*x**3 - 1.03941757389319*x**2 - 3.4502772919759*x - 4.3348490432041

In [42]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [43]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l_max, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

##### Results are really good. The output was deleted to save memory, which wasn't a big deal because this architecture eventually failed to deliver.

In [46]:
model.save('Research_split2.h5')

# Deleted, as this architecture also fails to achieve the goal successfully.

In [47]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 1.0741e-08 - mean_absolute_error: 3.4302e-05 - 622ms/epoch - 3ms/step


In [ ]:
for i in test_labels:
    print(i)

In [49]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 519ms/epoch - 2ms/step


In [ ]:
for i in predictions:
    print(i)

#### Issues: Predicting the same value for most of the samples; values are higher than the actual labels.<br>Similar type of prediction for this architecture, as with the case of considering both curvatures together.

### Similarly, this architecture also fails to estimate surface curvatures successfully. So finally, we decide to change the architecture one last time, increasing the width of the model so that it has enough nodes to capture more information. One motivation behind testing out this architecture was that it had previously given better results, but with different expressions.

## Architecture - Depth(16 layers in total), Width (36 nodes highest among the layers)
### No. of samples (128 x 128)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked

In [6]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [7]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [8]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l_max, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [10]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [11]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [12]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 8s - loss: 0.6088 - mean_absolute_error: 0.2237 - val_loss: 0.0014 - val_mean_absolute_error: 0.0156 - 8s/epoch - 19ms/step
Epoch 2/30
417/417 - 2s - loss: 0.0015 - mean_absolute_error: 0.0110 - val_loss: 0.0011 - val_mean_absolute_error: 0.0084 - 2s/epoch - 4ms/step
Epoch 3/30
417/417 - 2s - loss: 0.0015 - mean_absolute_error: 0.0093 - val_loss: 0.0011 - val_mean_absolute_error: 0.0082 - 2s/epoch - 5ms/step
Epoch 4/30
417/417 - 2s - loss: 0.0015 - mean_absolute_error: 0.0091 - val_loss: 0.0011 - val_mean_absolute_error: 0.0081 - 2s/epoch - 4ms/step
Epoch 5/30
417/417 - 2s - loss: 0.0015 - mean_absolute_error: 0.0090 - val_loss: 0.0011 - val_mean_absolute_error: 0.0079 - 2s/epoch - 4ms/step
Epoch 6/30
417/417 - 2s - loss: 0.0015 - mean_absolute_error: 0.0088 - val_loss: 0.0011 - val_mean_absolute_error: 0.0078 - 2s/epoch - 4ms/step
Epoch 7/30
417/417 - 2s - loss: 0.0015 - mean_absolute_error: 0.0087 - val_loss: 0.0011 - val_mean_absolute_error: 0.0076 - 2s/epoch - 

##### Loss was at a low value, and did decrease gradually; error was low as well, but had some weird fluctuations towards the last epochs.

In [13]:
model.save('Research_split3.h5')

In [14]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 3.6676e-04 - mean_absolute_error: 0.0021 - 556ms/epoch - 2ms/step


In [ ]:
for i in test_labels:
    print(i)

In [16]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 842ms/epoch - 4ms/step


In [ ]:
for i in predictions:
    print(i)

#### The predicted values were a bit higher than the actual labels; there were quite a lot negative values, but majority were positive; the values were different for different samples; one thing was interesting: the 7th and 8th label from the end were the same in case of both the test set and the predictions, indicating the model is identifying patterns within the dataset.

In [6]:
funct = exp_generator(1, 3)
funct

-3.96833965769284*x**3 - 1.03941757389319*x**2 - 3.4502772919759*x - 4.3348490432041

In [7]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [8]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l_max, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [10]:
from tensorflow.keras.models import load_model
model = load_model('Research_split3.h5')

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

##### Results are quite good. As like earlier, the output was deleted to save memory, which wasn't a problem as this experiment eventually failed.

In [12]:
model.save('Research_split3.h5')

In [13]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 4.6216e-06 - mean_absolute_error: 8.4387e-04 - 567ms/epoch - 2ms/step


In [ ]:
for i in test_labels:
    print(i)

In [15]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 834ms/epoch - 3ms/step


In [ ]:
for i in predictions:
    print(i)

#### Predicted values are quite higher than the actual labels; almost all the predictions are negative; values are different for different samples, though; one interesting thing for this polynomial as well: the network is good at identifying patterns, which can be inferred as the last two samples were the same for both the predictions and the actual labels.

In [17]:
funct = exp_generator(1, 4)
funct

1.22901694889702*x**4 + 2.41786989260729*x**3 + 2.95193565565697*x**2 + 4.4245028377705*x + 2.39898574739931

In [18]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [19]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l_max, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

##### Loss is low, but the error is a bit high. Output deleted for the same reason as earlier.

In [22]:
model.save('Research_split3.h5')

# This one is deleted as well, because the architecture eventually fails at the work.

In [23]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 5.3966e-05 - mean_absolute_error: 0.0073 - 439ms/epoch - 2ms/step


In [ ]:
for i in test_labels:
    print(i)

In [25]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 0s - 395ms/epoch - 2ms/step


In [ ]:
for i in predictions:
    print(i)

#### Predicts mostly the same value for all samples.

### Even though the architecture showed good potential at the beginning, it eventually produces the same problems as like the other architectures.<br>Since we tried out different architectures but couldn't make any of them to work, we next decide to experiment with the dataset and try to understand the issue. In most of the cases, we saw that the models were predicting negative values. A reason behind this could probably be that the input values were negative or really small themselves. Also, the actual curvatures, being very small values for the expressions being used, might also have some kind of effect in causing our models to behave poorly.  So, we first decide to check out the inputs, and if our assumption was correct, add a higher constant to the input values, in which case the curvatures remain unchanged, and test how our model behaves.

## Constant added to input values; curvatures remain unchanged
### Architecture - Depth(16 layers in total), Width (36 nodes highest among the layers)<br>No. of samples (128 x 128)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked

In [6]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [7]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [8]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l_max, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [ ]:
for i in train_samples:
    print(i)

### Looking at the inputs, we find that they are quite large in value and a number of them are negative.<br>Since a number of the inputs are negative, our models might incorrectly identify a pattern and transfer that to the output. Making the inputs unsigned may solve this problem of predictions being negative; however, we cannot just make the inputs unsigned, as it will change the meaning of the inputs and thus not serve our purpose.<br>Also, since the inputs are already large enough in value, the issue might have stemmed from the curvature values being quite small. So, adding a constant value might not bring about any major improvement in the results. 
### After all these careful deliberations, we decided not to continue with this particular experiment.

### So, next we decide to test out the inputs by multiplying them with a very large constant. Multiplying the inputs by a constant changes the respective curvatures by the same degree, so the curvatures are also multiplied by the same constant. Since the curvatures for the expressions used are quite small in value, multiplying them with a larger constant can significantly increase them in value, thereby helping to solve the problem our model is facing.

## Constant(10^10) multiplied to both input values and curvatures
### Architecture - Depth(16 layers in total), Width (36 nodes highest among the layers)<br>No. of samples (128 x 128)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked

In [6]:
# Function for multiplying input values and curvatures by a constant

def mult_scale(inp, curv):
    l_sample = []
    l_samples = []
    l_labels = []
    
    con = 500     # This value is changed for different experiments
                  # The current value was used in the last experiment
    
    for l in inp:
        for item in l:
            item *= con
            l_sample.append(item)
            
        l_samples.append(l_sample)
        l_sample = []
    
    for item in curv:
        item *= con
        l_labels.append(item)
    
    sc_samples = np.array(l_samples, dtype = 'float64')
    sc_labels = np.array(l_labels, dtype = 'float64')
    
    return sc_samples, sc_labels

In [7]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [8]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [9]:
sc_s, sc_l_max = mult_scale(s, l_max)

In [ ]:
for i in sc_s:
    print(i)

In [ ]:
for i in sc_l_max:
    print(i)

In [14]:
train_samples, test_samples, train_labels, test_labels = train_test_split(sc_s, sc_l_max, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [16]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [17]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

##### Results were 'NaN's

### The reason behind this could be either our dataset got corrupted while generating it or the inputs became too large after multiplying that our model couldn't handle them. We first check our dataset if it is corrupted or not, as a sanity check. If we fail to find any corruption, the latter reason would be the cause of the 'NaN' results, which can be solved by multiplying with a comparatively smaller constant.

## Solving the 'NaN' issue

### Sanity check for any corrupted data

### Iterating over all samples and labels and checking the values

In [7]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [8]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [9]:
sc_s, sc_l_max = mult_scale(s, l_max)

#### Labels

In [ ]:
for i in sc_l_max:
    if(i == i):
        print("Ok")
    elif(i != i):
        print("Not Ok")

##### Output had shown that All Labels were Ok

#### Input Samples

In [ ]:
for l in sc_s:
    for i in l:
        if(i == i):
            print("Ok")
        elif(i != i):
            print("Not Ok")
            
    print(" ")

##### Output had shown that All Samples were Ok

### Multiplying by a constant of 1 and checking if the Constant Multiplier function works fine, by training our model once

In [7]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [8]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [9]:
sc_s, sc_l_max = mult_scale(s, l_max)

In [ ]:
for i in sc_s:
    print(i)

In [ ]:
for i in sc_l_max:
    print(i)

In [12]:
train_samples, test_samples, train_labels, test_labels = train_test_split(sc_s, sc_l_max, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [14]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [15]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [16]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 3s - loss: 0.5685 - mean_absolute_error: 0.2190 - val_loss: 0.0018 - val_mean_absolute_error: 0.0162 - 3s/epoch - 6ms/step
Epoch 2/30
417/417 - 1s - loss: 0.0012 - mean_absolute_error: 0.0101 - val_loss: 0.0015 - val_mean_absolute_error: 0.0095 - 659ms/epoch - 2ms/step
Epoch 3/30
417/417 - 1s - loss: 0.0012 - mean_absolute_error: 0.0084 - val_loss: 0.0015 - val_mean_absolute_error: 0.0092 - 679ms/epoch - 2ms/step
Epoch 4/30
417/417 - 1s - loss: 0.0012 - mean_absolute_error: 0.0081 - val_loss: 0.0015 - val_mean_absolute_error: 0.0090 - 678ms/epoch - 2ms/step
Epoch 5/30
417/417 - 1s - loss: 0.0012 - mean_absolute_error: 0.0079 - val_loss: 0.0015 - val_mean_absolute_error: 0.0087 - 714ms/epoch - 2ms/step
Epoch 6/30
417/417 - 1s - loss: 0.0012 - mean_absolute_error: 0.0076 - val_loss: 0.0015 - val_mean_absolute_error: 0.0085 - 696ms/epoch - 2ms/step
Epoch 7/30
417/417 - 1s - loss: 0.0012 - mean_absolute_error: 0.0073 - val_loss: 0.0015 - val_mean_absolute_error: 0.0082

### The results are similar as before, indicating that the Constant Multiplier function works, and our data generated is not corrupted either. Hence, the cause of the NaN results was that the inputs became too large to handle for our model when multiplied by the large contant of 10^10.<br>So, next up, we continued our experiment by starting with a very small constant and checking if our model improved in its predictions, and gradually increasing the constant value when the results didn't improve.

## Constant(5) multiplied to both input values and curvatures
### Architecture - Depth(16 layers in total), Width (36 nodes highest among the layers)<br>No. of samples (128 x 128)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked

In [10]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [11]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [12]:
sc_s, sc_l_max = mult_scale(s, l_max)

In [ ]:
for i in sc_s:
    print(i)

In [ ]:
for i in sc_l_max:
    print(i)

In [15]:
train_samples, test_samples, train_labels, test_labels = train_test_split(sc_s, sc_l_max, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [20]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [21]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

##### Received the results on the 2nd try; the results were better than the ones with no constant multiplied.<br> Output was deleted to save memory.

In [23]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 7.4694e-08 - mean_absolute_error: 1.1421e-04 - 347ms/epoch - 1ms/step


In [ ]:
for i in test_labels:
    print(i)

In [25]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 0s - 490ms/epoch - 2ms/step


In [ ]:
for i in predictions:
    print(i)

#### Positives: Predicting positive values; the values are close to the originals.<br>Issues: Almost all of the predicted values are the same.

### So we see that, multiplying with a much smaller constant doesn't improve the performance of our model, since the same problem persists. Hence, we try out a slightly larger constant.

## Constant(10) multiplied to both input values and curvatures
### Architecture - Depth(16 layers in total), Width (36 nodes highest among the layers)<br>No. of samples (128 x 128)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked

In [7]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [8]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [9]:
sc_s, sc_l_max = mult_scale(s, l_max)

In [12]:
train_samples, test_samples, train_labels, test_labels = train_test_split(sc_s, sc_l_max, test_size = 0.3)

In [14]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [15]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

##### Results (loss and error) are quite high in value compared to other experiments with the same network. Output deleted to save memory, as this experiment didn't yield any fruitful result.

In [17]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 0.1173 - mean_absolute_error: 0.0364 - 633ms/epoch - 3ms/step


In [ ]:
for i in test_labels:
    print(i)

In [19]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 839ms/epoch - 4ms/step


In [ ]:
for i in predictions:
    print(i)

#### Postives: Predictions are different for different samples, just like the original labels; patterns in the original labels are being reflected in the predictions<br>Issues: The predictions are much larger in value than the originals, resulting in higher loss and error.

In [21]:
model.save('Research_constm1.h5') # Deleted, as the experiment eventually failed

In [ ]:
# Ran the training from scratch again to have better loss and error and check how the model predicts in that case

In [22]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [23]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

##### Good results.

In [25]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 1.4852e-07 - mean_absolute_error: 1.4253e-04 - 574ms/epoch - 2ms/step


In [ ]:
for i in test_labels:
    print(i)

In [27]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 764ms/epoch - 3ms/step


In [ ]:
for i in predictions:
    print(i)

#### Better loss and error yield predictions that are the identical for almost all the samples.

#### Checked the performance with another type of expression to better understand the impact of the constant multiplication, particularly with the trained model on quadratic samples that yielded a higher loss and error, but didn't break to predict a single curvature value. This was done to see if this trained model continued to predict different values for different samples and improved on the loss and error, or eventually broke and predicted identical values for different samples.

In [29]:
funct = exp_generator(1, 4)
funct

1.22901694889702*x**4 + 2.41786989260729*x**3 + 2.95193565565697*x**2 + 4.4245028377705*x + 2.39898574739931

In [30]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [31]:
sc_s, sc_l_max = mult_scale(s, l_max)

In [34]:
train_samples, test_samples, train_labels, test_labels = train_test_split(sc_s, sc_l_max, test_size = 0.3)

In [36]:
from tensorflow.keras.models import load_model
model = load_model('Research_constm1.h5')

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

##### Results are extremely high in value.

In [38]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 0.2543 - mean_absolute_error: 0.2692 - 474ms/epoch - 2ms/step


In [ ]:
for i in test_labels:
    print(i)

In [40]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 570ms/epoch - 2ms/step


In [ ]:
for i in predictions:
    print(i)

#### Issues: Predicts mostly a single value; the predicted values are quite larger than the originals.
#### Whatever the losses and errors, high or low, the network eventually breaks, even after multiplying with the constant

### So, multiplying with a small constant doesn't help us in improving the performance of our model. Therefore, we experiment with a much larger constant hereafter.

## Constant(1000) multiplied to both input values and curvatures
### Architecture - Depth(16 layers in total), Width (36 nodes highest among the layers)<br>No. of samples (128 x 128)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked

In [7]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [8]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [9]:
sc_s, sc_l_max = mult_scale(s, l_max)

In [12]:
train_samples, test_samples, train_labels, test_labels = train_test_split(sc_s, sc_l_max, test_size = 0.3)

In [42]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [43]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

##### Results are comparatively worse than previous experiments; these were obtained after multiple tries: At the first try, the results were extremely bad and the predictions were all similar; at the 2nd try the results improved slightly and were similar to this last iteration, but the predictions were mostly similar too. Eventually, we obtained the results at this last iteration which were still quite high, but the predictions by the model were different for different samples, so we continued with this training iteration.

In [45]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 0.6647 - mean_absolute_error: 0.1596 - 429ms/epoch - 2ms/step


In [ ]:
for i in test_labels:
    print(i)

In [47]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 553ms/epoch - 2ms/step


In [ ]:
for i in predictions:
    print(i)

#### Postives: Predicted values are different for different samples. The model is identifying patterns - the 12th and 14th sample have the same curvature for both the original and predicted case; the predicted values are all positive.
#### Issues: The predicted values are a bit higher compared to the original values, which is why the loss and error are high too.

In [49]:
model.save('Research_constm1.h5')

#### Since the model doesn't break in case of quadratic expression, when the parameters are multiplied with a higher constant, we experiment with another type of expression to verify the model continues to give such performance and improve on the loss, with this higher constant multiplication. 

In [7]:
funct = exp_generator(1, 4)
funct

1.22901694889702*x**4 + 2.41786989260729*x**3 + 2.95193565565697*x**2 + 4.4245028377705*x + 2.39898574739931

In [8]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [9]:
sc_s, sc_l_max = mult_scale(s, l_max)

In [12]:
train_samples, test_samples, train_labels, test_labels = train_test_split(sc_s, sc_l_max, test_size = 0.3)

In [14]:
from tensorflow.keras.models import load_model
model = load_model('Research_constm1.h5')

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

#### Results are 'NaNs'. The input values have probably gotten too large for our network to handle.

### So, multiplying with a higher constant like 1000 wouldn't work for our purpose, as the inputs become too large to handle for our network in that scenario. Hence, we run our experiment with a slightly lower constant than 1000.

## Constant(500) multiplied to both input values and curvatures
### Architecture - Depth(16 layers in total), Width (36 nodes highest among the layers)<br>No. of samples (128 x 128)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked

In [7]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [8]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [9]:
sc_s, sc_l_max = mult_scale(s, l_max)

In [12]:
train_samples, test_samples, train_labels, test_labels = train_test_split(sc_s, sc_l_max, test_size = 0.3)

In [38]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [39]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

##### Decent results obtained at the 5th try.

In [41]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 0.1544 - mean_absolute_error: 0.0754 - 441ms/epoch - 2ms/step


In [ ]:
for i in test_labels:
    print(i)

In [43]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 619ms/epoch - 3ms/step


In [ ]:
for i in predictions:
    print(i)

#### All the attempts gave similar predictions for different samples.

### Even though this experiment with constant multiplication showed potential signs that the model could improve and solve the problem it was facing, eventually the same issue started to show up anyways with the constants we could work with. Also, we had restriction on how large a constant we could use, as using a very large constant would cause the inputs to become too large for our model to handle. Hence, we decided not to continue with this experiment anymore.
### Next, we decided to change the type of model architecture used and try out a CNN architecture. We decided to employ Transfer Learning with the pretrained state-of-the-art VGG-16 model and check out if the performance improves.

## Architecture - Transfer Learning applied with VGG16 Model - Layers added at the top (2 Dense Hidden Layers, Max Width - 64)
### No. of samples (32 * 32 = 1024)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked

In [1]:
import sympy
import numpy as np
import tensorflow as tf
from tensorflow import keras
from random import uniform, seed
from sklearn.utils import shuffle
from tensorflow.keras.models import *
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.metrics import *
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

In [2]:
x = sympy.Symbol('x')
y = sympy.Symbol('y')

In [3]:
# Function for generating input sample and unsigned labels for VGG16 transfer-learned model for a particular point of interest

def data_generator(exp, center_x, center_y):
    l_sample = []
    
    for i in range(center_x-16, center_x+17):
        for j in range(center_y-16, center_y+17):
            value = exp.evalf(subs={x: i, y: j})
            l_sample.extend([value, value, value])
            
    sample = np.array(l_sample, dtype = 'float64')
    sample = sample.reshape((33,33,3), order = 'C') # VGG16 accepts only (33,33, 3) inputs
    l_sample_n = sample.tolist()
            
    fx = exp.diff(x, 1)
    fy = exp.diff(y, 1)
    fxx = exp.diff(x, 2)
    fyy = exp.diff(y, 2)
    fxy = exp.diff(x, y, 1)
    
    v_fx = fx.evalf(subs={x: center_x, y: center_y})
    v_fy = fy.evalf(subs={x: center_x, y: center_y})
    v_fxx = fxx.evalf(subs={x: center_x, y: center_y})
    v_fyy = fyy.evalf(subs={x: center_x, y: center_y})
    v_fxy = fxy.evalf(subs={x: center_x, y: center_y})
    
    K = (v_fxx*v_fyy - v_fxy**2) / (1 + v_fx**2 + v_fy**2)**2
    H = (v_fxx + v_fyy + v_fxx*v_fy**2 + v_fyy*v_fx**2 - 2*v_fx*v_fy*v_fxy) / (2 * (1 + v_fx**2 + v_fy**2)**1.5)
    
    k1 = H + (H**2 - K)**0.5
    k2 = H - (H**2 - K)**0.5
    
    #*********************************************
    # Changes made to create unsigned labels
    
    u_k1 = abs(k1)
    u_k2 = abs(k2)
    
    if(u_k1 < u_k2):
        temp = u_k1
        u_k1 = u_k2
        u_k2 = temp
    
    #*********************************************
    
    #*********************************************
    # Changes made to split labels into two groups
    
    l_label_max = u_k1
    l_label_min = u_k2
    
    return l_sample_n, l_label_max, l_label_min

    #*********************************************

In [4]:
# Function for generating a list of samples and labels for a range of points

def datalist_generator(exp, x_start, y_start, x_end, y_end):
    l_samples = []
    l_labels_max = []
    l_labels_min = []

    for i in range(x_start, x_end+1):
        for j in range(y_start, y_end+1):
            t_sample, t_label_max, t_label_min = data_generator(exp, i, j)
            l_samples.append(t_sample)
            l_labels_max.append(t_label_max)
            l_labels_min.append(t_label_min)
        
    samples = np.array(l_samples, dtype = 'float64')
    labels_max = np.array(l_labels_max, dtype = 'float64')
    labels_min = np.array(l_labels_min, dtype = 'float64')
    
    return samples, labels_max, labels_min

In [7]:
# Function for generating expressions

def exp_generator(num, h_deg):
    
    if (num == 1):
        
        seed(num + h_deg)
        
        a = uniform(-5.0, 5.0)
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-5.0, 5.0)
        e = uniform(-5.0, 5.0)
        
        if(h_deg == 3):
            a = 0.0
        elif(h_deg == 2):
            a = 0.0
            b = 0.0
        
        f = a*x**4 + b*x**3 + c*x**2 + d*x + e
        
        return f
    
    elif (num == 2):
        
        seed(num + h_deg)
        
        a = uniform(-5.0, 5.0)
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-5.0, 5.0)
        e = uniform(-5.0, 5.0)
        f = uniform(-5.0, 5.0)
        g = uniform(-5.0, 5.0)
        h = uniform(-5.0, 5.0)
        i = uniform(-5.0, 5.0)
        j = uniform(-5.0, 5.0)
        k = uniform(-5.0, 5.0)
        l = uniform(-5.0, 5.0)
        m = uniform(-5.0, 5.0)
        n = uniform(-5.0, 5.0)
        o = uniform(-5.0, 5.0)
        
        if(h_deg == 3):
            a = 0.0
            b = 0.0
            c = 0.0
            d = 0.0
            e = 0.0
        elif(h_deg == 2):
            a = 0.0
            b = 0.0
            c = 0.0
            d = 0.0
            e = 0.0
            f = 0.0
            g = 0.0
            h = 0.0
            i = 0.0
        
        f = a*x**4 + b*y**4 + c*x**3*y + d*x*y**3 + e*x**2*y**2 + f*x**3 + g*y**3 + h*x**2*y + i*x*y**2 + j*x**2 + k*y**2 + l*x*y + m*x + n*y + o
        
        return f
    
    elif (num == 3):
        
        seed(num**num)
        
        a = uniform(-2.0, 2.0)
        
        while(a == 0):
            a = uniform(-2.0, 2.0)
        
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-1.0, 1.0)
        
        f = a*sympy.sin(b*x-c) + d
        
        return f
    
    elif (num == 4):
        
        seed(num**num)
        
        a = uniform(-2.0, 2.0)
        
        while(a == 0):
            a = uniform(-2.0, 2.0)
        
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-1.0, 1.0)
        
        f = a*sympy.cos(b*x-c) + d
        
        return f
    
    else:
        f = 0
        return f

In [6]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [7]:
s, l_max, l_min = datalist_generator(funct, 16, 16, 47, 47)  # 32*32 number of input samples takes about 5 minutes to generate

In [8]:
s.shape

(1024, 33, 33, 3)

In [9]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l_max, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [13]:
base = VGG16(weights = 'imagenet', include_top = False, input_shape = (33, 33, 3))
base.trainable = False

inp = Input(shape = (33, 33, 3))
p_out = preprocess_input(inp)
temp = base(p_out)

temp = Flatten()(temp)
temp = Dense(units = 64, activation = 'relu')(temp)
temp = Dense(units = 16, activation = 'relu')(temp)
out = Dense(units = 1)(temp)

model = Model(inp, out)

In [14]:
model.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 33, 33, 3)]       0         
                                                                 
 tf.__operators__.getitem_1   (None, 33, 33, 3)        0         
 (SlicingOpLambda)                                               
                                                                 
 tf.nn.bias_add_1 (TFOpLambd  (None, 33, 33, 3)        0         
 a)                                                              
                                                                 
 vgg16 (Functional)          (None, 1, 1, 512)         14714688  
                                                                 
 flatten_1 (Flatten)         (None, 512)               0         
                                                                 
 dense_4 (Dense)             (None, 64)                3283

In [15]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [16]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
27/27 - 11s - loss: 4319.3457 - mean_absolute_error: 47.6824 - val_loss: 233.6760 - val_mean_absolute_error: 11.5133 - 11s/epoch - 413ms/step
Epoch 2/30
27/27 - 11s - loss: 161.6274 - mean_absolute_error: 9.5654 - val_loss: 59.9197 - val_mean_absolute_error: 7.4425 - 11s/epoch - 389ms/step
Epoch 3/30
27/27 - 10s - loss: 40.8117 - mean_absolute_error: 5.8180 - val_loss: 29.2938 - val_mean_absolute_error: 5.0762 - 10s/epoch - 367ms/step
Epoch 4/30
27/27 - 11s - loss: 19.6777 - mean_absolute_error: 3.9920 - val_loss: 17.0037 - val_mean_absolute_error: 3.8084 - 11s/epoch - 400ms/step
Epoch 5/30
27/27 - 11s - loss: 11.5842 - mean_absolute_error: 2.9392 - val_loss: 10.5965 - val_mean_absolute_error: 2.8503 - 11s/epoch - 393ms/step
Epoch 6/30
27/27 - 10s - loss: 7.4391 - mean_absolute_error: 2.2521 - val_loss: 7.3188 - val_mean_absolute_error: 2.3227 - 10s/epoch - 373ms/step
Epoch 7/30
27/27 - 11s - loss: 5.4594 - mean_absolute_error: 1.9567 - val_loss: 5.6852 - val_mean_absolute_e

##### Results gradually decreased, but are high in value.

In [17]:
model.save('Research_cnn1.h5') # Deleted eventually, as the model performance wasn't up to the mark

In [18]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

16/16 - 2s - loss: 0.5660 - mean_absolute_error: 0.5468 - 2s/epoch - 136ms/step


In [19]:
for i in test_labels:
    print(i)

9.925806126218425e-06
4.2657225000703795e-06
1.7833244808148006e-06
2.3800834666950492e-06
9.925806126218425e-06
2.0530721886499472e-06
4.2657225000703795e-06
1.4660855828911181e-05
2.295649025022784e-05
2.2085284234469827e-06
8.807736649288902e-06
3.228830343125016e-05
1.962028750622306e-05
6.317099213708548e-06
1.911869845939113e-06
7.851470380383595e-06
1.4660855828911181e-05
7.851470380383595e-06
5.698350623952945e-06
7.0287667596710785e-06
4.2657225000703795e-06
1.1241398250993103e-05
3.228830343125016e-05
2.7803964499161985e-06
2.295649025022784e-05
1.7833244808148006e-06
1.6660491428701165e-06
3.014552208693586e-06
1.6900404284117608e-05
4.68357807277168e-06
3.896129736174623e-06
1.962028750622306e-05
1.5588353320124995e-06
1.911869845939113e-06
3.896129736174623e-06
3.896129736174623e-06
1.6660491428701165e-06
5.698350623952945e-06
3.889978889453968e-05
1.4606267763883891e-06
2.0530721886499472e-06
2.569876357955127e-06
3.896129736174623e-06
2.295649025022784e-05
9.925806126218

In [20]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

16/16 - 3s - 3s/epoch - 160ms/step


In [21]:
for i in predictions:
    print(i)

[-0.28011954]
[-0.25222266]
[0.29872453]
[1.4246763]
[-0.28011954]
[1.6167825]
[-0.25222266]
[0.3360151]
[0.2846912]
[1.7998232]
[-0.21240962]
[-0.40938723]
[0.38694322]
[-0.32938826]
[0.94706285]
[-0.18611395]
[0.3360151]
[-0.18611395]
[-0.516256]
[-0.22125113]
[-0.25222266]
[-0.11519302]
[-0.40938723]
[0.4791702]
[0.2846912]
[0.29872453]
[-0.21874487]
[0.60852087]
[0.32649887]
[-0.22414076]
[-0.16529334]
[0.38694322]
[-1.0251776]
[0.94706285]
[-0.16529334]
[-0.16529334]
[-0.21874487]
[-0.5162426]
[-0.705063]
[-2.191285]
[1.6167825]
[0.93687093]
[-0.16529334]
[0.2846912]
[-0.28011954]
[-0.40938723]
[0.94706285]
[0.94706285]
[-0.22122633]
[-0.22414076]
[-0.41331828]
[-0.41331828]
[-0.25222266]
[0.23358667]
[0.3494209]
[-0.32938826]
[0.04473626]
[1.7998232]
[0.3869599]
[-0.11522926]
[0.38694322]
[0.29872453]
[0.2846912]
[-0.7050539]
[-0.21874487]
[-0.16529334]
[1.4246763]
[0.2846912]
[-0.11519302]
[1.6167825]
[0.04473626]
[-0.32938826]
[0.60852087]
[-0.21240962]
[0.3494209]
[-0.7050539]

#### High predicted values and quite a lot of negative numbers; however, good at identifying patterns and different predictions for different samples

### Even though this model didn't break into predicting a single value for different samples, its predictions were quite far off from the actual labels and mostly negative. Hence, we decided to use a deeper network with a slightly greater width and check if the performance improves.

## Architecture - Transfer Learning applied with VGG16 Model - Layers added to the Top (4 Dense Hidden Layers, Max Width - 128)
### No. of samples (32 * 32 = 1024)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked

In [6]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [7]:
s, l_max, l_min = datalist_generator(funct, 16, 16, 47, 47)

In [8]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l_max, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [14]:
base = VGG16(weights = 'imagenet', include_top = False, input_shape = (33, 33, 3))
base.trainable = False

inp = Input(shape = (33, 33, 3))
p_out = preprocess_input(inp)
temp = base(p_out)

temp = Flatten()(temp)
temp = Dense(units = 128, activation = 'relu')(temp)
temp = Dense(units = 64, activation = 'relu')(temp)
temp = Dense(units = 32, activation = 'relu')(temp)
temp = Dense(units = 8, activation = 'relu')(temp)
out = Dense(units = 1)(temp)

model = Model(inp, out)

In [15]:
model.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 33, 33, 3)]       0         
                                                                 
 tf.__operators__.getitem_1   (None, 33, 33, 3)        0         
 (SlicingOpLambda)                                               
                                                                 
 tf.nn.bias_add_1 (TFOpLambd  (None, 33, 33, 3)        0         
 a)                                                              
                                                                 
 vgg16 (Functional)          (None, 1, 1, 512)         14714688  
                                                                 
 flatten_1 (Flatten)         (None, 512)               0         
                                                                 
 dense_5 (Dense)             (None, 128)               6566

In [16]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [17]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
27/27 - 4s - loss: 1267.0276 - mean_absolute_error: 21.2445 - val_loss: 0.2526 - val_mean_absolute_error: 0.3225 - 4s/epoch - 132ms/step
Epoch 2/30
27/27 - 2s - loss: 0.1399 - mean_absolute_error: 0.2185 - val_loss: 0.0860 - val_mean_absolute_error: 0.1739 - 2s/epoch - 85ms/step
Epoch 3/30
27/27 - 2s - loss: 0.0698 - mean_absolute_error: 0.1494 - val_loss: 0.0557 - val_mean_absolute_error: 0.1387 - 2s/epoch - 84ms/step
Epoch 4/30
27/27 - 2s - loss: 0.0464 - mean_absolute_error: 0.1213 - val_loss: 0.0356 - val_mean_absolute_error: 0.1102 - 2s/epoch - 84ms/step
Epoch 5/30
27/27 - 2s - loss: 0.0303 - mean_absolute_error: 0.0977 - val_loss: 0.0229 - val_mean_absolute_error: 0.0873 - 2s/epoch - 84ms/step
Epoch 6/30
27/27 - 2s - loss: 0.0197 - mean_absolute_error: 0.0796 - val_loss: 0.0147 - val_mean_absolute_error: 0.0705 - 2s/epoch - 85ms/step
Epoch 7/30
27/27 - 2s - loss: 0.0131 - mean_absolute_error: 0.0658 - val_loss: 0.0091 - val_mean_absolute_error: 0.0570 - 2s/epoch - 85ms

##### Results are quite good and gradually decreasing; received these on 2nd attempt.

In [18]:
model.save('Research_cnn1.h5') # Eventually deleted, because using CNN model doesn't work out ultimately for our goal.

In [19]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

16/16 - 1s - loss: 5.5298e-04 - mean_absolute_error: 0.0215 - 933ms/epoch - 58ms/step


In [20]:
for i in test_labels:
    print(i)

2.2085284234469827e-06
7.851470380383595e-06
1.962028750622306e-05
4.68357807277168e-06
4.2657225000703795e-06
1.4660855828911181e-05
9.925806126218425e-06
4.68357807277168e-06
5.698350623952945e-06
1.6900404284117608e-05
4.68357807277168e-06
2.0530721886499472e-06
1.4660855828911181e-05
2.2085284234469827e-06
2.0530721886499472e-06
2.709393187812939e-05
3.014552208693586e-06
3.2757558996611484e-06
2.2085284234469827e-06
2.3800834666950492e-06
1.911869845939113e-06
4.2657225000703795e-06
3.228830343125016e-05
3.228830343125016e-05
7.851470380383595e-06
2.709393187812939e-05
1.962028750622306e-05
1.911869845939113e-06
7.851470380383595e-06
1.7833244808148006e-06
1.4660855828911181e-05
5.1578383563118926e-06
5.1578383563118926e-06
1.911869845939113e-06
1.1241398250993103e-05
1.2800127717989631e-05
2.7803964499161985e-06
7.0287667596710785e-06
2.0530721886499472e-06
4.68357807277168e-06
1.2800127717989631e-05
1.6660491428701165e-06
8.807736649288902e-06
5.698350623952945e-06
1.66604914287

In [21]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

16/16 - 1s - 1s/epoch - 74ms/step


In [22]:
for i in predictions:
    print(i)

[-0.0332998]
[-0.01504482]
[-0.027922]
[-0.00879296]
[-0.00892658]
[-0.02738332]
[-0.02034621]
[-0.00879296]
[-0.01031983]
[-0.02794004]
[-0.00879296]
[-0.0349365]
[-0.02738332]
[-0.0332998]
[-0.0349365]
[-0.02687946]
[-0.02000834]
[-0.01589647]
[-0.03329981]
[-0.02937409]
[-0.03509085]
[-0.00892658]
[-0.02597386]
[-0.02597386]
[-0.01504482]
[-0.02687946]
[-0.027922]
[-0.03509085]
[-0.01504482]
[-0.03542688]
[-0.02738332]
[-0.00931995]
[-0.00931995]
[-0.03509085]
[-0.02302229]
[-0.0254795]
[-0.02376546]
[-0.01276228]
[-0.03493652]
[-0.00879298]
[-0.0254795]
[-0.03623665]
[-0.01730401]
[-0.01031983]
[-0.03623665]
[-0.03509085]
[-0.02738332]
[-0.01730401]
[-0.03623665]
[-0.02000834]
[-0.01276228]
[-0.01589647]
[-0.02302229]
[-0.01276228]
[-0.01031983]
[-0.02570676]
[-0.02937408]
[-0.02376546]
[-0.02937409]
[-0.01276237]
[-0.01031983]
[-0.0254795]
[-0.03542688]
[-0.03542688]
[-0.02794004]
[-0.02376546]
[-0.02302229]
[-0.01504482]
[-0.02687946]
[-0.02000834]
[-0.02687946]
[-0.03623665]
[-0

#### Except a few predictions, all are negative; predictions are higher in value than the actual labels. Nevertheless, patterns are identified; predictions are not all the same.

### Though the predictions from this model were not close in value to the actual test labels and mostly negative, we can consider this particular architecture as showing potential as it did not put forth the problem of giving same prediction for different samples and gave decent losses and errors. So, we decided to continue with this architecture but increase the number of samples generated to put our model under a robust training.

## Architecture - Transfer Learning applied with VGG16 Model - Same architecture (4 Dense Hidden Layers added to the Top, Max Width - 128)<br>No. of samples (4 * 32 * 32 = 4096)
### "Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked

In [6]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [7]:
s, l_max, l_min = datalist_generator(funct, 16, 16, 47, 47)

In [8]:
s1, l_max1, l_min1 = datalist_generator(funct, 48, 48, 79, 79)

In [9]:
s2, l_max2, l_min2 = datalist_generator(funct, 80, 80, 111, 111)

In [10]:
s3, l_max3, l_min3 = datalist_generator(funct, 112, 112, 143, 143)

In [11]:
s_w = np.concatenate((s, s1, s2, s3), dtype = 'float64')
l_max_w = np.concatenate((l_max, l_max1, l_max2, l_max3), dtype = 'float64')

In [15]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s_w, l_max_w, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [25]:
base = VGG16(weights = 'imagenet', include_top = False, input_shape = (33, 33, 3))
base.trainable = False

inp = Input(shape = (33, 33, 3))
p_out = preprocess_input(inp)
temp = base(p_out)

temp = Flatten()(temp)
temp = Dense(units = 128, activation = 'relu')(temp)
temp = Dense(units = 64, activation = 'relu')(temp)
temp = Dense(units = 32, activation = 'relu')(temp)
temp = Dense(units = 8, activation = 'relu')(temp)
out = Dense(units = 1)(temp)

model = Model(inp, out)

In [26]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [27]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
108/108 - 12s - loss: 1712.1510 - mean_absolute_error: 14.9582 - val_loss: 0.5265 - val_mean_absolute_error: 0.5448 - 12s/epoch - 115ms/step
Epoch 2/30
108/108 - 13s - loss: 0.3823 - mean_absolute_error: 0.4102 - val_loss: 0.3147 - val_mean_absolute_error: 0.3390 - 13s/epoch - 120ms/step
Epoch 3/30
108/108 - 12s - loss: 0.2569 - mean_absolute_error: 0.2887 - val_loss: 0.2442 - val_mean_absolute_error: 0.2755 - 12s/epoch - 111ms/step
Epoch 4/30
108/108 - 12s - loss: 0.1985 - mean_absolute_error: 0.2408 - val_loss: 0.1931 - val_mean_absolute_error: 0.2382 - 12s/epoch - 115ms/step
Epoch 5/30
108/108 - 12s - loss: 0.1558 - mean_absolute_error: 0.2070 - val_loss: 0.1459 - val_mean_absolute_error: 0.2001 - 12s/epoch - 112ms/step
Epoch 6/30
108/108 - 12s - loss: 0.1164 - mean_absolute_error: 0.1733 - val_loss: 0.1082 - val_mean_absolute_error: 0.1670 - 12s/epoch - 114ms/step
Epoch 7/30
108/108 - 13s - loss: 0.0868 - mean_absolute_error: 0.1457 - val_loss: 0.0850 - val_mean_absolute

##### Received these results on 2nd attempt; decent results

In [28]:
model.save('Research_cnn2.h5') # Deleted, because CNN failed eventually

In [29]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

62/62 - 4s - loss: 1.3117e-04 - mean_absolute_error: 0.0032 - 4s/epoch - 63ms/step


In [ ]:
for i in test_labels:
    print(i)

##### Output deleted to save memory 

In [31]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

62/62 - 4s - 4s/epoch - 63ms/step


In [32]:
for i in predictions:
    print(i)

[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.04094931]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.07680527]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00895165]
[-0.06441644]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00066646]
[-0.00

#### Predicts the same value for most of the samples; all predictions are negative. So we can conclude that, this architecture doesn't work.

### Unfortunately, the same issue has crept up for this architecture too. Next, we experiment with a slightly deeper and wider network, keeping the dataset size larger, and see if this new architecture withstands failure.

## Architecture - Transfer Learning applied with VGG16 Model - Layers added to the Top (5 Dense Hidden Layers, Max Width - 256)
### No. of samples (5 * 32 * 32 = 5120)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked

In [33]:
funct = exp_generator(1, 2)
funct

-1.30044833451921*x**2 + 1.03920038596194*x + 1.25720304108054

In [34]:
s1, l_max1, l_min1 = datalist_generator(funct, 16, 16, 47, 47)

In [35]:
s2, l_max2, l_min2 = datalist_generator(funct, 48, 48, 79, 79)

In [36]:
s3, l_max3, l_min3 = datalist_generator(funct, 80, 80, 111, 111)

In [37]:
s4, l_max4, l_min4 = datalist_generator(funct, 112, 112, 143, 143)

In [38]:
s5, l_max5, l_min5 = datalist_generator(funct, 144, 144, 175, 175)

In [39]:
s = np.concatenate((s1, s2, s3, s4, s5), dtype = 'float64')
l_max = np.concatenate((l_max1, l_max2, l_max3, l_max4, l_max5), dtype = 'float64')

In [41]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l_max, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [51]:
base = VGG16(weights = 'imagenet', include_top = False, input_shape = (33, 33, 3))
base.trainable = False

inp = Input(shape = (33, 33, 3))
p_out = preprocess_input(inp)
temp = base(p_out)

temp = Flatten()(temp)
temp = Dense(units = 256, activation = 'relu')(temp)
temp = Dense(units = 128, activation = 'relu')(temp)
temp = Dense(units = 64, activation = 'relu')(temp)
temp = Dense(units = 32, activation = 'relu')(temp)
temp = Dense(units = 16, activation = 'relu')(temp)
out = Dense(units = 1)(temp)

model = Model(inp, out)

In [48]:
model.summary()

Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_8 (InputLayer)        [(None, 33, 33, 3)]       0         
                                                                 
 tf.__operators__.getitem_3   (None, 33, 33, 3)        0         
 (SlicingOpLambda)                                               
                                                                 
 tf.nn.bias_add_3 (TFOpLambd  (None, 33, 33, 3)        0         
 a)                                                              
                                                                 
 vgg16 (Functional)          (None, 1, 1, 512)         14714688  
                                                                 
 flatten_3 (Flatten)         (None, 512)               0         
                                                                 
 dense_16 (Dense)            (None, 256)               1313

In [52]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

##### (Output deleted to save memory, as this architecture also failed eventually)<br>Received the results after multiple tries: the results came about the same for all the attempts; no improvement in results in any try; the loss and error weren't low enough and there was continuous oscillation of the results between the epochs.

In [54]:
model.save('Research_cnn2.h5') # Deleted, because CNN failed eventually

In [55]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

77/77 - 6s - loss: 0.0407 - mean_absolute_error: 0.1360 - 6s/epoch - 75ms/step


In [56]:
for i in test_labels:
    print(i)

7.345527872589897e-07
1.8616727702968885e-07
3.502857383484312e-08
7.0287667596710785e-06
3.139965453708325e-08
1.255301125707987e-07
4.597125703624648e-08
3.889978889453968e-05
3.7763966597963686e-08
3.2757558996611484e-06
2.622965161562498e-07
6.981973225972229e-07
5.236137743830211e-07
1.0083382436019164e-07
5.318496072177817e-08
3.0441300031946815e-07
1.6398574516623683e-07
2.7205815835218863e-07
8.021789344366575e-08
1.8026402588674817e-07
7.345527872589897e-07
7.285210326656058e-08
6.061903203842485e-08
3.1968470586255805e-08
1.9233112445571278e-07
1.0926354918682403e-07
2.4413070239461836e-07
3.2757558996611484e-06
8.599789922730065e-07
5.206611131466296e-08
3.705414249478512e-08
2.3800834666950492e-06
3.923894468512524e-08
3.163334882943967e-07
1.6660491428701165e-06
3.636199694112855e-08
4.785023326002774e-07
9.925806126218425e-06
5.698350623952945e-06
4.2657225000703795e-06
1.1409146986677465e-06
6.636119279146072e-08
2.1253664521916422e-07
6.317099213708548e-06
1.01566558029

In [57]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

77/77 - 6s - 6s/epoch - 76ms/step


In [58]:
for i in predictions:
    print(i)

[-0.00974874]
[0.13146754]
[-0.15857153]
[-0.02253798]
[-0.8297427]
[0.1438901]
[-0.05990628]
[0.01761075]
[-0.373791]
[-0.05272035]
[0.05643817]
[-0.02177266]
[0.13106127]
[0.2003171]
[0.03313991]
[0.05767604]
[0.03128596]
[0.06910297]
[0.04941531]
[0.10276767]
[-0.00974874]
[0.03089114]
[0.14491053]
[-0.7102607]
[0.14731188]
[0.15931483]
[0.07707187]
[-0.05272035]
[-0.04281835]
[0.02744075]
[-0.30709293]
[-0.08139352]
[-0.45384625]
[0.06162044]
[-0.02266435]
[-0.1891158]
[-0.00889234]
[0.00509425]
[-0.04633597]
[-0.02092961]
[-0.05529717]
[0.00500651]
[0.10577746]
[-0.04562358]
[-0.03640393]
[0.48409626]
[-0.5083297]
[0.04136439]
[0.48409626]
[0.21376963]
[0.14355631]
[-0.02943829]
[-0.10892705]
[-0.05180005]
[0.10257693]
[0.14375277]
[-0.02257565]
[-0.01835278]
[-0.09244756]
[0.06497165]
[0.10577746]
[-0.05272035]
[0.058521]
[0.16256113]
[0.06388446]
[0.21376963]
[-0.01205663]
[-0.01921395]
[-0.02644948]
[-0.03491715]
[-0.256964]
[-0.02943829]
[0.07707187]
[0.05643817]
[0.14355631]


#### Patterns are identified; different predictions for different samples. But still there were some issues: about half of the predictions are negative; predictions are quite higher in value than actual labels.

In [6]:
funct = exp_generator(1, 3)
funct

-3.96833965769284*x**3 - 1.03941757389319*x**2 - 3.4502772919759*x - 4.3348490432041

In [7]:
s1, l_max1, l_min1 = datalist_generator(funct, 16, 16, 47, 47)

In [8]:
s2, l_max2, l_min2 = datalist_generator(funct, 48, 48, 79, 79)

In [9]:
s3, l_max3, l_min3 = datalist_generator(funct, 80, 80, 111, 111)

In [10]:
s4, l_max4, l_min4 = datalist_generator(funct, 112, 112, 143, 143)

In [11]:
s5, l_max5, l_min5 = datalist_generator(funct, 144, 144, 175, 175)

In [12]:
s = np.concatenate((s1, s2, s3, s4, s5), dtype = 'float64')
l_max = np.concatenate((l_max1, l_max2, l_max3, l_max4, l_max5), dtype = 'float64')

In [13]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l_max, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [15]:
from tensorflow.keras.models import load_model
model = load_model('Research_cnn2.h5')

In [16]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
135/135 - 12s - loss: 7766231.0000 - mean_absolute_error: 990.4601 - val_loss: 1.7076e-05 - val_mean_absolute_error: 0.0041 - 12s/epoch - 91ms/step
Epoch 2/30
135/135 - 12s - loss: 1.7075e-05 - mean_absolute_error: 0.0041 - val_loss: 1.7074e-05 - val_mean_absolute_error: 0.0041 - 12s/epoch - 86ms/step
Epoch 3/30
135/135 - 12s - loss: 1.7074e-05 - mean_absolute_error: 0.0041 - val_loss: 1.7074e-05 - val_mean_absolute_error: 0.0041 - 12s/epoch - 87ms/step
Epoch 4/30
135/135 - 12s - loss: 1.7073e-05 - mean_absolute_error: 0.0041 - val_loss: 1.7073e-05 - val_mean_absolute_error: 0.0041 - 12s/epoch - 89ms/step
Epoch 5/30
135/135 - 12s - loss: 1.7072e-05 - mean_absolute_error: 0.0041 - val_loss: 1.7072e-05 - val_mean_absolute_error: 0.0041 - 12s/epoch - 89ms/step
Epoch 6/30
135/135 - 12s - loss: 1.7071e-05 - mean_absolute_error: 0.0041 - val_loss: 1.7071e-05 - val_mean_absolute_error: 0.0041 - 12s/epoch - 90ms/step
Epoch 7/30
135/135 - 12s - loss: 1.7070e-05 - mean_absolute_error:

##### The loss is at a low value and decreases gradually but at a much slower rate; however, there is no change in error.

In [17]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

77/77 - 5s - loss: 1.7014e-05 - mean_absolute_error: 0.0041 - 5s/epoch - 61ms/step


In [ ]:
for i in test_labels:
    print(i)

##### Output deleted here as well to save memory, as the architecture fails to perform

In [19]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

77/77 - 5s - 5s/epoch - 63ms/step


In [ ]:
for i in predictions:
    print(i)

#### Predicting the same value for all samples; all predictions are negative; the predictions are quite high valued in comparison to the actual labels.

### Even though the current architecture didn't fail when working with quadratic samples, when it was trained on cubic samples with much smaller curvature values, it again gave the same problem of predicting the same value for all samples. Based on these observations, we came up with a hypothesis: since the curvature values for polynomial surfaces are very small in value, our models might have been facing the issue with vanishing gradients, which could have caused the models in some way to break into predicting the same value for all samples.
### So, to test our hypothesis, we decide to train and test our model next on those types of expressions that have high valued curvatures. We first chose Sine Function samples and used the regular MLP architecture with a depth of 16 layers and a width of 36 nodes, which we had found in earlier experiments to have provided the best performance among all the architectures tested.

## Architecture - Depth(16 layers in total), Width (36 nodes highest among the layers)<br>Training first on Sine Wave samples (High valued curvatures)
### No. of samples (128 x 128)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked

In [8]:
funct = exp_generator(3, 1)
funct

0.593988879915532*sin(2.01369540968675*x - 4.57047002658068) - 0.607193781823679

In [9]:
s, l_max, l_min = datalist_generator(funct, 1, 1, 126, 126)

In [10]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l_max, test_size = 0.3)

In [ ]:
for i in train_labels:
    print(i)

In [12]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [13]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [14]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 7s - loss: 0.8187 - mean_absolute_error: 0.7140 - val_loss: 0.4414 - val_mean_absolute_error: 0.5727 - 7s/epoch - 17ms/step
Epoch 2/30
417/417 - 2s - loss: 0.1888 - mean_absolute_error: 0.3248 - val_loss: 0.0160 - val_mean_absolute_error: 0.1021 - 2s/epoch - 4ms/step
Epoch 3/30
417/417 - 2s - loss: 0.0073 - mean_absolute_error: 0.0677 - val_loss: 0.0024 - val_mean_absolute_error: 0.0404 - 2s/epoch - 5ms/step
Epoch 4/30
417/417 - 2s - loss: 0.0015 - mean_absolute_error: 0.0299 - val_loss: 7.8324e-04 - val_mean_absolute_error: 0.0217 - 2s/epoch - 5ms/step
Epoch 5/30
417/417 - 2s - loss: 6.3282e-04 - mean_absolute_error: 0.0186 - val_loss: 6.0640e-04 - val_mean_absolute_error: 0.0181 - 2s/epoch - 4ms/step
Epoch 6/30
417/417 - 2s - loss: 2.9906e-04 - mean_absolute_error: 0.0126 - val_loss: 2.1622e-04 - val_mean_absolute_error: 0.0110 - 2s/epoch - 4ms/step
Epoch 7/30
417/417 - 2s - loss: 1.8051e-04 - mean_absolute_error: 0.0099 - val_loss: 1.3886e-04 - val_mean_absolute

##### The metrics do not go down smoothly, but has a general downward trend. Both the loss and the error values are significantly low: the loss goes down to a pretty decent value, while the error is low enough, even if it seems high compared to some other experiments and this can be attributed to the high valued curvatures for this experiment.

In [15]:
model.save('Research_dnn_sin.h5')

# Deleted, because this model is trained only on sine function samples, which isn't our research goal.

In [16]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 1s - loss: 1.6815e-05 - mean_absolute_error: 0.0029 - 584ms/epoch - 2ms/step


In [17]:
for i in test_labels:
    print(i)

0.6438758583218204
2.312504891702772
1.7448054274766782
0.09121879825813574
2.3038528271841914
2.131973399314668
0.7773553871680068
0.05335861564470971
0.29198858221514434
0.652439943471833
1.084504328072518
0.3509385613655042
1.9897550938814967
0.2706262034240012
2.3863473043443904
0.044821792108735846
0.38122063277132173
0.45212709162163045
0.06441750644817072
0.37542165010302037
1.31131220582084
0.34542051614909947
0.38122063277132173
0.45212709162163045
0.22769284063567924
0.04848338866354726
0.6727361813732983
1.1104570693733495
2.157632584649323
1.5154408182060406
0.7577213628501995
0.6727361813732983
2.331335715057423
0.5366664560429131
2.1982866505352483
0.3509385613655042
0.17350438753752273
0.17350438753752273
0.2706262034240012
2.1449126730981924
0.9711488571467721
2.331335715057423
0.16522466048792264
1.7289167798067884
0.6354018266976316
0.2706262034240012
0.20842051281282897
0.044821792108735846
0.7873259312392566
0.5540855635598655
1.282357521290466
2.3038528271841914
0.

In [18]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

239/239 - 1s - 868ms/epoch - 4ms/step


In [19]:
for i in predictions:
    print(i)

[0.63943267]
[2.3136735]
[1.7455995]
[0.09053181]
[2.2907114]
[2.1359117]
[0.77585876]
[0.04974179]
[0.29391462]
[0.6571333]
[1.0874226]
[0.3519669]
[1.9801117]
[0.26885346]
[2.385714]
[0.04516065]
[0.37991112]
[0.45221993]
[0.06260268]
[0.37639028]
[1.3099616]
[0.34633723]
[0.37991112]
[0.45221993]
[0.22413395]
[0.04901462]
[0.67029476]
[1.11217]
[2.1558313]
[1.503423]
[0.7585141]
[0.67029476]
[2.3238113]
[0.53562796]
[2.1939502]
[0.3519669]
[0.17520836]
[0.17520836]
[0.2688534]
[2.1459868]
[0.965963]
[2.3238113]
[0.16789216]
[1.7083669]
[0.6399362]
[0.26885346]
[0.20684674]
[0.04516065]
[0.78306794]
[0.5529563]
[1.2819673]
[2.2907114]
[0.6795675]
[0.20684674]
[1.7627177]
[0.7585141]
[0.5274609]
[1.5790824]
[0.639936]
[0.05559212]
[0.44015166]
[0.06260255]
[0.20684674]
[0.09053181]
[2.3370397]
[0.6399362]
[2.385714]
[2.3912752]
[0.5274609]
[0.06260255]
[1.2819673]
[2.1844597]
[0.9827797]
[1.2707984]
[1.9801117]
[0.16789216]
[0.1445064]
[0.0076905]
[0.10810354]
[0.435082]
[1.1639764]
[

[0.43508205]
[1.341859]
[0.2154416]
[1.5634567]
[2.1844597]
[0.8081334]
[2.3996472]
[1.3253585]
[1.7627177]
[2.393294]
[0.37991112]
[0.37991112]
[1.5790824]
[2.2907114]
[0.639936]
[0.5274609]
[0.0076905]
[0.22060667]
[0.21518673]
[1.9554316]
[1.5790824]
[0.95832485]
[1.3554542]
[1.1370482]
[0.28015792]
[0.15206617]
[1.7455995]
[0.27889085]
[0.965963]
[2.2964885]
[0.15206617]
[2.2907114]
[2.385714]
[1.1121705]
[0.20684674]
[0.43508205]
[1.11217]
[0.28015792]
[1.4744533]
[1.7943672]
[0.05559215]
[2.3996472]
[2.393294]
[1.5634567]
[0.47331938]
[0.22413395]
[2.0008307]
[1.4744533]
[1.503423]
[2.009128]
[1.2707984]
[0.22413395]
[1.1370482]
[0.1150557]
[0.37991115]
[0.90732366]
[0.53562796]
[1.9554316]
[1.1639764]
[0.20684674]
[0.17520836]
[0.05559215]
[1.503423]
[0.79903656]
[0.79903656]
[0.29391462]
[0.15206617]
[1.5634567]
[0.26885346]
[0.53562796]
[1.7132899]
[0.04516065]
[0.27889085]
[2.1844597]
[0.0076905]
[1.4744533]
[1.2819673]
[1.7943672]
[1.1370482]
[0.4243621]
[1.2707984]
[1.15196

[2.3996472]
[1.5005301]
[0.95832485]
[2.1459873]
[0.5602319]
[0.6795675]
[0.46440932]
[0.26885346]
[1.9554316]
[0.29598317]
[0.15206617]
[2.2964885]
[2.009128]
[0.79903656]
[1.7455995]
[2.2907114]
[0.6394326]
[0.9827798]
[0.9827797]
[1.7627177]
[0.965963]
[0.0076905]
[0.04901462]
[0.47331938]
[1.7455995]
[0.9221317]
[1.9554316]
[2.2964885]
[0.5529563]
[0.2154416]
[2.385714]
[0.5602319]
[1.1519686]
[0.78306794]
[0.90732366]
[2.3208816]
[0.9659629]
[0.14450638]
[2.3912752]
[2.4008377]
[0.15206617]
[1.7132899]
[2.3370397]
[0.37639025]
[0.35768098]
[1.7943672]
[0.27889085]
[0.1166991]
[0.1445064]
[0.6399362]
[2.2907114]
[0.22060667]
[2.4008377]
[2.2907114]
[0.2154416]
[2.384939]
[0.9073238]
[0.4401517]
[0.30471298]
[0.2154416]
[2.1939502]
[0.28015792]
[1.341859]
[2.1459873]
[0.82039833]
[2.384939]
[0.17520836]
[0.95832485]
[0.5206961]
[1.5790824]
[2.1558313]
[1.5491023]
[2.385714]
[0.06260255]
[1.7083669]
[0.22060667]
[1.5491021]
[2.323811]
[0.46440932]
[1.1519686]
[0.67029476]
[0.56918496

#### The predicted outputs are negligibly different than the actual labels, and all predictions are positive.

### This experiment has yielded the best performance among all the experiments conducted. So, we can say that our hypothesis seems to be proven true.
### Next, we train and test our model on a combination of samples that has both high and low valued curvatures. This is done for two reasons. One is to test our hypothesis to be true with more confidence. Secondly, our model needs to be trained on samples of varied types of surfaces to be effectively employed for real-life use, because real-life surfaces are comprised of both types of surfaces having high and low valued curvatures. We use the same architecture and have a greater percentage of samples with high valued curvatures in our combination, so that our model doesn't face the same problem as earlier. For the first experiment, we use a combination of Sine Wave, Parabolic Cylinder (Univariate Quadratic Polynomial), and Circular Paraboloid samples.

## First combination of samples used - Sine Wave Surface, Parabolic Cylinder (Quadratic Univariate Polynomial), Circular Paraboloid (z = x^2 + y^2)
### Architecture - Depth(16 layers in total), Width (36 nodes highest among the layers)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked

In [2]:
import sympy
import numpy as np
import tensorflow as tf
from tensorflow import keras
from random import uniform, seed
from sklearn.utils import shuffle
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

In [4]:
x = sympy.Symbol('x')
y = sympy.Symbol('y')

In [5]:
# Function for generating input sample and unsigned labels for a particular point of interest

def data_generator(exp, center_x, center_y, ss):  # A parameter (ss) representing step size has been added, which is utilized
                                                  # in the later experiments
    l_sample = []
    
    i = center_x - ss
    j = center_y - ss
    
    while (i < center_x+ss+ss):
        while(j < center_y+ss+ss):
            value = exp.evalf(subs={x: i, y: j})
            l_sample.append(value)
            
            j = j + ss
        
        j = center_y - ss
        i = i + ss
    
    fx = exp.diff(x, 1)
    fy = exp.diff(y, 1)
    fxx = exp.diff(x, 2)
    fyy = exp.diff(y, 2)
    fxy = exp.diff(x, y, 1)
    
    v_fx = fx.evalf(subs={x: center_x, y: center_y})
    v_fy = fy.evalf(subs={x: center_x, y: center_y})
    v_fxx = fxx.evalf(subs={x: center_x, y: center_y})
    v_fyy = fyy.evalf(subs={x: center_x, y: center_y})
    v_fxy = fxy.evalf(subs={x: center_x, y: center_y})
    
    K = (v_fxx*v_fyy - v_fxy**2) / (1 + v_fx**2 + v_fy**2)**2
    H = (v_fxx + v_fyy + v_fxx*v_fy**2 + v_fyy*v_fx**2 - 2*v_fx*v_fy*v_fxy) / (2 * (1 + v_fx**2 + v_fy**2)**1.5)
    
    k1 = H + (H**2 - K)**0.5
    k2 = H - (H**2 - K)**0.5
    
    #*********************************************
    # Changes made to create unsigned labels
    
    u_k1 = abs(k1)
    u_k2 = abs(k2)
    
    if(u_k1 < u_k2):
        temp = u_k1
        u_k1 = u_k2
        u_k2 = temp
    
    #*********************************************
    
    #*********************************************
    # Changes made to split labels into two groups
    
    l_label_max = u_k1
    l_label_min = u_k2
    
    return l_sample, l_label_max, l_label_min

    #*********************************************

In [6]:
# Function for generating a list of samples and labels for a range of points

def datalist_generator(exp, x_start, y_start, x_end, y_end, ss):
    l_samples = []
    l_labels_max = []
    l_labels_min = []
    
    i = x_start
    j = y_start
    
    while (i < x_end+ss):
        while(j < y_end+ss):
            
            t_sample, t_label_max, t_label_min = data_generator(exp, i, j, ss)
            
            if(t_label_max >= 0.0002):          # Changed the limit from 10^-6 to 10^-5 to 10^-4 as we continued with different
                                                # experiments, since lower values were disrupting results
                l_samples.append(t_sample)
                l_labels_max.append(t_label_max)
                l_labels_min.append(t_label_min)
            
            j = j + ss
        
        j = y_start
        i = i + ss
        
    samples = np.array(l_samples, dtype = 'float64')
    labels_max = np.array(l_labels_max, dtype = 'float64')
    labels_min = np.array(l_labels_min, dtype = 'float64')
    
    return samples, labels_max, labels_min

In [7]:
# Function for generating expressions

def exp_generator(num, h_deg):
    
    if (num == 1):
        
        seed(num + h_deg**4)
        
        # The constraints on the coefficient values ensured the generation of expressions with comparatively higher curvature
        
        a = uniform(-1.0, 1.0)
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 1.0)
        d = uniform(-0.5, 0.5)
        e = uniform(-3.0, 3.0)
        
        if(h_deg == 3):
            a = 0.0
            b = uniform(-0.5, 0.5)
            c = uniform(-2.0, 2.0)
            d = uniform(-5.0, 0.5)
        elif(h_deg == 2):
            a = 0.0
            b = 0.0
            c = uniform(-0.5, 0.5)
        
        f = a*x**4 + b*x**3 + c*x**2 + d*x + e
        
        return f
    
    elif (num == 2):
        
        seed(num + h_deg)
        
        a = uniform(-5.0, 5.0)
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-5.0, 5.0)
        e = uniform(-5.0, 5.0)
        f = uniform(-5.0, 5.0)
        g = uniform(-5.0, 5.0)
        h = uniform(-5.0, 5.0)
        i = uniform(-5.0, 5.0)
        j = uniform(-5.0, 5.0)
        k = uniform(-5.0, 5.0)
        l = uniform(-5.0, 5.0)
        m = uniform(-5.0, 5.0)
        n = uniform(-5.0, 5.0)
        o = uniform(-5.0, 5.0)
        
        if(h_deg == 3):
            a = 0.0
            b = 0.0
            c = 0.0
            d = 0.0
            e = 0.0
        elif(h_deg == 2):
            a = 0.0
            b = 0.0
            c = 0.0
            d = 0.0
            e = 0.0
            f = 0.0
            g = 0.0
            h = 0.0
            i = 0.0
        
        f = a*x**4 + b*y**4 + c*x**3*y + d*x*y**3 + e*x**2*y**2 + f*x**3 + g*y**3 + h*x**2*y + i*x*y**2 + j*x**2 + k*y**2 + l*x*y + m*x + n*y + o
        
        return f
    
    elif (num == 3):
        
        seed(num**num)
        
        a = uniform(-2.0, 2.0)
        
        while(a == 0):
            a = uniform(-2.0, 2.0)
        
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-1.0, 1.0)
        
        f = a*sympy.sin(b*x-c) + d
        
        return f
    
    elif (num == 4):
        
        seed(num**num)
        
        a = uniform(-2.0, 2.0)
        
        while(a == 0):
            a = uniform(-2.0, 2.0)
        
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-1.0, 1.0)
        
        f = a*sympy.cos(b*x-c) + d
        
        return f
    
    elif (num == 5):
        
        f = x**2 + y**2
        
        return f
        
    elif (num == 6):
        
        seed(num*4)
        
        a = uniform(0.0, 3.0)
        b = uniform(0.0, 3.0)
        
        f = a*x**2 + b*y**2
        
        return f
    
    elif (num == 7):
        
        seed(num**h_deg)
        
        a = uniform(-2.0, 2.0)
        
        while(a == 0):
            a = uniform(-2.0, 2.0)
        
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-2.0, 2.0)
        e = uniform(-1.0, 1.0)
        g = uniform(-5.0, 5.0)
        h = uniform(-3.0, 3.0)
        
        f = a*(sympy.sin(b*x-c))**2 + d*sympy.sin(e*x-g) + h
        
        return f
    
    else:
        f = 0
        return f

##### Sinusoidal Surface and Paraboloid samples were in a greater percentage in the combination, as these have high curvatures

In [6]:
funct = exp_generator(3, 1)
funct

0.593988879915532*sin(2.01369540968675*x - 4.57047002658068) - 0.607193781823679

In [7]:
s1, l_max1, l_min1 = datalist_generator(funct, 1, 1, 96, 96)

In [8]:
funct1 = exp_generator(5, 1)
funct1

x**2 + y**2

In [9]:
s2, l_max2, l_min2 = datalist_generator(funct1, 10, 10, 105, 105)

In [10]:
s = np.concatenate((s1, s2))
l_max = np.concatenate((l_max1, l_max2))

s, l_max = shuffle(s, l_max)

In [11]:
train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s, l_max, test_size = 0.3)

In [12]:
funct2 = exp_generator(1, 2)
funct2

0.204219866843413*x**2 - 2.10374622235534*x + 2.66107437797953

In [13]:
s3, l_max3, l_min3 = datalist_generator(funct2, -10, -10, 45, 45) # This range of coordinates have higher curvature values

In [14]:
train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s3, l_max3, test_size = 0.3)

In [15]:
train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [16]:
train_samples.shape

(15097, 9)

In [17]:
for i in train_labels:
    print(i)

In [21]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [22]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [23]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
567/567 - 4s - loss: 0.8210 - mean_absolute_error: 0.4072 - val_loss: 0.3058 - val_mean_absolute_error: 0.3217 - 4s/epoch - 7ms/step
Epoch 2/30
567/567 - 2s - loss: 0.3001 - mean_absolute_error: 0.3279 - val_loss: 0.2648 - val_mean_absolute_error: 0.3055 - 2s/epoch - 3ms/step
Epoch 3/30
567/567 - 2s - loss: 0.2176 - mean_absolute_error: 0.2811 - val_loss: 0.1007 - val_mean_absolute_error: 0.1956 - 2s/epoch - 3ms/step
Epoch 4/30
567/567 - 2s - loss: 0.0271 - mean_absolute_error: 0.0859 - val_loss: 0.0075 - val_mean_absolute_error: 0.0502 - 2s/epoch - 3ms/step
Epoch 5/30
567/567 - 2s - loss: 0.0039 - mean_absolute_error: 0.0374 - val_loss: 0.0028 - val_mean_absolute_error: 0.0295 - 2s/epoch - 3ms/step
Epoch 6/30
567/567 - 1s - loss: 0.0024 - mean_absolute_error: 0.0308 - val_loss: 0.0018 - val_mean_absolute_error: 0.0231 - 1s/epoch - 2ms/step
Epoch 7/30
567/567 - 2s - loss: 0.0051 - mean_absolute_error: 0.0311 - val_loss: 0.0013 - val_mean_absolute_error: 0.0205 - 2s/epoch - 3

##### The results are lower in value, but they are oscillating instead of converging to a minimum value.

In [24]:
model.save('Research_comb.h5')

In [25]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

324/324 - 0s - loss: 3.5016e-05 - mean_absolute_error: 0.0033 - 496ms/epoch - 2ms/step


In [26]:
for i in test_labels:
    print(i)

2.3863473043443904
0.01625371782824498
2.3863473043443904
0.013748708433955138
0.009504754457434086
0.007744021505457482
0.9096141284141153
1.7289167798067884
0.009723758093313231
2.338842482648719
1.5833225832642632
0.4307769307365432
0.22769284063567924
0.008867544887757461
0.010577273914566329
0.011230108301861795
0.009990887470879062
1.7289167798067884
0.014589215480012886
0.009001979527896976
0.0007493265734755108
0.0007493265734755108
2.2948705367927276
0.005734127603511738
0.7773553871680068
0.9711488571467721
0.008217104407757869
0.017248430868424924
0.11540148396937845
1.7130168624890667
0.0033839974298701015
0.011577727121044396
0.0008728880004371243
0.09435816580857573
0.0002231377254814533
1.7289167798067884
0.04116645244171622
0.013621841881321942
0.009955178160675858
0.008591924593476546
0.16522466048792264
0.0005640296312052273
0.007391960256197651
0.1693525197521385
0.20842051281282897
0.4307769307365432
0.02192513290610496
0.569389837324733
0.009750994271208447
0.00946

In [27]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

324/324 - 1s - 638ms/epoch - 2ms/step


In [28]:
for i in predictions:
    print(i)

[2.3874526]
[0.01647545]
[2.3874526]
[0.0145997]
[0.00654878]
[0.01590075]
[0.909717]
[1.7362502]
[0.00740232]
[2.3398547]
[1.5887611]
[0.43035534]
[0.22726479]
[0.00434913]
[0.00905313]
[0.01061763]
[0.00751008]
[1.7362502]
[0.01538028]
[0.00427617]
[0.00278876]
[0.00278876]
[2.2961252]
[0.00743173]
[0.783377]
[0.9733787]
[0.00119437]
[0.01675071]
[0.11471069]
[1.7133273]
[0.01216585]
[0.0114633]
[0.00353272]
[0.09436444]
[0.00113241]
[1.7362502]
[0.03414512]
[0.01434602]
[0.00808086]
[0.00305976]
[0.16654152]
[0.00144784]
[-0.00359021]
[0.16921204]
[0.21354805]
[0.43035534]
[0.01889224]
[0.5689111]
[0.00721778]
[0.00642718]
[0.6358055]
[0.00743173]
[0.00536431]
[0.0081438]
[0.47101852]
[0.00214233]
[0.00113241]
[1.9434433]
[0.00700913]
[0.14822927]
[0.36933568]
[0.00720187]
[0.00734367]
[2.0042295]
[0.01737632]
[0.01564206]
[0.3552659]
[1.1430202]
[0.01374139]
[1.7362502]
[2.1474257]
[0.0194089]
[0.22726479]
[0.27342182]
[0.06611927]
[0.01563562]
[0.014403]
[0.06611927]
[0.537442]
[0

[0.00763644]
[0.01219048]
[0.00098504]
[0.00800265]
[0.01056792]
[2.1303344]
[0.00377213]
[0.00756826]
[0.01108445]
[0.45920187]
[2.0042295]
[0.01313628]
[0.783377]
[0.81504697]
[0.08673601]
[0.01224847]
[0.36933586]
[0.14822927]
[1.349362]
[-0.005468]
[0.01041736]
[0.01896609]
[0.00377213]
[0.01128449]
[0.01147093]
[0.00845475]
[1.2697004]
[2.3398547]
[1.2697004]
[0.98563385]
[0.05737863]
[0.00048293]
[0.00765409]
[2.19756]
[0.4241971]
[0.00377213]
[0.00792493]
[0.01967736]
[-0.00382958]
[0.0019268]
[0.01230754]
[1.2697004]
[0.6797261]
[2.4111197]
[2.3484433]
[0.0060357]
[1.1712896]
[0.01098956]
[0.00878562]
[0.01042737]
[0.00555553]
[0.23501527]
[0.11471069]
[0.018057]
[0.01170959]
[0.43035534]
[0.01005592]
[0.36933586]
[0.00933351]
[0.22726479]
[0.05737863]
[0.7647987]
[0.00720187]
[0.3043721]
[0.529827]
[0.14822927]
[-0.00438557]
[0.01064433]
[0.0648349]
[0.6797261]
[0.00129866]
[1.0963225]
[0.15301305]
[0.01172961]
[0.01326336]
[0.00953139]
[0.00698747]
[0.30437222]
[1.9548234]
[0

#### Predictions are positive and almost similar to the actual labels.

### *We further experiment with another combination of samples of different surface types to test our model's performance even more.*

### Second Combination of samples used - Cosine Wave Surface, Cubic Cylinder (Univariate Cubic Polynomial), Elliptic Paraboloid (z = ax^2 + by^2)
#### Same architecture used
##### Similar to the earlier experiment, Cosine Wave Surface and Paraboloid samples were in a greater percentage in the combination due to them having higher curvatures.

In [30]:
funct = exp_generator(4, 1)
funct

-0.0445981827274462*cos(0.677404823452205*x - 1.24950596636108) - 0.920229845386649

In [31]:
s1, l_max1, l_min1 = datalist_generator(funct, 1, 1, 96, 96)

In [32]:
funct1 = exp_generator(6, 1)
funct1

2.13702896348076*x**2 + 2.51939920893562*y**2

In [33]:
s2, l_max2, l_min2 = datalist_generator(funct1, 1, 1, 96, 96)

In [34]:
s = np.concatenate((s1, s2))
l_max = np.concatenate((l_max1, l_max2))

s, l_max = shuffle(s, l_max)

In [35]:
train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s, l_max, test_size = 0.3)

In [36]:
funct2 = exp_generator(1, 3)
funct2

0.290167489401872*x**3 + 0.720178969521534*x**2 - 4.13877058255077*x + 3.11987259832019

In [37]:
s3, l_max3, l_min3 = datalist_generator(funct2, -14, -14, 13, 13) # This range of coordinates have higher curvature values

In [38]:
train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s3, l_max3, test_size = 0.3)

In [39]:
train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [46]:
train_samples.shape

(13450, 9)

In [40]:
for i in train_labels:
    print(i)

In [41]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
505/505 - 1s - loss: 0.0713 - mean_absolute_error: 0.0267 - val_loss: 0.0143 - val_mean_absolute_error: 0.0142 - 1s/epoch - 3ms/step
Epoch 2/30
505/505 - 1s - loss: 0.0067 - mean_absolute_error: 0.0110 - val_loss: 0.0142 - val_mean_absolute_error: 0.0133 - 1s/epoch - 3ms/step
Epoch 3/30
505/505 - 2s - loss: 0.0066 - mean_absolute_error: 0.0106 - val_loss: 0.0142 - val_mean_absolute_error: 0.0123 - 2s/epoch - 4ms/step
Epoch 4/30
505/505 - 2s - loss: 0.0066 - mean_absolute_error: 0.0103 - val_loss: 0.0141 - val_mean_absolute_error: 0.0139 - 2s/epoch - 4ms/step
Epoch 5/30
505/505 - 2s - loss: 0.0066 - mean_absolute_error: 0.0106 - val_loss: 0.0140 - val_mean_absolute_error: 0.0122 - 2s/epoch - 4ms/step
Epoch 6/30
505/505 - 2s - loss: 0.0065 - mean_absolute_error: 0.0100 - val_loss: 0.0139 - val_mean_absolute_error: 0.0126 - 2s/epoch - 4ms/step
Epoch 7/30
505/505 - 2s - loss: 0.0065 - mean_absolute_error: 0.0096 - val_loss: 0.0138 - val_mean_absolute_error: 0.0131 - 2s/epoch - 4

##### The results are a bit higher. There are minor fluctuations, but the general trend is downward.

In [47]:
model.save('Research_comb.h5')

In [42]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

289/289 - 0s - loss: 0.0041 - mean_absolute_error: 0.0117 - 438ms/epoch - 2ms/step


In [43]:
for i in test_labels:
    print(i)

0.020451394702413243
0.006178782515920178
0.020000472256625018
0.00019420088912512864
0.026448926122890543
0.008687880629639995
0.003734690395919941
0.011636135796647403
0.007668240906576532
0.011230228743119984
0.04253722253342883
0.008002804083764404
0.01719943746369718
0.01876794778807358
0.014703504925368413
0.009831298472251247
0.007660801569752544
0.011720904487952606
0.0105578298749677
0.013624944249189131
0.014703504925368413
0.008609872724648138
0.013775023025413748
0.010313289212213334
0.017040222948463406
0.010730296241137362
0.003734690395919941
0.016274618897087888
0.016396375167229114
0.015646930643003588
0.019733839076718364
0.013775023025413748
0.025130455811586506
0.009827206740307515
0.013624944249189131
3.425582193912699e-05
0.020371723343127607
0.01980859642843908
0.009093463768777626
0.012176470927816
0.009764492947146333
0.020189110602626743
0.02023467086304257
0.007315144084225689
0.009280507360248044
0.014569268872087664
0.008084758042404466
0.07727624675953194


In [44]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

289/289 - 0s - 373ms/epoch - 1ms/step


In [45]:
for i in predictions:
    print(i)

[0.01205719]
[0.01239631]
[0.01263148]
[0.04742045]
[0.0234201]
[-0.01032501]
[0.01225949]
[0.00087494]
[-0.0206781]
[0.00150198]
[0.05077697]
[-0.01839977]
[0.01208604]
[0.01209358]
[0.01260705]
[-0.00247055]
[-0.02015357]
[0.00683922]
[0.01246131]
[0.0125098]
[0.01260705]
[0.01226459]
[0.01218025]
[-0.00374513]
[0.01256851]
[0.0122304]
[0.01225949]
[0.0126268]
[0.0120957]
[0.012619]
[0.01205937]
[0.01218025]
[0.02202231]
[0.00024409]
[0.01250974]
[0.03709794]
[0.01264442]
[0.01781709]
[-0.00464778]
[0.00633139]
[0.01254106]
[0.01263729]
[0.01205508]
[0.01228533]
[-0.00354772]
[0.00920481]
[-0.01614242]
[0.03709794]
[0.07286591]
[-0.00263362]
[-0.01756053]
[0.0120799]
[0.01213942]
[0.00368924]
[0.01256851]
[0.01225016]
[0.03709794]
[0.0027184]
[0.01510448]
[0.01573152]
[0.01210637]
[0.01261193]
[0.09852333]
[0.01205514]
[0.01225016]
[-0.00103908]
[0.012619]
[0.00849719]
[0.01232488]
[0.0125917]
[0.01244802]
[0.01213942]
[0.01232175]
[0.01234395]
[0.01256851]
[0.012619]
[0.01252357]
[-

[0.01252664]
[0.01246131]
[0.01231147]
[0.0121494]
[0.0019073]
[0.0195945]
[0.01246131]
[0.01236317]
[0.0002665]
[0.01264397]
[0.06315473]
[0.01206414]
[0.01395196]
[0.00648732]
[0.00936551]
[0.01544256]
[-0.00138002]
[0.01232175]
[0.012063]
[0.01234335]
[0.00854106]
[0.01240227]
[0.01404304]
[0.01221121]
[0.012063]
[0.06117848]
[0.01211337]
[0.01246131]
[0.02551711]
[-0.00035911]
[0.01264442]
[0.00355525]
[0.00588364]
[0.01264397]
[0.01280589]
[0.01231147]
[0.01216433]
[0.01264853]
[0.01213942]
[0.01206804]
[0.0125576]
[0.01211423]
[0.01096053]
[0.01473755]
[0.01238055]
[0.018053]
[0.02012975]
[0.01237697]
[0.01206414]
[0.04811684]
[0.00405545]
[0.012063]
[0.00747008]
[-0.00802856]
[0.01255379]
[-0.01127296]
[0.01266066]
[0.0124974]
[0.0120799]
[-0.00274473]
[0.01237697]
[-0.01484256]
[0.00705285]
[-0.00478034]
[-0.00447565]
[0.01216433]
[-0.00411802]
[0.01207254]
[0.0125576]
[0.01464052]
[0.01226459]
[0.01238055]
[0.0124974]
[0.01219804]
[0.01256893]
[0.01251144]
[0.01290173]
[0.0088

#### There are a few negative predictions; significant difference between the predictions and the actual labels, but the difference is not so large; different predictions for different samples.

### The experiment, thus, shows that our model performed well on the first combination, but slightly struggled with the second combination. In order to understand it better how our model performs on the 2nd dataset, we next train and test our model only with this dataset and check out the results.

## Combination of samples used - Cosine Wave Surface, Cubic Cylinder (Univariate Cubic Polynomial), Elliptic Paraboloid (z = ax^2 + by^2)<br>Training & Testing from scratch on Max Valued Curvatures - exclusively on this 2nd dataset
### Architecture - Depth(16 layers in total), Width (36 nodes highest among the layers)<br>"Unsigned" & "Split" Labels<br>Seeded expressions used & predictions checked

In [6]:
funct = exp_generator(4, 1)
funct

-0.0445981827274462*cos(0.677404823452205*x - 1.24950596636108) - 0.920229845386649

In [7]:
s1, l_max1, l_min1 = datalist_generator(funct, 1, 1, 96, 96)

In [8]:
funct1 = exp_generator(6, 1)
funct1

2.13702896348076*x**2 + 2.51939920893562*y**2

In [9]:
s2, l_max2, l_min2 = datalist_generator(funct1, 1, 1, 96, 96)

In [10]:
s = np.concatenate((s1, s2))
l_max = np.concatenate((l_max1, l_max2))

s, l_max = shuffle(s, l_max)

In [11]:
train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s, l_max, test_size = 0.3)

In [12]:
funct2 = exp_generator(1, 3)
funct2

0.290167489401872*x**3 + 0.720178969521534*x**2 - 4.13877058255077*x + 3.11987259832019

In [13]:
s3, l_max3, l_min3 = datalist_generator(funct2, -14, -14, 13, 13)

In [14]:
train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s3, l_max3, test_size = 0.3)

In [15]:
train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [16]:
train_samples.shape

(13450, 9)

In [ ]:
for i in train_labels:
    print(i)

In [21]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [22]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

##### Retrained the model from scratch a 2nd time, as the validation loss and error were higher than the training metrics

In [23]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
505/505 - 4s - loss: 42.1269 - mean_absolute_error: 1.3355 - val_loss: 0.0678 - val_mean_absolute_error: 0.1516 - 4s/epoch - 8ms/step
Epoch 2/30
505/505 - 2s - loss: 0.0329 - mean_absolute_error: 0.1058 - val_loss: 0.0258 - val_mean_absolute_error: 0.0709 - 2s/epoch - 4ms/step
Epoch 3/30
505/505 - 2s - loss: 0.0103 - mean_absolute_error: 0.0490 - val_loss: 0.0179 - val_mean_absolute_error: 0.0389 - 2s/epoch - 4ms/step
Epoch 4/30
505/505 - 2s - loss: 0.0063 - mean_absolute_error: 0.0276 - val_loss: 0.0158 - val_mean_absolute_error: 0.0253 - 2s/epoch - 4ms/step
Epoch 5/30
505/505 - 2s - loss: 0.0052 - mean_absolute_error: 0.0181 - val_loss: 0.0150 - val_mean_absolute_error: 0.0194 - 2s/epoch - 5ms/step
Epoch 6/30
505/505 - 2s - loss: 0.0048 - mean_absolute_error: 0.0138 - val_loss: 0.0145 - val_mean_absolute_error: 0.0169 - 2s/epoch - 4ms/step
Epoch 7/30
505/505 - 2s - loss: 0.0046 - mean_absolute_error: 0.0119 - val_loss: 0.0141 - val_mean_absolute_error: 0.0154 - 2s/epoch - 

##### The results are quite good.

In [24]:
model.save('Research_comb_t.h5') # Deleted, because this was a temporary experiment

In [25]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

289/289 - 0s - loss: 1.5521e-04 - mean_absolute_error: 0.0049 - 435ms/epoch - 2ms/step


In [26]:
for i in test_labels:
    print(i)

0.025013185686983797
0.008678192795676385
0.010690190014378733
0.018028504255522687
0.009508459897929937
0.03521461562292524
0.013189049717871795
0.010642300460907701
0.015041865652167889
0.012309966285775112
0.00930557781140949
0.00939599242620015
0.019679283043475063
0.016927436464010202
0.008186053726351193
0.015515907235385715
0.019710210108030212
0.040714931344066015
0.02023467086304257
0.011388178603900196
0.018294501132030475
0.009226336068568299
0.009586581353501968
0.011960149585672259
0.011868258044399546
0.014306741964070922
0.017722492739378783
0.020379977886175113
0.011736803530399742
0.015625873781670394
0.012074143664964489
0.01719943746369718
0.020019036209060113
0.020189110602626743
0.010730296241137362
0.33949960206294283
0.016927436464010202
0.009144011697864749
0.010696498331125417
0.041097777496099726
0.0012594953762348861
0.013434732243113482
0.02035142310550929
0.02045996890030247
0.00245790471803447
0.004825269291420569
0.008615000437976032
0.017089040679878657


In [27]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

289/289 - 0s - 498ms/epoch - 2ms/step


In [28]:
for i in predictions:
    print(i)

[0.03051291]
[0.01272038]
[0.01305417]
[0.01505724]
[0.01358206]
[0.02530087]
[0.01560525]
[0.01302091]
[0.012803]
[0.01171593]
[0.01194123]
[0.0131729]
[0.01269752]
[0.01397188]
[0.01152925]
[0.01420659]
[0.01419019]
[0.02952888]
[0.01433577]
[0.01286844]
[0.01271548]
[0.01261524]
[0.01395798]
[0.01348094]
[0.01357583]
[0.01146165]
[0.01428991]
[0.01270417]
[0.01314576]
[0.01377479]
[0.01448587]
[0.01427157]
[0.01269709]
[0.01275261]
[0.01362893]
[0.3589117]
[0.01397188]
[0.0107365]
[0.01319698]
[0.04118013]
[0.01328473]
[0.01179747]
[0.0142705]
[0.01271281]
[0.01348788]
[0.01340441]
[0.01421265]
[0.01273925]
[0.04031596]
[0.01633171]
[0.01385682]
[0.01293201]
[0.01283506]
[0.01359604]
[0.01348788]
[0.01269752]
[0.00832275]
[0.03591204]
[0.01301888]
[0.01402814]
[0.01047234]
[0.01073627]
[0.01385682]
[0.01432247]
[0.01409723]
[0.01310101]
[0.01322517]
[0.01293201]
[0.01558379]
[0.01298746]
[0.01271281]
[0.01424122]
[0.03423644]
[0.01431527]
[0.01136259]
[0.0251394]
[0.01821868]
[0.012

[0.04857553]
[0.01376352]
[0.01315711]
[0.01393154]
[0.01383761]
[0.00869826]
[0.01380968]
[0.02944932]
[0.02444998]
[0.01918213]
[0.01284235]
[0.02312211]
[0.01391702]
[0.01263885]
[0.01430301]
[0.01402553]
[0.01328473]
[0.01273521]
[0.01231078]
[0.01273925]
[0.01310101]
[0.01402553]
[0.01622955]
[0.01429567]
[0.01469354]
[0.0137474]
[0.01311203]
[0.02082036]
[0.01292721]
[0.03349961]
[0.01385683]
[0.0370699]
[0.01424925]
[0.01349501]
[0.01357023]
[0.0344978]
[0.01428991]
[0.03218231]
[0.10209406]
[0.01310114]
[0.0131195]
[0.01314644]
[0.01284102]
[0.01243953]
[0.01314576]
[0.01125244]
[0.01271548]
[0.02217106]
[0.01769791]
[0.00908259]
[0.01352386]
[0.01110534]
[0.01376352]
[0.01420659]
[0.0119702]
[0.03706993]
[0.01105765]
[0.01311203]
[0.02790643]
[0.01430527]
[0.01221959]
[0.02166633]
[0.01323696]
[0.01335563]
[0.01319175]
[0.01433995]
[0.01286532]
[0.03923772]
[0.01269752]
[0.01224283]
[0.0147535]
[0.01348789]
[0.0313063]
[0.03341744]
[0.01275261]
[0.02399589]
[0.01616148]
[0.027

#### Better predictions compared to the previous experiment; all predicitons are positive; different predictions for different samples. However, even though the predictions are quite close in value to the actual labels, there are noticable differences between them. This does give a hint that the model is struggling slightly more with the 2nd dataset.

### In order to better understand the difference in performance for our model on the two datasets, we next compare them by examining the mean and standard deviation of the inputs and labels within the datasets.

## Comparing the 2 datasets

In [29]:
funct = exp_generator(3, 1)
funct

0.593988879915532*sin(2.01369540968675*x - 4.57047002658068) - 0.607193781823679

In [30]:
s1, l_max1, l_min1 = datalist_generator(funct, 1, 1, 96, 96)

In [31]:
funct1 = exp_generator(5, 1)
funct1

x**2 + y**2

In [32]:
s2, l_max2, l_min2 = datalist_generator(funct1, 10, 10, 105, 105)

In [33]:
funct2 = exp_generator(1, 2)
funct2

0.204219866843413*x**2 - 2.10374622235534*x + 2.66107437797953

In [34]:
s3, l_max3, l_min3 = datalist_generator(funct2, -10, -10, 45, 45)

In [35]:
s = np.concatenate((s1, s2, s3))
l_max = np.concatenate((l_max1, l_max2, l_max3))

In [36]:
m_s1 = np.mean(s)
m_s1

3493.9954860504686

In [37]:
sd_s1 = np.std(s)
sd_s1

5026.493560190871

In [38]:
m_l1 = np.mean(l_max)
m_l1

0.42890199615592384

In [39]:
sd_l1 = np.std(l_max)
sd_l1

0.7022647604963806

In [40]:
funct = exp_generator(4, 1)
funct

-0.0445981827274462*cos(0.677404823452205*x - 1.24950596636108) - 0.920229845386649

In [41]:
s1, l_max1, l_min1 = datalist_generator(funct, 1, 1, 96, 96)

In [42]:
funct1 = exp_generator(6, 1)
funct1

2.13702896348076*x**2 + 2.51939920893562*y**2

In [43]:
s2, l_max2, l_min2 = datalist_generator(funct1, 1, 1, 96, 96)

In [44]:
funct2 = exp_generator(1, 3)
funct2

0.290167489401872*x**3 + 0.720178969521534*x**2 - 4.13877058255077*x + 3.11987259832019

In [45]:
s3, l_max3, l_min3 = datalist_generator(funct2, -14, -14, 13, 13)

In [46]:
s_ = np.concatenate((s1, s2, s3))
l_max_ = np.concatenate((l_max1, l_max2, l_max3))

In [47]:
m_s2 = np.mean(s_)
m_s2

6970.063480991042

In [48]:
sd_s2 = np.std(s_)
sd_s2

9645.467067903037

In [49]:
m_l2 = np.mean(l_max_)
m_l2

0.01941357501010266

In [50]:
sd_l2 = np.std(l_max_)
sd_l2

0.0903433635420389

### Looking at the means and standard deviations, what can be inferred is that, the 1st dataset has comparatively lower input values that are comparatively less dispersed, and these inputs result in relatively higher curvatures, compared to the 2nd dataset. The 2nd dataset has comparatively lower curvature values than the 1st dataset. If we consider the hypothesis we made earlier about lower curvature values causing our model to struggle and look at our comparison, we can get some insight as to why the model struggles with the 2nd dataset in comparison to the 1st.

### Next, we experiment with a new combination of samples of the surface types represented by the Quadratic Sine Function and the Univariate Quartic Polynomial Function.

## Combination of samples used - Surface expressed by Quadratic Sine Function, Quartic Cylinder (Univariate Quartic Polynomial)<br>Training & Testing on Max Valued Curvatures with previously trained model
### Architecture - Depth(16 layers in total), Width (36 nodes highest among the layers)<br>"Unsigned" & "Split" Labels<br>Seeded expressions used & predictions checked

##### Quadratic Sine samples were in greater percentage in the combination because of them having high curvature values as labels.

In [6]:
funct1 = exp_generator(7, 2)
funct1

1.00844584567414*sin(0.0225702197513038*x + 4.50326005519469) - 1.73246409462899*sin(0.867571835717406*x - 3.89495266053861)**2 - 2.73115509338606

In [7]:
s1, l_max1, l_min1 = datalist_generator(funct1, 1, 1, 112, 112)

In [8]:
train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s1, l_max1, test_size = 0.3)

In [9]:
funct2 = exp_generator(1, 4)
funct2

-0.365871870349403*x**4 - 3.98445820291899*x**3 - 1.81213331822093*x**2 + 0.0236177013158286*x - 2.92715721059097

In [10]:
s2, l_max2, l_min2 = datalist_generator(funct2, -7, -7, 6, 6) # This small range of coordinates has high enough curvature
                                                              # suited for our purpose

In [11]:
train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s2, l_max2, test_size = 0.3)

In [12]:
train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [13]:
train_samples.shape

(8897, 9)

In [ ]:
for i in train_labels:
    print(i)

In [15]:
m_s = np.mean(train_samples)
m_s

-3.328700831586618

In [16]:
sd_s = np.std(train_samples)
sd_s

27.121036999842183

In [17]:
m_l = np.mean(train_labels)
m_l

0.9184099876436893

In [18]:
sd_l = np.std(train_labels)
sd_l

0.8455923514532678

In [21]:
from tensorflow.keras.models import load_model
model = load_model('Research_comb.h5')

In [22]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
334/334 - 3s - loss: 0.1107 - mean_absolute_error: 0.1851 - val_loss: 0.0236 - val_mean_absolute_error: 0.0799 - 3s/epoch - 8ms/step
Epoch 2/30
334/334 - 1s - loss: 0.0220 - mean_absolute_error: 0.0576 - val_loss: 0.0160 - val_mean_absolute_error: 0.0551 - 745ms/epoch - 2ms/step
Epoch 3/30
334/334 - 1s - loss: 0.0199 - mean_absolute_error: 0.0469 - val_loss: 0.0144 - val_mean_absolute_error: 0.0468 - 823ms/epoch - 2ms/step
Epoch 4/30
334/334 - 1s - loss: 0.0198 - mean_absolute_error: 0.0460 - val_loss: 0.0136 - val_mean_absolute_error: 0.0414 - 864ms/epoch - 3ms/step
Epoch 5/30
334/334 - 1s - loss: 0.0193 - mean_absolute_error: 0.0433 - val_loss: 0.0130 - val_mean_absolute_error: 0.0353 - 1s/epoch - 3ms/step
Epoch 6/30
334/334 - 1s - loss: 0.0194 - mean_absolute_error: 0.0434 - val_loss: 0.0136 - val_mean_absolute_error: 0.0411 - 1s/epoch - 3ms/step
Epoch 7/30
334/334 - 1s - loss: 0.0187 - mean_absolute_error: 0.0387 - val_loss: 0.0139 - val_mean_absolute_error: 0.0440 - 1s/

##### Results are a bit high.

In [23]:
model.save('Research_comb.h5')

In [24]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

191/191 - 0s - loss: 0.0093 - mean_absolute_error: 0.0182 - 277ms/epoch - 1ms/step


In [25]:
for i in test_labels:
    print(i)

0.07317057219210395
0.1270916012270449
0.15830291739259367
1.0932931928559868
1.1275414497028675
0.42375911575066827
0.36863612893023473
0.5413168839386479
2.2636048565511326
0.46534868354012315
2.4864704306598178
0.9205753490370685
0.30463851020492877
0.33134111992182463
1.4159825092707412
0.23766738258481485
0.15830291739259367
1.3464306622619115
1.865248705422586
1.6127996894129355
0.00023196293719868498
0.34649951145259505
0.19826794025644787
0.05328175734776965
1.6263468572085054
0.46175766243640726
1.0673919751339738
0.7424177417224346
0.14050334727237151
0.09832918574710332
0.3956370068310924
1.970050797822835
0.8640765771265051
1.987626234585897
2.604476889411056
1.336153338327431
2.5533559322992674
2.2636048565511326
0.19826794025644787
0.3977131135037523
1.055253432443024
1.6263468572085054
0.7424177417224346
0.5677580860684418
0.3977131135037523
1.6127996894129355
2.359373158724329
0.7424177417224346
0.2575519819999859
0.1427664748282771
1.3300833660746025
0.2183882055393519

In [26]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

191/191 - 0s - 396ms/epoch - 2ms/step


In [27]:
for i in predictions:
    print(i)

[0.0918093]
[0.13267973]
[0.15949705]
[1.1063226]
[1.1182495]
[0.43583974]
[0.3670644]
[0.5179323]
[2.3134053]
[0.4653264]
[2.4922364]
[0.91462255]
[0.290859]
[0.33785006]
[1.4505025]
[0.23785964]
[0.15949705]
[1.3606044]
[1.9005265]
[1.635754]
[0.1982564]
[0.3584313]
[0.20181194]
[0.05669513]
[1.6537808]
[0.46087328]
[1.0863692]
[0.7406156]
[0.13027921]
[0.1188682]
[0.4047903]
[1.9594371]
[0.8784592]
[1.9714609]
[2.615466]
[1.3267641]
[2.5508294]
[2.3134053]
[0.2018117]
[0.40764794]
[1.0656846]
[1.6537808]
[0.7406156]
[0.55692196]
[0.40764764]
[1.6357466]
[2.3476393]
[0.7406156]
[0.25155392]
[0.1534864]
[1.2989119]
[0.20928785]
[0.8147546]
[0.04161862]
[1.6537808]
[1.6357466]
[0.20928785]
[0.08451518]
[0.07323101]
[0.9318905]
[1.2989119]
[0.17027834]
[0.00547847]
[0.55692196]
[2.3476393]
[0.2132887]
[1.1929649]
[0.57785296]
[0.64903164]
[0.4047903]
[0.4697871]
[1.0270905]
[0.1902822]
[1.1929649]
[1.7900325]
[0.30872896]
[0.06808695]
[0.43583974]
[0.28191796]
[0.35843286]
[0.07343712]


[0.15348676]
[1.1063238]
[2.3134053]
[0.37250558]
[0.40764764]
[0.13027921]
[0.37250558]
[0.13027921]
[0.13027921]
[1.1929649]
[2.4956849]
[0.13267973]
[1.3606044]
[2.3476393]
[0.06285301]
[0.04148665]
[0.6970295]
[0.11194643]
[0.17027834]
[0.13027921]
[2.3476396]
[0.17027822]
[1.9005281]
[2.3476393]
[0.07343712]
[1.9714609]
[1.1635252]
[0.9318906]
[1.6357466]
[0.8147546]
[1.7797124]
[0.4200035]
[1.6507462]
[0.20181194]
[1.9714609]
[1.6357466]
[0.28191873]
[0.1188682]
[1.8281147]
[0.07343712]
[1.7900281]
[1.5725669]
[0.04961315]
[0.05573913]
[0.6627681]
[1.6537808]
[1.2989119]
[0.4200035]
[0.33785006]
[2.549698]
[1.6357466]
[2.4956849]
[0.2132887]
[0.06808695]
[0.91462255]
[0.3584313]
[0.62020636]
[0.05669513]
[0.4697871]
[0.00547847]
[0.14929584]
[0.11194697]
[0.1702722]
[1.9594371]
[1.9888555]
[0.3158733]
[0.1902822]
[0.20181194]
[1.4505025]
[0.23785964]
[1.1929649]
[1.0270905]
[0.13027921]
[1.1929649]
[0.28191873]
[1.3267641]
[2.4956849]
[2.552412]
[0.30872896]
[1.6507462]
[0.315874

#### For this dataset, the model was predicting negative values for those samples that had actual labels in the range 10^-5, further proving our hypothesis about our model struggling with low curvatures. After this, we put restrictions in our data generation function accordingly.
#### Besides this, the differences in the predictions and actual labels in general were small, differing in the 2nd or 3rd decimal point digits. However, the values of the curvatures for this dataset being high (the mean of curvatures for this dataset was higher compared to the other datasets) might have resulted in the higher losses and errors during training and evaluation.

### We can conclude that, our model, previously trained on the other combinations, performs well on the latest combination as well. However, now we again train and test our model from scratch exclusively on this latest combination used, in order to properly understand our model's performance on this dataset.

## Combination of samples used - Surface expressed by Quadratic Sine Function, Quartic Cylinder (Univariate Quartic Polynomial)<br>Training & Testing from scratch on Max Valued Curvatures - exclusively on this dataset
### Architecture - Depth(16 layers in total), Width (36 nodes highest among the layers)<br>"Unsigned" & "Split" Labels<br>Seeded expressions used & predictions checked

In [6]:
funct1 = exp_generator(7, 2)
funct1

1.00844584567414*sin(0.0225702197513038*x + 4.50326005519469) - 1.73246409462899*sin(0.867571835717406*x - 3.89495266053861)**2 - 2.73115509338606

In [7]:
s1, l_max1, l_min1 = datalist_generator(funct1, 1, 1, 112, 112)
train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s1, l_max1, test_size = 0.3)

In [8]:
funct2 = exp_generator(1, 4)
funct2

-0.365871870349403*x**4 - 3.98445820291899*x**3 - 1.81213331822093*x**2 + 0.0236177013158286*x - 2.92715721059097

In [9]:
s2, l_max2, l_min2 = datalist_generator(funct2, -7, -7, 6, 6)
train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s2, l_max2, test_size = 0.3)

In [10]:
train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [12]:
m_s = np.mean(train_samples)
m_s

-3.5522710915465607

In [13]:
sd_s = np.std(train_samples)
sd_s

25.12122431229878

In [14]:
m_l = np.mean(train_labels)
m_l

0.9086590564485308

In [15]:
sd_l = np.std(train_labels)
sd_l

0.8396719379381462

In [16]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [17]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [18]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
334/334 - 3s - loss: 1.5114 - mean_absolute_error: 0.8986 - val_loss: 1.4666 - val_mean_absolute_error: 0.8808 - 3s/epoch - 10ms/step
Epoch 2/30
334/334 - 1s - loss: 1.4481 - mean_absolute_error: 0.8647 - val_loss: 1.4116 - val_mean_absolute_error: 0.8526 - 725ms/epoch - 2ms/step
Epoch 3/30
334/334 - 1s - loss: 1.3947 - mean_absolute_error: 0.8385 - val_loss: 1.3596 - val_mean_absolute_error: 0.8276 - 1s/epoch - 3ms/step
Epoch 4/30
334/334 - 1s - loss: 1.3440 - mean_absolute_error: 0.8153 - val_loss: 1.3100 - val_mean_absolute_error: 0.8055 - 963ms/epoch - 3ms/step
Epoch 5/30
334/334 - 1s - loss: 1.2960 - mean_absolute_error: 0.7947 - val_loss: 1.2632 - val_mean_absolute_error: 0.7853 - 853ms/epoch - 3ms/step
Epoch 6/30
334/334 - 1s - loss: 1.2504 - mean_absolute_error: 0.7763 - val_loss: 1.2185 - val_mean_absolute_error: 0.7674 - 917ms/epoch - 3ms/step
Epoch 7/30
334/334 - 1s - loss: 1.2070 - mean_absolute_error: 0.7602 - val_loss: 1.1763 - val_mean_absolute_error: 0.7526 -

##### The results gradually decrease, but they are significantly higher in value.

In [19]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

191/191 - 0s - loss: 0.4959 - mean_absolute_error: 0.4632 - 210ms/epoch - 1ms/step


In [20]:
for i in test_labels:
    print(i)

2.2531466595708403
0.061837865776418116
0.07576577989003856
2.591309852749588
1.191619397077886
0.18567794345158858
2.5297959907617016
0.6202804989412962
1.3464306622619115
0.20485851332951568
0.6202804989412962
1.1597393839588628
2.5895105747615443
2.5895105747615443
0.6828589579420863
0.6202804989412962
0.03668222476291964
0.028552297971799402
1.987626234585897
1.4159825092707412
0.33134111992182463
1.6263468572085054
1.6263468572085054
0.0002587186799580598
0.07576577989003856
2.591309852749588
0.1427664748282771
0.5762110584450302
0.913255974447654
0.45496374584545984
0.17250311795260068
2.604476889411056
2.2746991163282346
0.14418849869304395
0.18996487325337577
0.10112121459315382
0.07576577989003856
1.3300833660746025
0.9197970854102411
0.30414523279808414
0.4968980866084505
0.17684047979289388
0.23766738258481485
2.5145716033311536
1.1275414497028675
2.429003090862608
0.09728402933945132
0.913255974447654
0.08681230526435987
0.20485851332951568
0.18996487325337577
2.59130985274

In [21]:
predictions = model.predict(x = test_samples, batch_size = 20, verbose = 2)

191/191 - 0s - 324ms/epoch - 2ms/step


In [22]:
for i in predictions:
    print(i)

[0.87378985]
[0.31261337]
[0.20578855]
[0.87378985]
[0.86838317]
[0.20578855]
[0.87378985]
[0.87378985]
[0.87378985]
[0.15165854]
[0.87378985]
[0.83870965]
[0.87378985]
[0.87378985]
[0.20578855]
[0.87378985]
[0.16944396]
[0.20578855]
[0.87378985]
[0.87378985]
[0.58783185]
[0.87378985]
[0.87378985]
[-0.0057801]
[0.20578855]
[0.87378985]
[0.20578855]
[0.71943104]
[0.87378985]
[0.21378678]
[0.24910873]
[0.87378985]
[0.87378985]
[0.20578855]
[0.5338388]
[0.20578855]
[0.20578855]
[0.8391003]
[0.87378985]
[0.87378985]
[0.86911553]
[0.20578855]
[0.14199144]
[0.87378985]
[0.87378985]
[0.87378985]
[0.13502878]
[0.87378985]
[0.20578855]
[0.15165854]
[0.5338388]
[0.87378985]
[0.12988287]
[0.87378985]
[0.87378985]
[0.20578855]
[0.20578855]
[0.43779776]
[0.68007195]
[0.87378985]
[0.87378985]
[0.14826554]
[0.31261337]
[0.13502878]
[0.5381549]
[0.20578855]
[0.87378985]
[0.16944396]
[0.14826554]
[0.87378985]
[0.21378678]
[0.22147995]
[0.15165854]
[0.87378985]
[0.87378985]
[0.58783185]
[0.87378985]
[0.

#### This time the model is predicting similar curvature values for different samples, as a result of which the results are comparatively higher than earlier experiments.

### The outcome for the current experiment is worse, compared to the previous experiment which worked with the same dataset but using a model previously trained on other datasets. This hints at the fact that the architecture might need to be changed. Another thing to note is that, in the earlier experiment, the model was previously trained for a certain number of epochs on other datasets and then trained on the 3rd dataset for more epochs, in comparison to the current experiment where the model was only trained on the 3rd dataset for a smaller number of epochs. This could have resulted in the model giving a poor performance in the latest experiment, since the model was not trained for enough epochs.<br>So, next we perform the same experiment once more, but this time the only difference being that the model is trained for a higher number of epochs, and see if any improvement in performance occurs or not.

## Combination of samples used - Surface expressed by Quadratic Sine Function, Quartic Cylinder (Univariate Quartic Polynomial)<br>Training & Testing from scratch on Max Valued Curvatures - exclusively on the current dataset<br>Training done for a higher number of epochs
### Architecture - Depth(16 layers in total), Width (36 nodes highest among the layers)<br>"Unsigned" & "Split" Labels<br>Seeded expressions used & predictions checked

In [6]:
funct1 = exp_generator(7, 2)
funct1

1.00844584567414*sin(0.0225702197513038*x + 4.50326005519469) - 1.73246409462899*sin(0.867571835717406*x - 3.89495266053861)**2 - 2.73115509338606

In [7]:
s1, l_max1, l_min1 = datalist_generator(funct1, 1, 1, 112, 112, 1)
train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s1, l_max1, test_size = 0.3, random_state = 1)

In [8]:
funct2 = exp_generator(1, 4)
funct2

-0.365871870349403*x**4 - 3.98445820291899*x**3 - 1.81213331822093*x**2 + 0.0236177013158286*x - 2.92715721059097

In [9]:
s2, l_max2, l_min2 = datalist_generator(funct2, -7, -7, 6, 6, 1)
train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s2, l_max2, test_size = 0.3, random_state = 5)

#### The parameter 'random_state' was included within the train_test_split function call to replicate the dataset split into training and test sets across different experiments, so that the performance of our model can be properly evaluated on a particular dataset. Without the inclusion of the 'random_state' parameter, the training and test sets will be different across different experiments due to differing dataset split, resulting in varying training and in turn, varying performance of our model across the different experiments. It might be possible that this had been occurring in the earlier experiments since we didn't use the 'random_state' parameter before.

In [10]:
train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [11]:
m_s = np.mean(train_samples)
m_s

-3.3723664891118883

In [12]:
sd_s = np.std(train_samples)
sd_s

25.49058448855376

In [13]:
m_l = np.mean(train_labels)
m_l

0.9054668874247073

In [14]:
sd_l = np.std(train_labels)
sd_l

0.8393537146626985

In [15]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 34, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 22, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [16]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [17]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 150, shuffle = True, verbose = 2)

Epoch 1/150
334/334 - 3s - loss: 1.4991 - mean_absolute_error: 0.8923 - val_loss: 1.4744 - val_mean_absolute_error: 0.8847 - 3s/epoch - 8ms/step
Epoch 2/150
334/334 - 1s - loss: 1.4376 - mean_absolute_error: 0.8589 - val_loss: 1.4191 - val_mean_absolute_error: 0.8569 - 606ms/epoch - 2ms/step
Epoch 3/150
334/334 - 1s - loss: 1.3847 - mean_absolute_error: 0.8326 - val_loss: 1.3669 - val_mean_absolute_error: 0.8326 - 745ms/epoch - 2ms/step
Epoch 4/150
334/334 - 1s - loss: 1.3345 - mean_absolute_error: 0.8094 - val_loss: 1.3174 - val_mean_absolute_error: 0.8111 - 830ms/epoch - 2ms/step
Epoch 5/150
334/334 - 1s - loss: 1.2869 - mean_absolute_error: 0.7886 - val_loss: 1.2702 - val_mean_absolute_error: 0.7914 - 988ms/epoch - 3ms/step
Epoch 6/150
334/334 - 1s - loss: 1.2417 - mean_absolute_error: 0.7700 - val_loss: 1.2255 - val_mean_absolute_error: 0.7742 - 669ms/epoch - 2ms/step
Epoch 7/150
334/334 - 1s - loss: 1.1988 - mean_absolute_error: 0.7540 - val_loss: 1.1831 - val_mean_absolute_error:

334/334 - 1s - loss: 0.1593 - mean_absolute_error: 0.2701 - val_loss: 0.1473 - val_mean_absolute_error: 0.2622 - 784ms/epoch - 2ms/step
Epoch 57/150
334/334 - 1s - loss: 0.1502 - mean_absolute_error: 0.2611 - val_loss: 0.1412 - val_mean_absolute_error: 0.2599 - 641ms/epoch - 2ms/step
Epoch 58/150
334/334 - 1s - loss: 0.1432 - mean_absolute_error: 0.2565 - val_loss: 0.1337 - val_mean_absolute_error: 0.2512 - 598ms/epoch - 2ms/step
Epoch 59/150
334/334 - 1s - loss: 0.1363 - mean_absolute_error: 0.2515 - val_loss: 0.1267 - val_mean_absolute_error: 0.2463 - 903ms/epoch - 3ms/step
Epoch 60/150
334/334 - 1s - loss: 0.1294 - mean_absolute_error: 0.2471 - val_loss: 0.1202 - val_mean_absolute_error: 0.2406 - 666ms/epoch - 2ms/step
Epoch 61/150
334/334 - 1s - loss: 0.1223 - mean_absolute_error: 0.2411 - val_loss: 0.1146 - val_mean_absolute_error: 0.2367 - 586ms/epoch - 2ms/step
Epoch 62/150
334/334 - 1s - loss: 0.1159 - mean_absolute_error: 0.2358 - val_loss: 0.1081 - val_mean_absolute_error: 0.

334/334 - 1s - loss: 0.0169 - mean_absolute_error: 0.1017 - val_loss: 0.0312 - val_mean_absolute_error: 0.1076 - 670ms/epoch - 2ms/step
Epoch 112/150
334/334 - 1s - loss: 0.0172 - mean_absolute_error: 0.1005 - val_loss: 0.0154 - val_mean_absolute_error: 0.1006 - 609ms/epoch - 2ms/step
Epoch 113/150
334/334 - 1s - loss: 0.0156 - mean_absolute_error: 0.0972 - val_loss: 0.0134 - val_mean_absolute_error: 0.0939 - 637ms/epoch - 2ms/step
Epoch 114/150
334/334 - 1s - loss: 0.0157 - mean_absolute_error: 0.0979 - val_loss: 0.0130 - val_mean_absolute_error: 0.0945 - 586ms/epoch - 2ms/step
Epoch 115/150
334/334 - 1s - loss: 0.0155 - mean_absolute_error: 0.0973 - val_loss: 0.0133 - val_mean_absolute_error: 0.0939 - 661ms/epoch - 2ms/step
Epoch 116/150
334/334 - 1s - loss: 0.0151 - mean_absolute_error: 0.0957 - val_loss: 0.0125 - val_mean_absolute_error: 0.0919 - 811ms/epoch - 2ms/step
Epoch 117/150
334/334 - 1s - loss: 0.0149 - mean_absolute_error: 0.0951 - val_loss: 0.0137 - val_mean_absolute_err

##### The results have gradually decreased to low values with very little fluctuations towards the later epochs; the losses are better than the ones obtained in the experiment with the pretrained model, but the errors are not.

In [18]:
result1 = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

191/191 - 0s - loss: 0.0072 - mean_absolute_error: 0.0588 - 212ms/epoch - 1ms/step


In [19]:
for i in test_labels:
    print(i)

0.20293630238201488
1.6263468572085054
0.19826794025644787
1.1597393839588628
0.20485851332951568
1.1691860560178502
0.42375911575066827
0.2575519819999859
2.4051894815667643
1.6127996894129355
0.913255974447654
0.6431450248031988
3.794346029120063e-05
0.3956370068310924
0.6431450248031988
0.5677580860684418
0.6431450248031988
1.5418086831519906
1.6263468572085054
0.03668222476291964
0.061837865776418116
1.9697727846590312
0.7375748070332325
0.913255974447654
0.5677580860684418
0.5762110584450302
1.1275414497028675
1.0673919751339738
1.3300833660746025
1.9697727846590312
1.055253432443024
1.191619397077886
1.6127996894129355
1.3300833660746025
1.5418086831519906
0.018120258764773363
1.970050797822835
0.36863612893023473
1.5418086831519906
0.14050334727237151
0.6581447381704736
0.45496374584545984
1.5418086831519906
0.4968980866084505
0.03668222476291964
0.1427664748282771
0.05328175734776965
2.1831065195270627
0.2183882055393519
0.2842559525890907
0.047826300093705906
0.257551981999985

In [20]:
predictions1 = model.predict(x = test_samples, batch_size = 20, verbose = 2)

191/191 - 0s - 304ms/epoch - 2ms/step


In [21]:
for i in predictions1:
    print(i)

[0.234864]
[1.640269]
[0.12227321]
[1.0544882]
[0.17659736]
[1.174635]
[0.234864]
[0.234864]
[2.4024065]
[1.6628865]
[0.97303724]
[0.6745068]
[0.00625849]
[0.3516605]
[0.6745068]
[0.6227199]
[0.6745068]
[1.5378544]
[1.6402695]
[0.08036447]
[0.11725354]
[1.8770554]
[0.7577355]
[0.97303724]
[0.6227199]
[0.6134441]
[1.241005]
[1.193426]
[1.253439]
[1.8770554]
[1.0648195]
[1.1631352]
[1.6628865]
[1.253439]
[1.5378544]
[0.234864]
[1.9135964]
[0.234864]
[1.5378544]
[0.1756854]
[0.6578244]
[0.40362215]
[1.5378544]
[0.55359507]
[0.08036447]
[0.234864]
[0.234864]
[2.1488457]
[0.234864]
[0.2860849]
[0.10259199]
[0.234864]
[2.3094647]
[0.3516605]
[1.0138127]
[0.6227199]
[0.234864]
[0.234864]
[0.8104371]
[1.788193]
[2.4897108]
[2.5888295]
[0.38045812]
[0.8472985]
[2.4056635]
[0.234864]
[0.02650785]
[2.239528]
[0.4559915]
[2.5730124]
[0.234864]
[0.18231964]
[0.234864]
[1.1631352]
[2.296151]
[2.5113232]
[0.37668586]
[1.6766571]
[2.3464997]
[1.877057]
[0.234864]
[0.234864]
[0.234864]
[0.7013048]
[0.4

#### Very little number of negative predictions; the predictions have quite small differences with the actual labels this time. However, for a few samples, the predictions were the same value, even though there weren't any such pattern in the actual labels.

#### We also evaluate the performance of our model on the training set in order to check whether there is any case of overffiting.

In [22]:
result2 = model.evaluate(train_samples, train_labels, batch_size = 20, verbose = 2)

445/445 - 0s - loss: 0.0078 - mean_absolute_error: 0.0597 - 496ms/epoch - 1ms/step


In [23]:
for i in train_labels:
    print(i)

0.3533310229835167
1.033394536584167
0.3533310229835167
0.30463851020492877
0.14050334727237151
1.4159825092707412
0.1427664748282771
0.018120258764773363
0.15830291739259367
0.913255974447654
0.7404613200794526
2.4051894815667643
0.30499953035446087
0.30499953035446087
0.3533310229835167
2.5297959907617016
0.17250311795260068
0.9205753490370685
1.6127996894129355
2.359373158724329
2.4864704306598178
0.7850571172351063
2.5297959907617016
0.14418849869304395
0.7404613200794526
1.055253432443024
0.7404613200794526
2.2531466595708403
0.028552297971799402
1.970050797822835
0.05204205925125898
1.9697727846590312
0.17189229443939066
0.7375748070332325
0.45496374584545984
0.05204205925125898
1.7313895754151782
0.018120258764773363
1.1691860560178502
1.987626234585897
1.815053703080649
0.9197970854102411
0.09728402933945132
1.055253432443024
0.30463851020492877
1.781682711301299
0.061837865776418116
0.5413168839386479
2.5533559322992674
0.9949379756732198
0.8640765771265051
0.2575519819999859


In [24]:
predictions2 = model.predict(x = train_samples, batch_size = 20, verbose = 2)

445/445 - 0s - 481ms/epoch - 1ms/step


In [25]:
for i in predictions2:
    print(i)

[0.3679998]
[1.0850097]
[0.3679998]
[0.34778547]
[0.1756854]
[1.412613]
[0.234864]
[0.234864]
[0.17457795]
[0.97303724]
[0.7013048]
[2.4024065]
[0.234864]
[0.234864]
[0.3679998]
[2.4872422]
[0.18231964]
[1.0138127]
[1.6628852]
[2.3464997]
[2.5113232]
[0.81043696]
[2.4872422]
[0.234864]
[0.7013048]
[1.0648195]
[0.7013048]
[2.239528]
[0.234864]
[1.9135964]
[0.234864]
[1.8770554]
[0.234864]
[0.7577355]
[0.40362215]
[0.234864]
[1.7914717]
[0.234864]
[1.1746352]
[1.9211462]
[1.7881929]
[0.9400691]
[0.05778646]
[1.0648195]
[0.34778547]
[1.7901262]
[0.11725354]
[0.5640571]
[2.513883]
[1.0198107]
[0.8472985]
[0.234864]
[0.2860849]
[0.37668586]
[0.234864]
[1.1631352]
[2.5651333]
[0.698213]
[0.23486447]
[0.4036231]
[0.6745068]
[0.05778646]
[0.1756854]
[0.6578244]
[0.97303724]
[1.7901262]
[0.37668586]
[1.412613]
[2.4872422]
[1.9211472]
[0.04914188]
[0.234864]
[0.04914188]
[2.0168622]
[0.6227199]
[2.296151]
[0.7013048]
[1.9211472]
[0.6134442]
[2.1488454]
[2.5888295]
[2.4872422]
[0.9400691]
[0.0809

[2.4056635]
[0.6134441]
[0.3679998]
[0.06369948]
[1.253439]
[1.3090334]
[0.6227199]
[0.234864]
[1.8770554]
[1.0245823]
[1.0544882]
[0.1747315]
[2.296151]
[0.1756854]
[0.45599222]
[1.676659]
[1.8770554]
[2.4056635]
[1.7914717]
[0.4559915]
[1.9211472]
[0.234864]
[0.234864]
[0.37668586]
[0.38045812]
[0.6686506]
[0.234864]
[1.241005]
[0.19519639]
[2.4056635]
[0.234864]
[0.234864]
[2.0168622]
[1.6628865]
[1.1631342]
[1.3090336]
[0.234864]
[0.234864]
[0.234864]
[1.7699276]
[0.10957122]
[0.234864]
[0.698213]
[0.234864]
[1.0544882]
[0.17457795]
[0.234864]
[1.3090334]
[0.1756854]
[1.241005]
[1.0850097]
[0.97303724]
[0.37668586]
[2.296151]
[2.5138824]
[1.08501]
[1.7914717]
[1.7881929]
[0.5348978]
[0.3516605]
[0.234864]
[0.3516605]
[0.234864]
[1.2902442]
[0.97303724]
[2.5494683]
[1.2902442]
[0.234864]
[0.37668586]
[0.17659736]
[0.3679998]
[1.9211472]
[1.2254956]
[2.4024065]
[0.6675793]
[1.7901258]
[2.4872422]
[0.698213]
[1.0138127]
[2.3094647]
[0.06369948]
[0.234864]
[0.55359507]
[0.34778547]
[2.

#### The train set predictions also have the same behavior as like the test set ones.

### Even though the performance of the model improved quite a bit from the previous experiment after training it for more epochs, the same problem of predicting the same curvature value was still seen with some samples. So, we decide to change the model architecture slightly and check if any improvement is obtained in the performance. 

### While we try to figure out an architecture that works best with all types of dataset, we conduct an experiment side-by-side with our already trained model where we again train it on the 2nd dataset, but this time we use a smaller step size for the input variable arguments when generating Univariate Cubic Polynomial samples. The reason behind this is that, in the earlier experiments, we used a smaller percentage of Univariate Cubic Polynomial samples as these have smaller curvature values, but this could cause the model to develop a training bias since the model is getting trained more on the samples with higher-valued curvatures. In order to balance the number of samples having lower-valued curvatures with that of the samples with high curvatures, we devise a strategy when generating the samples with low curvatures (in this case, the Univariate Cubic Polynomial samples), where we use a smaller step size (less than 1) for the input variable arguments within the small interval of coordinates giving the highest possible curvature values for this type of expression, resulting in a higher number of samples compared to the earlier experiment where we used a step size of 1. We used the already trained model we have and train it for a much higher number of epochs for proper training and evaluate the performance. We also made sure to use the random_state parameter for the train_test_split function in this experiment.

## Combination of samples used - Cosine Wave Surface, Cubic Cylinder (Univariate Cubic Polynomial), Elliptic Paraboloid (z = ax^2 + by^2)<br>Training & Testing on Max Valued Curvatures with previously trained model<br>Using a smaller step size for Cubic Cylinder samples; trained for a higher number of epochs
### Architecture - Depth(16 layers in total), Width (36 nodes highest among the layers)<br>"Unsigned" & "Split" Labels<br>Seeded expressions used & predictions checked

In [6]:
funct1 = exp_generator(4, 1)
funct1

-0.0445981827274462*cos(0.677404823452205*x - 1.24950596636108) - 0.920229845386649

In [7]:
s1, l_max1, l_min1 = datalist_generator(funct1, 101, 101, 150, 150, 1)

In [8]:
funct2 = exp_generator(6, 1)
funct2

2.13702896348076*x**2 + 2.51939920893562*y**2

In [9]:
s2, l_max2, l_min2 = datalist_generator(funct2, 101, 101, 150, 150, 1)

##### We used a different range of coordinates for these two types of expressions in this experiment, compared to the earlier experiments involving this dataset, so that we can train our model on unseen data samples of similar types of surfaces.

In [10]:
s = np.concatenate((s1, s2))
l_max = np.concatenate((l_max1, l_max2))

s, l_max = shuffle(s, l_max)

train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s, l_max, test_size = 0.3, random_state = 2)

In [11]:
funct3 = exp_generator(1, 3)
funct3

0.290167489401872*x**3 + 0.720178969521534*x**2 - 4.13877058255077*x + 1.87192355899211

In [12]:
s3, l_max3, l_min3 = datalist_generator(funct3, -14, -14, 13, 13, 0.5)
train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s3, l_max3, test_size = 0.3, random_state = 17)

In [13]:
train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [15]:
m_s = np.mean(train_samples)
m_s

23090.742927050218

In [16]:
sd_s = np.std(train_samples)
sd_s

34983.056396434375

In [17]:
m_l = np.mean(train_labels)
m_l

0.065315312011629

In [18]:
sd_l = np.std(train_labels)
sd_l

0.396471786381353

In [19]:
from tensorflow.keras.models import load_model
model = load_model('Research_comb.h5')

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 100, shuffle = True, verbose = 2)

##### (Output deleted to save memory, as the model fails ultimately)<br>The results are decent enough; however, the training loss oscillates around 0.0050, along with the other results around their respective values when the training loss hits 0.0050

In [21]:
model.save('Research_comb.h5') # Deleted, because the model eventually fails

In [22]:
result1 = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

121/121 - 0s - loss: 0.0040 - mean_absolute_error: 0.0154 - 183ms/epoch - 2ms/step


In [23]:
for i in test_labels:
    print(i)

5.693461829688935e-06
3.729702488783223e-05
0.0002945576257815709
8.417339503402771e-05
1.1046873279509826e-05
0.007208984482816517
0.006418311123000699
0.000944663930455033
0.006070448413060749
0.002658109523447659
5.693461829688935e-06
0.004985562900488245
3.425582193912699e-05
0.04623998935243249
0.014782343158238289
9.334697742103088e-05
0.005239463962750905
0.01962280501843827
0.005727359805766434
0.005475294879714968
0.007208984482816517
0.00609641211787001
0.00481506134712419
0.00632983237585868
0.018925553851277588
0.005216551459238964
0.004714869379522877
0.005211428096407185
0.1848004380743929
4.5361526050613526e-05
0.01935148043538494
0.007503301778525001
0.01164355214679571
5.693461829688935e-06
0.013539935474920849
0.0014608303590021039
1.6115047351038672e-05
0.005359864686693584
0.007208984482816517
0.006577002226701123
0.00046664330397470845
8.405671107907427e-06
0.014562243918174523
0.015719968885982014
0.002658109523447659
0.0014608303590021039
0.00017201126773013704
0

In [24]:
predictions1 = model.predict(x = test_samples, batch_size = 20, verbose = 2)

121/121 - 0s - 303ms/epoch - 3ms/step


In [25]:
for i in predictions1:
    print(i)

[0.0393864]
[-0.00276517]
[0.00823165]
[0.0393864]
[0.00183345]
[0.01451219]
[0.0033126]
[0.01121761]
[0.01157821]
[0.01556851]
[0.0393864]
[0.00333406]
[0.0393864]
[0.01887966]
[0.01547266]
[-0.00476693]
[0.00332786]
[0.0136006]
[0.0033188]
[0.00332261]
[0.01451219]
[0.00331403]
[0.03973795]
[0.00753326]
[0.01483549]
[0.01094843]
[0.01147045]
[0.00332595]
[0.18489684]
[0.0393864]
[0.01142825]
[0.01089622]
[0.01213397]
[0.0393864]
[0.01237584]
[0.01109136]
[0.0393864]
[0.0033269]
[0.01451195]
[0.00330926]
[0.01570095]
[0.0393864]
[0.01252187]
[0.01535846]
[0.01556851]
[0.01109136]
[0.0393864]
[0.0033331]
[0.0139928]
[0.00331355]
[0.00333024]
[0.00332309]
[0.02491136]
[-0.00276517]
[0.01312817]
[0.0136006]
[0.04375071]
[0.01312817]
[0.03938628]
[0.00332118]
[0.00331355]
[0.01237584]
[0.01147045]
[0.00331784]
[0.00219394]
[0.01153291]
[0.0393864]
[0.00332452]
[0.00183345]
[0.00332452]
[0.01286722]
[0.0139351]
[0.01121761]
[-0.00476693]
[0.01547266]
[0.01367302]
[0.0393864]
[0.0393864]
[0

#### There are a few negative predictions, but they can't be traced to really small labels. Significant difference can be seen between the predictions and the actual labels. There is also the problem of having the same prediction for some samples even though the actual labels are different for those samples.

In [26]:
result2 = model.evaluate(train_samples, train_labels, batch_size = 20, verbose = 2)

281/281 - 0s - loss: 0.0045 - mean_absolute_error: 0.0157 - 416ms/epoch - 1ms/step


In [27]:
for i in train_labels:
    print(i)

0.013539935474920849
0.0061099196249833326
0.006070448413060749
0.3517144185987097
0.005946350183752917
0.004729889468501197
0.006370891783502074
0.33949960206294283
0.005918658770292268
0.018202976283169546
0.004656138075990865
0.012065399406056684
0.00013266354454781874
0.015719968885982014
0.006613620619607227
0.041097777496099726
0.0004026373761829719
0.0054056854990442
0.006700048059243256
0.020251388912184476
0.006042742996444183
4.9651895465459124e-05
0.0049829754069754
0.018925553851277588
0.005131988340458837
0.016151278226838543
0.013017061327536102
0.005252588997387652
0.00017201126773013704
8.405671107907427e-06
2.6282409205329398e-05
0.009407740691296976
0.02044691455723443
0.006171408584552151
0.004714869379522877
0.005288826060602252
0.01935148043538494
0.00632983237585868
0.006557975116219019
0.020170170349622478
0.01261352095067763
0.0047624409668471125
1.1046873279509826e-05
0.061272552794954756
0.0002585277017031552
1.1046873279509826e-05
0.00650128812335263
0.005324

In [28]:
predictions2 = model.predict(x = train_samples, batch_size = 20, verbose = 2)

281/281 - 0s - 378ms/epoch - 1ms/step


In [29]:
for i in predictions2:
    print(i)

[0.01237584]
[0.00331498]
[0.01157821]
[0.34705573]
[0.00331784]
[0.0033393]
[0.01573576]
[0.33799237]
[0.00331737]
[0.013191]
[0.00334598]
[0.01088823]
[-0.00232219]
[0.01535846]
[0.00330974]
[0.02413989]
[0.0393864]
[0.00332547]
[0.00330902]
[0.01172925]
[0.00331689]
[-0.00487899]
[0.00333406]
[0.01483549]
[0.00333024]
[0.01277794]
[0.01562513]
[0.00332833]
[0.0393864]
[0.0393864]
[0.0393864]
[0.01430143]
[0.0120777]
[0.00331451]
[0.01147045]
[0.00332738]
[0.01142825]
[0.00753326]
[0.00330926]
[0.01385118]
[0.0139351]
[0.00333835]
[0.00183345]
[0.02491136]
[0.01539053]
[0.00183345]
[0.00331069]
[0.0148703]
[0.0393864]
[0.0393864]
[0.01246155]
[0.0393864]
[0.00823165]
[0.00219394]
[0.00331689]
[0.0393864]
[0.0393864]
[0.00332214]
[0.3470558]
[0.00332166]
[0.01087607]
[0.01417066]
[4.0518627]
[0.00332309]
[0.01157821]
[0.00333072]
[0.0139928]
[0.01575316]
[4.0518627]
[0.00332166]
[0.00331069]
[0.0393864]
[0.01422502]
[0.00332833]
[0.0033331]
[0.00333739]
[0.00212241]
[0.0110649]
[0.015

[0.00332738]
[0.01547266]
[2.3006396]
[0.00242377]
[0.00331546]
[0.04209676]
[0.01965971]
[0.0393864]
[0.01172925]
[0.01547266]
[0.01153291]
[0.0110649]
[0.01562513]
[0.00332118]
[0.00331308]
[0.00332833]
[0.04375071]
[0.0139928]
[0.04209676]
[0.00333406]
[0.0393864]
[0.00332452]
[0.03938628]
[0.01172925]
[0.00331927]
[0.0110649]
[0.01093317]
[0.01435257]
[0.0293252]
[0.01172925]
[0.01142825]
[0.00332595]
[0.01312817]
[0.00331784]
[0.00333739]
[0.00332595]
[0.00331308]
[0.00331451]
[0.01087989]
[0.33799237]
[-0.00276517]
[0.02491141]
[0.0136729]
[0.00334026]
[0.01367302]
[0.00421764]
[0.0148703]
[0.01614202]
[0.01157821]
[0.0393864]
[0.0393864]
[0.0393864]
[0.00332643]
[0.01451219]
[0.01512278]
[0.0393864]
[0.0393864]
[0.01246155]
[-0.00232219]
[0.00332071]
[0.01564671]
[0.00212241]
[0.00332738]
[0.00331308]
[0.01547266]
[0.01483549]
[0.00331784]
[0.00421764]
[0.0139351]
[0.01435257]
[0.00332071]
[0.00332547]
[0.01455523]
[0.01286722]
[0.01153291]
[0.0033269]
[0.01385118]
[0.00331784]


#### Similar type of predictions for the train set as well; the predictions are maybe just very slightly better in this case.

### So, similar to the other experiment involving the 3rd dataset, this architecture fails with this dataset as well, as we tried to increase the percentage of the smaller-valued samples. So, we have to continue this experiment with a different architecture next as like we had decided in case of the other experiment. Another thing to note is that, a lot of the samples in this experiment had curvatures as low as 10^-6, since we didn't put strict restrictions during data generation. This could have also played some role in the poor performance of our architecture. So, in the future experiments, we put restrictions on our data generation accordingly.

### Now, we continue the other experiment with the dataset involving Quadratic Sine and Quartic Cylinder samples with a new architecure with more depth and width.

## Architecture - Depth(20 layers in total), Width (40 nodes highest among the layers)<br>Combination of samples used - Surface expressed by Quadratic Sine Function, Quartic Cylinder (Univariate Quartic Polynomial)
### Training & Testing from scratch on Max Valued Curvatures - exclusively on this dataset<br>"Unsigned" & "Split" Labels<br>Seeded expressions used & predictions checked<br>Training done for higher number of epochs

In [6]:
funct1 = exp_generator(7, 2)
funct1

1.00844584567414*sin(0.0225702197513038*x + 4.50326005519469) - 1.73246409462899*sin(0.867571835717406*x - 3.89495266053861)**2 - 2.73115509338606

In [7]:
s1, l_max1, l_min1 = datalist_generator(funct1, 1, 1, 112, 112, 1)
train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s1, l_max1, test_size = 0.3, random_state = 1)

In [8]:
funct2 = exp_generator(1, 4)
funct2

-0.365871870349403*x**4 - 3.98445820291899*x**3 - 1.81213331822093*x**2 + 0.0236177013158286*x - 2.92715721059097

In [9]:
s2, l_max2, l_min2 = datalist_generator(funct2, -7, -7, 6, 6, 1)
train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s2, l_max2, test_size = 0.3, random_state = 5)

In [10]:
train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [11]:
m_s = np.mean(train_samples)
m_s

-3.372366489111889

In [12]:
sd_s = np.std(train_samples)
sd_s

25.49058448855376

In [13]:
m_l = np.mean(train_labels)
m_l

0.9054668874247072

In [14]:
sd_l = np.std(train_labels)
sd_l

0.8393537146626985

In [18]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 32, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 40, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 32, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [19]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [20]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 150, shuffle = True, verbose = 2)

Epoch 1/150
334/334 - 4s - loss: 1.4084 - mean_absolute_error: 0.8452 - val_loss: 1.3185 - val_mean_absolute_error: 0.7300 - 4s/epoch - 11ms/step
Epoch 2/150
334/334 - 1s - loss: 1.0814 - mean_absolute_error: 0.7146 - val_loss: 0.9048 - val_mean_absolute_error: 0.6901 - 1s/epoch - 3ms/step
Epoch 3/150
334/334 - 1s - loss: 0.8378 - mean_absolute_error: 0.6882 - val_loss: 0.7707 - val_mean_absolute_error: 0.6817 - 1s/epoch - 3ms/step
Epoch 4/150
334/334 - 1s - loss: 0.7340 - mean_absolute_error: 0.6937 - val_loss: 0.7016 - val_mean_absolute_error: 0.6933 - 1s/epoch - 3ms/step
Epoch 5/150
334/334 - 1s - loss: 0.6957 - mean_absolute_error: 0.7035 - val_loss: 0.6913 - val_mean_absolute_error: 0.7009 - 1s/epoch - 3ms/step
Epoch 6/150
334/334 - 1s - loss: 0.6900 - mean_absolute_error: 0.7067 - val_loss: 0.6869 - val_mean_absolute_error: 0.7135 - 1s/epoch - 3ms/step
Epoch 7/150
334/334 - 1s - loss: 0.6857 - mean_absolute_error: 0.7061 - val_loss: 0.6805 - val_mean_absolute_error: 0.7117 - 1s/e

Epoch 57/150
334/334 - 1s - loss: 0.0171 - mean_absolute_error: 0.0284 - val_loss: 0.0127 - val_mean_absolute_error: 0.0344 - 987ms/epoch - 3ms/step
Epoch 58/150
334/334 - 1s - loss: 0.0167 - mean_absolute_error: 0.0292 - val_loss: 0.0113 - val_mean_absolute_error: 0.0211 - 1s/epoch - 3ms/step
Epoch 59/150
334/334 - 1s - loss: 0.0169 - mean_absolute_error: 0.0282 - val_loss: 0.0111 - val_mean_absolute_error: 0.0228 - 1s/epoch - 3ms/step
Epoch 60/150
334/334 - 1s - loss: 0.0167 - mean_absolute_error: 0.0298 - val_loss: 0.0114 - val_mean_absolute_error: 0.0232 - 1s/epoch - 3ms/step
Epoch 61/150
334/334 - 1s - loss: 0.0170 - mean_absolute_error: 0.0315 - val_loss: 0.0112 - val_mean_absolute_error: 0.0263 - 991ms/epoch - 3ms/step
Epoch 62/150
334/334 - 1s - loss: 0.0163 - mean_absolute_error: 0.0277 - val_loss: 0.0113 - val_mean_absolute_error: 0.0314 - 975ms/epoch - 3ms/step
Epoch 63/150
334/334 - 1s - loss: 0.0162 - mean_absolute_error: 0.0291 - val_loss: 0.0105 - val_mean_absolute_error

Epoch 112/150
334/334 - 1s - loss: 5.4145e-04 - mean_absolute_error: 0.0175 - val_loss: 3.5569e-04 - val_mean_absolute_error: 0.0143 - 1s/epoch - 3ms/step
Epoch 113/150
334/334 - 1s - loss: 4.9693e-04 - mean_absolute_error: 0.0168 - val_loss: 2.2480e-04 - val_mean_absolute_error: 0.0109 - 1s/epoch - 3ms/step
Epoch 114/150
334/334 - 1s - loss: 6.3125e-04 - mean_absolute_error: 0.0190 - val_loss: 0.0011 - val_mean_absolute_error: 0.0304 - 1s/epoch - 3ms/step
Epoch 115/150
334/334 - 1s - loss: 6.9614e-04 - mean_absolute_error: 0.0192 - val_loss: 2.0408e-04 - val_mean_absolute_error: 0.0106 - 1s/epoch - 3ms/step
Epoch 116/150
334/334 - 1s - loss: 4.1578e-04 - mean_absolute_error: 0.0155 - val_loss: 2.4300e-04 - val_mean_absolute_error: 0.0117 - 1s/epoch - 3ms/step
Epoch 117/150
334/334 - 1s - loss: 3.7096e-04 - mean_absolute_error: 0.0147 - val_loss: 2.5663e-04 - val_mean_absolute_error: 0.0122 - 1s/epoch - 3ms/step
Epoch 118/150
334/334 - 1s - loss: 5.0071e-04 - mean_absolute_error: 0.016

##### The results are good; obtained them on the 2nd try. There are minor oscillations around the middle and last epochs.

In [21]:
result1 = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

191/191 - 0s - loss: 3.1420e-04 - mean_absolute_error: 0.0147 - 252ms/epoch - 1ms/step


In [22]:
for i in test_labels:
    print(i)

1.5418086831519906
0.15830291739259367
1.781682711301299
0.3977131135037523
1.0932931928559868
0.42375911575066827
0.6431450248031988
0.30463851020492877
0.14050334727237151
2.146349701198934
0.011865072022205221
1.4159825092707412
1.7313895754151782
0.1270916012270449
2.2636048565511326
0.30414523279808414
0.5762110584450302
0.4029653077012798
0.49289110875505127
0.3956370068310924
2.5895105747615443
2.589110220241025
0.061837865776418116
0.6431450248031988
0.34649951145259505
0.18996487325337577
0.20293630238201488
0.5781665801612047
1.336153338327431
1.781682711301299
1.5418086831519906
1.1691860560178502
0.3533310229835167
0.07317057219210395
0.03668222476291964
2.429003090862608
0.05204205925125898
0.6431450248031988
1.0673919751339738
0.05204205925125898
0.6828589579420863
1.336153338327431
0.10112121459315382
2.146349701198934
0.17250311795260068
2.056139815315291
0.8640765771265051
1.987626234585897
1.191619397077886
0.4029653077012798
0.17684047979289388
0.012149623799903388
1

In [23]:
predictions1 = model.predict(x = test_samples, batch_size = 20, verbose = 2)

191/191 - 0s - 419ms/epoch - 2ms/step


In [24]:
for i in predictions1:
    print(i)

[1.5245409]
[0.1380511]
[1.7875733]
[0.41429156]
[1.0914251]
[0.44493788]
[0.650495]
[0.2871061]
[0.12114006]
[2.1275024]
[0.02200561]
[1.4398509]
[1.6994038]
[0.1082161]
[2.2586315]
[0.3183462]
[0.56744236]
[0.3895002]
[0.5048292]
[0.37323958]
[2.5861263]
[2.536924]
[0.04307299]
[0.650495]
[0.35746533]
[0.17396218]
[0.20070142]
[0.6048673]
[1.3449749]
[1.7875733]
[1.5245409]
[1.1579102]
[0.3378728]
[0.05422025]
[0.03111558]
[2.4263406]
[0.06939297]
[0.650495]
[1.0842448]
[0.06939476]
[0.6873942]
[1.3449749]
[0.09565096]
[2.1275024]
[0.15699512]
[2.05692]
[0.8506215]
[1.9774137]
[1.1803925]
[0.3895002]
[0.18842179]
[0.00118142]
[1.0842463]
[0.03585105]
[1.0286647]
[0.78277296]
[1.2471287]
[0.09565096]
[0.42264205]
[2.1275015]
[1.9644133]
[1.6446027]
[0.1082161]
[0.19244343]
[1.1579102]
[2.5922294]
[0.6499581]
[2.1275024]
[1.9644133]
[0.24003285]
[0.16172773]
[1.5245409]
[0.35746533]
[0.5048309]
[0.5048309]
[0.07470661]
[0.72932273]
[0.3378728]
[0.44818288]
[0.78277344]
[2.4263406]
[0.6

[0.5048309]
[0.00045495]
[0.1082161]
[1.0914251]
[0.20070142]
[0.07755607]
[1.1333071]
[0.27088457]
[2.2419405]
[1.8815563]
[0.15699512]
[1.2471287]
[1.9605377]
[0.12114006]
[1.8815563]
[0.03194194]
[1.0286647]
[0.25846666]
[1.3449749]
[0.44493788]
[1.7892025]
[2.536924]
[0.04307299]
[1.1579102]
[0.12114006]
[1.7892025]
[0.9100891]
[0.65049547]
[0.24003285]
[0.05422025]
[2.1275024]
[0.03194194]
[1.3449749]
[0.41429156]
[0.2708829]
[1.9644133]
[0.9163602]
[0.3183462]
[2.5861263]
[0.03111558]
[0.15346473]
[0.03585105]
[0.3865853]
[1.5245409]
[2.407021]
[0.2250281]
[0.8506225]
[0.3080123]
[2.1906323]
[2.1275024]
[0.60395366]
[0.02200561]
[0.05867677]
[0.05274301]
[1.9605377]
[0.22683531]
[0.16405731]
[1.4398509]
[0.13522607]
[1.3449749]
[2.3409505]
[0.00167053]
[0.03111558]
[2.407021]
[0.13522607]
[0.9016867]
[0.15346533]
[0.18842286]
[1.7875733]
[0.03585105]
[0.72932273]
[1.1803925]
[1.4398509]
[0.05274301]
[0.03585105]
[0.08127051]
[0.17396218]
[2.5922294]
[2.5538967]
[0.72932273]
[0.22

#### There were a very few negative predictions; those were most likely corresponding to the actual labels which were infinitesimal in value. Different predictions for different samples; difference between the actual and predicted labels was very low; patterns were almost captured.

In [25]:
result2 = model.evaluate(train_samples, train_labels, batch_size = 20, verbose = 2)

445/445 - 1s - loss: 3.2023e-04 - mean_absolute_error: 0.0149 - 705ms/epoch - 2ms/step


In [ ]:
for i in train_labels:
    print(i)

In [27]:
predictions2 = model.predict(x = train_samples, batch_size = 20, verbose = 2)

445/445 - 1s - 593ms/epoch - 1ms/step


In [ ]:
for i in predictions2:
    print(i)

#### Similar type of predictions as like the case with the test set. (Output was deleted to save memory)

### So, the change in model architecture gave a better performance on this dataset. Hence, we continue working using this particular architecture.

### Next, as a continuation of using the new architecture, we check how this architecture performs in the other experiment with the 2nd dataset where we used a smaller step size for the Univariate Cubic Polynomial samples.

## Architecture - Depth(20 layers in total), Width (40 nodes highest among the layers)<br>Combination of samples used - Cosine Wave Surface, Cubic Cylinder (Univariate Cubic Polynomial), Elliptic Paraboloid (z = ax^2 + by^2)
### Training & Testing from scratch on Max Valued Curvatures - exclusively on this dataset<br>Using a smaller step size for Cubic Cylinder samples; Trained for a higher number of epochs<br>"Unsigned" & "Split" Labels<br>Seeded expressions used & predictions checked

In [6]:
funct1 = exp_generator(4, 1)
funct1

-0.0445981827274462*cos(0.677404823452205*x - 1.24950596636108) - 0.920229845386649

In [7]:
s1, l_max1, l_min1 = datalist_generator(funct1, 101, 101, 150, 150, 1)

In [8]:
funct2 = exp_generator(6, 1)
funct2

2.13702896348076*x**2 + 2.51939920893562*y**2

In [9]:
s2, l_max2, l_min2 = datalist_generator(funct2, 101, 101, 150, 150, 1)

##### To properly evaluate the performance of the new architecture, we had to make sure all the variables and hyperparameters of this experiment except the model architecture remains the same in this iteration compared to the last one. That is why, the same range of coordinates were used like the last iteration.

In [10]:
s = np.concatenate((s1, s2))
l_max = np.concatenate((l_max1, l_max2))

s, l_max = shuffle(s, l_max)

train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s, l_max, test_size = 0.3, random_state = 2)

In [11]:
funct3 = exp_generator(1, 3)
funct3

0.290167489401872*x**3 + 0.720178969521534*x**2 - 4.13877058255077*x + 1.87192355899211

In [12]:
s3, l_max3, l_min3 = datalist_generator(funct3, -14, -14, 13, 13, 0.5)
train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s3, l_max3, test_size = 0.3, random_state = 17)

In [13]:
train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [14]:
m_s = np.mean(train_samples)
m_s

23536.078932716202

In [15]:
sd_s = np.std(train_samples)
sd_s

35247.499687166484

In [16]:
m_l = np.mean(train_labels)
m_l

0.06526540240716776

In [17]:
sd_l = np.std(train_labels)
sd_l

0.39647883565209047

In [24]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 32, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 40, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 32, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [25]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [26]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 100, shuffle = True, verbose = 2)

Epoch 1/100
211/211 - 8s - loss: 0.3889 - mean_absolute_error: 0.1547 - val_loss: 0.1817 - val_mean_absolute_error: 0.0713 - 8s/epoch - 37ms/step
Epoch 2/100
211/211 - 1s - loss: 0.1523 - mean_absolute_error: 0.0607 - val_loss: 0.1798 - val_mean_absolute_error: 0.0738 - 1s/epoch - 6ms/step
Epoch 3/100
211/211 - 1s - loss: 0.1508 - mean_absolute_error: 0.0665 - val_loss: 0.1780 - val_mean_absolute_error: 0.0819 - 1s/epoch - 6ms/step
Epoch 4/100
211/211 - 1s - loss: 0.1495 - mean_absolute_error: 0.0740 - val_loss: 0.1765 - val_mean_absolute_error: 0.0886 - 1s/epoch - 6ms/step
Epoch 5/100
211/211 - 1s - loss: 0.1486 - mean_absolute_error: 0.0831 - val_loss: 0.1755 - val_mean_absolute_error: 0.1013 - 1s/epoch - 6ms/step
Epoch 6/100
211/211 - 1s - loss: 0.1477 - mean_absolute_error: 0.0850 - val_loss: 0.1739 - val_mean_absolute_error: 0.0955 - 1s/epoch - 6ms/step
Epoch 7/100
211/211 - 1s - loss: 0.1467 - mean_absolute_error: 0.0883 - val_loss: 0.1727 - val_mean_absolute_error: 0.1046 - 1s/e

211/211 - 1s - loss: 0.0134 - mean_absolute_error: 0.0380 - val_loss: 0.0102 - val_mean_absolute_error: 0.0324 - 1s/epoch - 6ms/step
Epoch 58/100
211/211 - 1s - loss: 0.0132 - mean_absolute_error: 0.0305 - val_loss: 0.0100 - val_mean_absolute_error: 0.0262 - 1s/epoch - 6ms/step
Epoch 59/100
211/211 - 1s - loss: 0.0104 - mean_absolute_error: 0.0358 - val_loss: 0.0101 - val_mean_absolute_error: 0.0353 - 1s/epoch - 6ms/step
Epoch 60/100
211/211 - 1s - loss: 0.0224 - mean_absolute_error: 0.0669 - val_loss: 0.0323 - val_mean_absolute_error: 0.0984 - 1s/epoch - 5ms/step
Epoch 61/100
211/211 - 1s - loss: 0.0120 - mean_absolute_error: 0.0420 - val_loss: 0.0106 - val_mean_absolute_error: 0.0343 - 1s/epoch - 6ms/step
Epoch 62/100
211/211 - 1s - loss: 0.0118 - mean_absolute_error: 0.0398 - val_loss: 0.0160 - val_mean_absolute_error: 0.0371 - 1s/epoch - 6ms/step
Epoch 63/100
211/211 - 1s - loss: 0.0179 - mean_absolute_error: 0.0475 - val_loss: 0.0157 - val_mean_absolute_error: 0.0551 - 1s/epoch - 

##### Received these results after third attempt. There was constant oscillation of the results; however, they had a general downward trend.

In [27]:
result1 = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

121/121 - 0s - loss: 0.0259 - mean_absolute_error: 0.0309 - 311ms/epoch - 3ms/step


In [28]:
for i in test_labels:
    print(i)

0.013017061327536102
0.041097777496099726
0.3517144185987097
0.0002578610102518507
0.019763682532969354
8.417339503402771e-05
0.00547299513569348
0.002777955589888771
0.00017201126773013704
5.693461829688935e-06
0.005028541398883574
0.005624688573690399
0.016976979026165507
0.005455069630971287
0.005622461160534326
0.01261352095067763
0.0055888756717811655
0.006370891783502074
0.006043817849117212
0.0006614759046012209
0.00541068047803558
0.0052305259604171934
0.00019420088912512864
2.6282409205329398e-05
0.0022210961865479937
1.2843416008384391e-05
0.013251082303644308
0.01962280501843827
0.016151278226838543
0.005317371973763716
0.006370891783502074
0.00497773483698952
0.0022210961865479937
0.0026954744902400346
8.953646456373955e-06
0.061272552794954756
2.2069575884167174e-05
0.004714869379522877
0.005659841161622407
0.0002945576257815709
0.005175691264607424
0.020360548409069913
8.417339503402771e-05
2.045087282176784e-05
0.00017201126773013704
0.005137448951471406
0.18480043807439

##### Even though we had decided earlier that we'll put restriction on our input generation in order to avoid samples with very small curvatures, like of the order 10^-6, we didn't do that for this iteration of the experiment to keep the variables and hyperparameters the same as the last iteration.

In [29]:
predictions1 = model.predict(x = test_samples, batch_size = 20, verbose = 2)

121/121 - 1s - 684ms/epoch - 6ms/step


In [30]:
for i in predictions1:
    print(i)

[0.01367592]
[0.5878628]
[0.38189614]
[-0.00568273]
[0.00893091]
[0.00364243]
[0.01306628]
[0.00056099]
[-0.0017679]
[-0.00902289]
[0.01141332]
[0.01267217]
[0.01971291]
[0.01351379]
[0.01353406]
[0.00995313]
[0.01363038]
[0.01243268]
[0.0147426]
[-0.0018442]
[0.01200341]
[0.01273512]
[0.00082277]
[-0.00170752]
[0.43377405]
[-0.01527264]
[0.00870429]
[0.0226339]
[0.01875554]
[0.01187705]
[0.01243268]
[0.01036762]
[0.43377405]
[-0.00216438]
[0.00082277]
[0.05600958]
[0.00082277]
[0.01233648]
[0.01275681]
[0.00082277]
[0.01130508]
[0.02328836]
[0.00364243]
[-0.00617444]
[-0.0017679]
[0.01109122]
[0.5471698]
[0.01625298]
[0.01722561]
[0.01408182]
[-2.7371978e-05]
[0.00825249]
[0.00231541]
[0.014181]
[0.01251601]
[0.0132017]
[-0.01326611]
[0.01190208]
[0.00995325]
[0.01380061]
[0.01765238]
[0.01233648]
[0.01243268]
[-0.0017679]
[0.00082277]
[0.00888299]
[0.01169692]
[0.01400314]
[0.01536261]
[0.01537429]
[0.00056099]
[0.01295661]
[0.00082277]
[0.01329635]
[0.01590644]
[0.01271819]
[0.01363

[0.01235794]
[0.00193199]
[0.01423572]
[0.04288439]
[0.38189447]
[0.01425599]
[0.05600958]
[-0.00568273]
[0.01036941]
[0.02110468]
[0.00499432]
[0.00056099]
[0.01021098]
[0.02315055]
[0.38189614]
[0.01351021]
[0.00927375]
[0.01116489]
[-0.01417664]
[0.01103114]
[0.00893091]
[0.00995313]
[0.00193199]
[0.01379441]
[0.01295565]
[0.01184939]
[0.01453851]
[0.01255535]
[0.00082277]
[0.00893091]
[0.00082277]
[0.00930606]
[0.01131033]
[0.01168965]
[-0.0017679]
[0.0226339]
[0.01225947]
[0.00794266]
[0.02304553]
[0.01423846]
[0.00082277]
[0.01222919]
[0.01366113]
[0.00961755]
[0.01480256]
[0.01367592]
[0.01011132]
[0.01297854]
[0.01378702]
[0.01539444]
[0.01615142]
[0.01875554]
[0.01395343]
[-2.7371978e-05]
[0.01239275]
[0.01237296]
[0.00896703]
[0.38189614]
[0.02243351]
[0.00825249]
[-0.01326611]
[0.01149486]
[0.01190208]
[0.02073191]
[-0.00617438]
[0.01428507]
[0.04481171]
[0.35231698]
[0.01971291]
[0.01195835]
[0.00888299]
[0.01644014]
[0.00957249]
[0.01674198]
[0.01329635]
[0.00960945]
[0.00

#### There is significant difference between the predictions and actual labels, mostly in case of the samples with low label values. In general, this model gave different predictions for different samples - which is a good thing; for a few samples with really low curvature values, a specific prediction was obtained. Unfortunately, there are quite a number of negative predictions.

In [31]:
result2 = model.evaluate(train_samples, train_labels, batch_size = 20, verbose = 2)

281/281 - 1s - loss: 0.0292 - mean_absolute_error: 0.0335 - 721ms/epoch - 3ms/step


In [32]:
for i in train_labels:
    print(i)

0.0004026373761829719
8.953646456373955e-06
0.017151349391302917
0.005173074604922643
0.016976979026165507
5.693461829688935e-06
0.013539935474920849
0.020462199457089044
2.6282409205329398e-05
0.009864161384152725
0.012636557468015447
8.417339503402771e-05
0.015383374244456963
4.9651895465459124e-05
8.953646456373955e-06
9.334697742103088e-05
1.2843416008384391e-05
0.018030138190115157
0.04623998935243249
0.0050355200503129326
0.00481506134712419
0.0035361037193085173
3.729702488783223e-05
0.00555359520098099
0.061272552794954756
0.005843438623447841
0.005107996350454189
6.115814541154617e-05
0.00522620867758241
0.005788381915061624
0.006370891783502074
0.0026954744902400346
0.005325091192914046
0.005752932203316425
0.0035361037193085173
0.020251388912184476
0.009407740691296976
0.005500345528539034
0.33949960206294283
0.061272552794954756
3.729702488783223e-05
0.020251388912184476
0.005980220634793628
0.020390032523054928
0.00554168194545571
0.01164355214679571
0.0002578610102518507


In [33]:
predictions2 = model.predict(x = train_samples, batch_size = 20, verbose = 2)

281/281 - 1s - 617ms/epoch - 2ms/step


In [34]:
for i in predictions2:
    print(i)

[-0.00564959]
[0.00082277]
[0.01819407]
[0.01113557]
[0.01971291]
[-0.00902289]
[0.01566588]
[0.02324055]
[-0.00170752]
[0.01295661]
[0.00825249]
[0.00364243]
[0.00958894]
[0.00082277]
[0.00082277]
[0.00082277]
[-0.01527264]
[0.04288374]
[0.04481111]
[0.01085137]
[-0.00332172]
[0.01112043]
[0.00082277]
[0.01238202]
[0.05600958]
[0.01340173]
[0.01178955]
[0.00499432]
[0.01252006]
[0.01397537]
[0.01243268]
[-0.00216438]
[0.0128045]
[0.01442741]
[0.01112043]
[0.00888299]
[0.01036941]
[0.01278947]
[0.35231745]
[0.05600952]
[0.00082277]
[0.00888299]
[0.01531719]
[0.02304553]
[0.01349949]
[0.01413273]
[-0.00568273]
[0.01039456]
[0.01082324]
[0.00870429]
[0.0138266]
[0.01230715]
[0.02328836]
[0.02324055]
[0.5471698]
[0.01362394]
[0.00082277]
[0.03952569]
[0.05600952]
[0.0187553]
[0.01233648]
[0.01394127]
[0.00231541]
[0.01065181]
[0.01875554]
[0.01699899]
[0.00082277]
[0.01546441]
[-0.0114613]
[0.01135491]
[-0.01326611]
[0.01169692]
[-0.00216438]
[0.01566588]
[0.01112043]
[0.01317047]
[0.0008

[0.01166581]
[-2.7371978e-05]
[0.00082277]
[0.00991605]
[0.00907062]
[0.00082277]
[-0.01326611]
[0.00729428]
[0.00082277]
[0.00056099]
[0.02315115]
[0.01688455]
[0.00231541]
[0.02328836]
[0.02304553]
[0.01229905]
[0.0091729]
[0.01620852]
[0.01330613]
[0.00781296]
[0.00729428]
[0.00082277]
[-0.00617444]
[0.0127921]
[0.01455079]
[0.01590239]
[0.00825249]
[0.01285957]
[0.00082277]
[0.5471698]
[0.00888299]
[0.00893043]
[0.00082277]
[0.01169692]
[0.00888299]
[0.38189614]
[-0.00216438]
[0.01065587]
[0.01573347]
[0.00082277]
[-0.01527264]
[0.52523977]
[-0.00617444]
[-0.0018442]
[0.0107591]
[0.01270043]
[0.00888299]
[0.01487778]
[0.35231698]
[0.03952569]
[0.00870447]
[0.00082277]
[0.01233648]
[0.01446329]
[0.01092099]
[0.02328836]
[0.04481171]
[-0.0018442]
[0.01209305]
[0.01036941]
[0.00888299]
[0.00231541]
[0.02324055]
[0.01236771]
[0.02324055]
[0.00193199]
[0.00082277]
[0.01442205]
[0.00082277]
[0.01072692]
[0.00879549]
[0.52523935]
[0.00995313]
[0.01039456]
[0.00082277]
[0.00082277]
[0.0047

[0.00983451]
[0.01215314]
[0.01819407]
[0.01434706]
[-0.00568273]
[0.43377405]
[-0.00564959]
[0.01244615]
[0.01367592]
[0.01417051]
[0.01223706]
[0.05600958]
[0.01160024]
[0.05600958]
[0.01190208]
[0.01208233]
[0.01620935]
[0.35231698]
[0.5878628]
[0.01270436]
[0.43377405]
[0.01492571]
[0.01147877]
[0.5878628]
[0.0120356]
[0.01474272]
[0.35231698]
[-2.7371978e-05]
[0.01395677]
[0.01160024]
[0.35231698]
[0.01169692]
[0.01688419]
[0.01268338]
[-0.00568273]
[-0.0114613]
[0.52523935]
[0.00474475]
[0.01163124]
[0.01112043]
[0.01021098]
[0.00887989]
[-0.0017679]
[0.52523977]
[0.01024078]
[0.00082277]
[-0.00617444]
[0.01519095]
[-0.00617444]
[0.00825249]
[0.00082277]
[0.01112043]
[0.01705454]
[-0.00216438]
[0.00906621]
[0.00893043]
[0.0152676]
[0.00825249]
[0.02073275]
[0.01023577]
[0.01051735]
[0.5471698]
[0.0112028]
[-0.00568273]
[0.015056]
[0.01243268]
[0.00082277]
[0.01554679]
[0.00082277]
[0.00896703]
[0.00082277]
[-0.00902289]
[0.01021098]
[-0.0114613]
[-2.7371978e-05]
[0.00082277]
[0.0

#### Similar behavior as like in the case of test set.

### Though the performance isn't great, the new architecture still gives better results than the previous architecture, which indicates that this architecture has potential. Hence, we decide to continue our work with this architecture.

### So now, we redo our experiments with combination of samples from scratch using this new architecture, starting with the combination of Sine Wave, Parabolic Cylinder (Univariate Quadratic Polynomial), and Circular Paraboloid samples, but this time we use a smaller step size for the low curvature-valued Parabolic Cylinder samples. We also conduct the training for a higher number of epochs.

## Working from the very beginning with the experiments involving combination of samples using the new architecture<br>Architecture - Depth(20 layers in total), Width (40 nodes highest among the layers)<br>Combination of samples used - Sine Wave Surface, Parabolic Cylinder (Quadratic Univariate Polynomial), Circular Paraboloid (z = x^2 + y^2)
### "Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked<br>Using a smaller step size for Parabolic Cylinder samples; Trained for a higher number of epochs

In [7]:
funct1 = exp_generator(3, 1)
funct1

0.593988879915532*sin(2.01369540968675*x - 4.57047002658068) - 0.607193781823679

In [9]:
s1, l_max1, l_min1 = datalist_generator(funct1, 1, 1, 96, 96, 1)

In [10]:
funct2 = exp_generator(5, 1)
funct2

x**2 + y**2

In [11]:
s2, l_max2, l_min2 = datalist_generator(funct2, 10, 10, 105, 105, 1)

In [12]:
s = np.concatenate((s1, s2))
l_max = np.concatenate((l_max1, l_max2))

s, l_max = shuffle(s, l_max)

train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s, l_max, test_size = 0.3, random_state = 6)

In [13]:
funct3 = exp_generator(1, 2)
funct3

0.204219866843413*x**2 - 0.210374622235534*x + 1.59664462678772

In [15]:
s3, l_max3, l_min3 = datalist_generator(funct3, -10, -10, 45, 45, 0.5)

train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s3, l_max3, test_size = 0.3, random_state = 21)

In [16]:
train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [17]:
m_s = np.mean(train_samples)
m_s

2499.16872773777

In [18]:
sd_s = np.std(train_samples)
sd_s

4506.786564187108

In [19]:
m_l = np.mean(train_labels)
m_l

0.3105299565759982

In [20]:
sd_l = np.std(train_labels)
sd_l

0.6177219607115388

In [21]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 32, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 40, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 32, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [22]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [23]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 125, shuffle = True, verbose = 2)

Epoch 1/125
808/808 - 4s - loss: 1.1584 - mean_absolute_error: 0.4267 - val_loss: 0.1930 - val_mean_absolute_error: 0.2360 - 4s/epoch - 5ms/step
Epoch 2/125
808/808 - 2s - loss: 0.1033 - mean_absolute_error: 0.1557 - val_loss: 0.0318 - val_mean_absolute_error: 0.0908 - 2s/epoch - 2ms/step
Epoch 3/125
808/808 - 2s - loss: 0.0190 - mean_absolute_error: 0.0680 - val_loss: 0.0118 - val_mean_absolute_error: 0.0569 - 2s/epoch - 3ms/step
Epoch 4/125
808/808 - 2s - loss: 0.0082 - mean_absolute_error: 0.0458 - val_loss: 0.0059 - val_mean_absolute_error: 0.0372 - 2s/epoch - 2ms/step
Epoch 5/125
808/808 - 2s - loss: 0.0050 - mean_absolute_error: 0.0352 - val_loss: 0.0040 - val_mean_absolute_error: 0.0335 - 2s/epoch - 2ms/step
Epoch 6/125
808/808 - 1s - loss: 0.0031 - mean_absolute_error: 0.0284 - val_loss: 0.0032 - val_mean_absolute_error: 0.0299 - 1s/epoch - 2ms/step
Epoch 7/125
808/808 - 2s - loss: 0.0017 - mean_absolute_error: 0.0221 - val_loss: 0.0010 - val_mean_absolute_error: 0.0176 - 2s/ep

Epoch 55/125
808/808 - 2s - loss: 8.9204e-05 - mean_absolute_error: 0.0066 - val_loss: 5.5616e-05 - val_mean_absolute_error: 0.0056 - 2s/epoch - 2ms/step
Epoch 56/125
808/808 - 2s - loss: 1.1711e-04 - mean_absolute_error: 0.0074 - val_loss: 5.3778e-05 - val_mean_absolute_error: 0.0057 - 2s/epoch - 3ms/step
Epoch 57/125
808/808 - 2s - loss: 9.2875e-05 - mean_absolute_error: 0.0068 - val_loss: 1.1498e-04 - val_mean_absolute_error: 0.0082 - 2s/epoch - 2ms/step
Epoch 58/125
808/808 - 2s - loss: 1.1303e-04 - mean_absolute_error: 0.0070 - val_loss: 1.1607e-04 - val_mean_absolute_error: 0.0085 - 2s/epoch - 3ms/step
Epoch 59/125
808/808 - 2s - loss: 1.0179e-04 - mean_absolute_error: 0.0070 - val_loss: 1.0667e-04 - val_mean_absolute_error: 0.0070 - 2s/epoch - 2ms/step
Epoch 60/125
808/808 - 2s - loss: 9.7874e-05 - mean_absolute_error: 0.0069 - val_loss: 5.9954e-05 - val_mean_absolute_error: 0.0060 - 2s/epoch - 2ms/step
Epoch 61/125
808/808 - 1s - loss: 1.2514e-04 - mean_absolute_error: 0.0072 -

Epoch 109/125
808/808 - 2s - loss: 9.8869e-05 - mean_absolute_error: 0.0066 - val_loss: 5.4206e-05 - val_mean_absolute_error: 0.0056 - 2s/epoch - 2ms/step
Epoch 110/125
808/808 - 2s - loss: 8.1775e-05 - mean_absolute_error: 0.0062 - val_loss: 4.8802e-05 - val_mean_absolute_error: 0.0051 - 2s/epoch - 3ms/step
Epoch 111/125
808/808 - 2s - loss: 7.5635e-05 - mean_absolute_error: 0.0061 - val_loss: 1.2681e-04 - val_mean_absolute_error: 0.0080 - 2s/epoch - 2ms/step
Epoch 112/125
808/808 - 2s - loss: 9.2834e-05 - mean_absolute_error: 0.0065 - val_loss: 2.0269e-04 - val_mean_absolute_error: 0.0106 - 2s/epoch - 2ms/step
Epoch 113/125
808/808 - 2s - loss: 1.6044e-04 - mean_absolute_error: 0.0076 - val_loss: 4.3446e-05 - val_mean_absolute_error: 0.0046 - 2s/epoch - 2ms/step
Epoch 114/125
808/808 - 2s - loss: 5.3812e-05 - mean_absolute_error: 0.0052 - val_loss: 4.6895e-05 - val_mean_absolute_error: 0.0051 - 2s/epoch - 2ms/step
Epoch 115/125
808/808 - 2s - loss: 8.7870e-05 - mean_absolute_error: 0

##### Results are good. However, they were continuously in oscillation across almost all of the training epochs.

In [24]:
model.save('Research_mcomb.h5') # Overwritten in later experiments, as this one yielded a failed model

In [25]:
result1 = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

462/462 - 0s - loss: 5.9136e-05 - mean_absolute_error: 0.0057 - 462ms/epoch - 1000us/step


In [26]:
for i in test_labels:
    print(i)

0.1693525197521385
0.027283905293899607
0.013914074578324727
2.405157376276527
0.19144158851282628
0.0006356435361348629
0.2706262034240012
0.006377056356736636
0.012065500411434702
0.4653030052959377
0.013385489628205093
0.0025981677244649127
0.6354018266976316
0.42449966701694725
0.001417443013775595
1.1681183350534432
2.0044595303392625
0.0001903305044877379
7.51078486029046e-05
0.0018907926941605807
0.008043292908057019
0.04116645244171622
0.18800903613731773
0.9711488571467721
0.8109447896194174
0.9096141284141153
0.0007343980543634965
0.0012949982914153728
0.16522466048792264
0.0073532204373310305
1.781726235842214
0.012969252307032712
0.0005180729107614556
0.0018907926941605807
1.781726235842214
1.8133039901847174
2.2948705367927276
0.9096141284141153
0.009237013157108152
2.4032664081451545
0.007835292812796678
0.013226483448808496
0.010845725440437968
2.3038528271841914
0.2511464934662333
0.017779973317363508
0.014614019946498306
0.9209225254743553
0.3219574621901694
0.00059283

In [27]:
predictions1 = model.predict(x = test_samples, batch_size = 20, verbose = 2)

462/462 - 1s - 774ms/epoch - 2ms/step


In [28]:
for i in predictions1:
    print(i)

[0.17617]
[0.02872472]
[0.0079399]
[2.3960562]
[0.19413835]
[0.0079399]
[0.27374315]
[0.0079399]
[0.0079399]
[0.457575]
[0.0079399]
[0.0079399]
[0.63241386]
[0.42002183]
[0.0079399]
[1.1617758]
[2.006077]
[0.0079399]
[0.0079399]
[0.0079399]
[0.0079399]
[0.04368911]
[0.1939215]
[0.96761453]
[0.81950605]
[0.90638924]
[0.0079399]
[0.0079399]
[0.15175083]
[0.0079399]
[1.7734108]
[0.0079399]
[0.0079399]
[0.0079399]
[1.7734108]
[1.8038504]
[2.2948115]
[0.90638924]
[0.0079399]
[2.400214]
[0.03594638]
[0.0079399]
[0.0079399]
[2.3008451]
[0.25966075]
[0.01939461]
[0.01683433]
[0.9171541]
[0.33172646]
[0.0079399]
[0.0079399]
[1.1364121]
[1.4730211]
[0.14459564]
[0.0079399]
[0.01108824]
[1.3516263]
[0.0079399]
[0.0079399]
[0.0079399]
[0.0079399]
[0.05931527]
[0.0079399]
[0.0079399]
[0.0079399]
[0.0079399]
[0.42943412]
[0.0079399]
[0.42943412]
[2.2058666]
[0.02269159]
[1.7956034]
[0.0079399]
[0.96761453]
[0.0079399]
[0.556623]
[0.0079399]
[0.0079399]
[0.39066663]
[0.96761364]
[0.0079399]
[1.914117

[0.0079399]
[0.04368911]
[0.0079399]
[0.0079399]
[0.0079399]
[0.0079399]
[1.091714]
[2.3875346]
[0.0079399]
[0.0079399]
[0.0079399]
[0.26845095]
[0.0079399]
[0.0079399]
[0.0079399]
[0.0079399]
[0.3903391]
[0.754517]
[0.0079399]
[0.0079399]
[0.0079399]
[0.0079399]
[2.2948115]
[0.0079399]
[0.3903387]
[0.01939461]
[0.0079399]
[0.0079399]
[0.0079399]
[0.0079399]
[2.1377454]
[0.0079399]
[1.7049582]
[1.9451497]
[0.0079399]
[0.23804204]
[0.02872472]
[0.39066693]
[0.0079399]
[0.0079399]
[0.0079399]
[1.5609999]
[0.0079399]
[0.0079399]
[2.3844914]
[0.42002195]
[1.9451497]
[0.28280455]
[0.0079399]
[2.2058666]
[0.63241386]
[0.0079399]
[0.0079399]
[0.05410771]
[0.02269159]
[0.0079399]
[0.0079399]
[0.0079399]
[0.0079399]
[0.8150598]
[0.96761453]
[0.0079399]
[0.3903391]
[0.90638924]
[1.9451497]
[0.0079399]
[0.01683433]
[0.0079399]
[0.0079399]
[0.08958651]
[0.0079399]
[0.0079399]
[0.01397392]
[0.0079399]
[0.0079399]
[0.40952373]
[0.0079399]
[0.0079399]
[0.0079399]
[0.1477693]
[0.57932246]
[0.0079399]


#### The predictions are similar for a lot of different samples. However, one notable aspect was that the model worked well in predicting the curvatures of the samples that had actual labels of higher value.

In [29]:
result2 = model.evaluate(train_samples, train_labels, batch_size = 20, verbose = 2)

1077/1077 - 1s - loss: 6.4226e-05 - mean_absolute_error: 0.0059 - 1s/epoch - 979us/step


In [ ]:
for i in train_labels:
    print(i)

In [31]:
predictions2 = model.predict(x = train_samples, batch_size = 20, verbose = 2)

1077/1077 - 1s - 852ms/epoch - 791us/step


In [ ]:
for i in predictions2:
    print(i)

#### Similar type of predictions as like the test set.

### Unfortunately, this architecture fails to perform when working with this particular dataset, which is weird, given that the previous architecture performed well on this dataset, but struggled with the other 2 datasets, while it is the opposite in the case of the current architecture.

### To better understand the reason behind the poor performance of the current architecture on our first dataset, next we perform the same experiment again, but first with a smaller dataset, to see if the model can figure out patterns in our dataset when there are less outliers, and then if the smaller dataset doesn't work, with a much larger dataset, to see whether the model requires more data to understand patterns in our data.

## Using a smaller dataset<br>Combination of samples used - Sine Wave Surface, Parabolic Cylinder (Quadratic Univariate Polynomial), Circular Paraboloid (z = x^2 + y^2)
### Architecture - Depth(20 layers in total), Width (40 nodes highest among the layers)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked<br>Using a smaller step size for Parabolic Cylinder samples; Trained for a higher number of epochs

In [6]:
funct1 = exp_generator(3, 1)
funct1

0.593988879915532*sin(2.01369540968675*x - 4.57047002658068) - 0.607193781823679

In [7]:
s1, l_max1, l_min1 = datalist_generator(funct1, 31, 31, 80, 80, 1)

In [8]:
funct2 = exp_generator(5, 1)
funct2

x**2 + y**2

In [9]:
s2, l_max2, l_min2 = datalist_generator(funct2, 70, 70, 125, 125, 1)

In [10]:
s = np.concatenate((s1, s2))
l_max = np.concatenate((l_max1, l_max2))

s, l_max = shuffle(s, l_max)

train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s, l_max, test_size = 0.3, random_state = 6)

In [11]:
funct3 = exp_generator(1, 2)
funct3

0.204219866843413*x**2 - 0.210374622235534*x + 1.59664462678772

In [12]:
s3, l_max3, l_min3 = datalist_generator(funct3, -10, -10, 15, 15, 0.5)

train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s3, l_max3, test_size = 0.3, random_state = 21)

In [13]:
train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [14]:
m_s = np.mean(train_samples)
m_s

7434.6389497552955

In [15]:
sd_s = np.std(train_samples)
sd_s

9882.45191877329

In [16]:
m_l = np.mean(train_labels)
m_l

0.3278904009741813

In [17]:
sd_l = np.std(train_labels)
sd_l

0.6213722726093861

In [18]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 32, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 40, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 32, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [19]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 125, shuffle = True, verbose = 2)

##### Results are good enough. They converged gradually until the training loss reached a low value in the range of 10^-5, after which they started oscillating. (Output deleted to save memory)

In [21]:
model.save('Research_mcomb.h5') # Overwritten again in a later experiment, as the model failed in this case too.

In [22]:
result1 = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

124/124 - 0s - loss: 1.0946e-05 - mean_absolute_error: 0.0021 - 328ms/epoch - 3ms/step


In [23]:
for i in test_labels:
    print(i)

0.5293148806682709
0.007422323289873542
0.0068184767870779995
0.00729863819203575
0.07759248637902613
0.0020945780488848867
0.006908184683211419
0.007601486235852868
0.3219574621901694
0.004763364832230142
0.008502328628439184
1.36030663860646
0.0069161106085124085
0.13862153170143926
0.006886691407188246
0.0074450340386571084
0.0061124363234293586
0.10417930577530653
0.008569496280424605
0.00768407105424199
0.05859359763850934
0.9209225254743553
0.5293148806682709
0.00622509472866811
0.008415478349147743
1.567568215798886
1.084504328072518
0.004802425537784127
0.13862153170143926
0.007572227386968011
0.006117466712636235
0.56169754076809
0.3509385613655042
0.007308768024516221
0.11540148396937845
0.5293148806682709
2.1449126730981924
0.3855321534430764
0.0072164555380657795
0.008705779149402841
0.3855321534430764
0.3509385613655042
0.23223004445970752
0.11154062043962387
0.43711677227308515
2.1982866505352483
1.36030663860646
2.2948705367927276
0.0072149527555392485
0.1516501151370714

In [24]:
predictions1 = model.predict(x = test_samples, batch_size = 20, verbose = 2)

124/124 - 1s - 685ms/epoch - 6ms/step


In [25]:
for i in predictions1:
    print(i)

[0.53173506]
[0.00668206]
[0.00668206]
[0.00668206]
[0.07979382]
[0.00668206]
[0.00668206]
[0.00668206]
[0.33031073]
[0.00668206]
[0.00668206]
[1.3604331]
[0.00668206]
[0.13910069]
[0.00668206]
[0.00668206]
[0.00668206]
[0.10343241]
[0.00668206]
[0.00668206]
[0.06059055]
[0.9201706]
[0.53173506]
[0.00668206]
[0.00668206]
[1.5660201]
[1.0817616]
[0.00668206]
[0.13910069]
[0.00668206]
[0.00668206]
[0.5626652]
[0.34836444]
[0.00668206]
[0.10736697]
[0.53173506]
[2.148931]
[0.38576594]
[0.00668206]
[0.00668206]
[0.38576594]
[0.34836444]
[0.22965723]
[0.10198215]
[0.43969032]
[2.1955445]
[1.3604331]
[2.29337]
[0.00668206]
[0.14971006]
[0.00668206]
[0.00668206]
[0.00668206]
[0.9688489]
[0.06008249]
[0.29603365]
[0.00668206]
[0.46716148]
[0.00668206]
[0.00947712]
[0.04452523]
[0.02818462]
[0.00668206]
[0.00668206]
[0.01388457]
[0.07979382]
[0.21051389]
[1.0957494]
[0.03620917]
[0.02818462]
[1.5013125]
[2.3337862]
[0.18860911]
[0.34836444]
[0.454878]
[0.2542586]
[0.00668206]
[0.33031073]
[2.29

[0.00668206]
[1.0957494]
[0.00668206]
[0.19133726]
[0.2579783]
[0.00668206]
[0.03620917]
[0.01098644]
[0.00668206]
[0.06797428]
[0.02818462]
[0.32608634]
[0.00668206]
[0.9201697]
[0.00668206]
[0.1439114]
[0.01098644]
[0.00668206]
[0.01778291]
[0.00668206]
[0.00668206]
[0.00668206]
[0.00668206]
[0.00668206]
[2.29337]
[0.10343241]
[0.04452523]
[0.00668206]
[0.00668206]
[0.00668206]
[2.148931]
[0.00668206]
[0.00668206]
[1.5013113]
[0.00995742]
[0.00668206]
[2.1955445]
[0.01243837]
[0.00668206]
[0.3871914]
[0.2070235]
[0.01098644]
[0.01952006]
[0.40725848]
[0.9688489]
[0.10299318]
[0.00668206]
[0.03620917]
[0.8227431]
[0.22965723]
[1.1471004]
[0.00947715]
[0.5626652]
[0.25797802]
[0.00668206]
[0.06139949]
[0.07979382]
[0.06139949]
[2.3337862]
[0.00668206]
[0.00668206]
[1.5660201]
[1.1471004]
[1.0957494]
[0.07979382]
[0.00831209]
[1.3604331]
[0.00668206]
[0.01388457]
[0.00668206]
[0.00668206]
[2.300969]
[0.27447125]
[0.29603344]
[0.32608616]
[0.08932724]
[1.9945594]
[0.00668206]
[0.00668206

#### Same prediction for different samples that have different actual labels.

In [26]:
result2 = model.evaluate(train_samples, train_labels, batch_size = 20, verbose = 2)

289/289 - 1s - loss: 1.1971e-05 - mean_absolute_error: 0.0022 - 716ms/epoch - 2ms/step


In [ ]:
for i in train_labels:
    print(i)

In [28]:
predictions2 = model.predict(x = train_samples, batch_size = 20, verbose = 2)

289/289 - 1s - 633ms/epoch - 2ms/step


In [ ]:
for i in predictions2:
    print(i)

#### Similar behavior like that for test set.

### So we see that the model fails even with a smaller dataset. Next, we look at its performance with a much larger dataset than the one used in the original experiment.

## Using a much larger dataset<br>Combination of samples used - Sine Wave Surface, Parabolic Cylinder (Quadratic Univariate Polynomial), Circular Paraboloid (z = x^2 + y^2)
### Architecture - Depth(20 layers in total), Width (40 nodes highest among the layers)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked<br>Using a smaller step size for Parabolic Cylinder samples; Trained for a higher number of epochs

In [30]:
funct1 = exp_generator(3, 1)
funct1

0.593988879915532*sin(2.01369540968675*x - 4.57047002658068) - 0.607193781823679

In [31]:
s1, l_max1, l_min1 = datalist_generator(funct1, 5, 5, 130, 130, 1)

In [32]:
funct2 = exp_generator(5, 1)
funct2

x**2 + y**2

In [33]:
s2, l_max2, l_min2 = datalist_generator(funct2, 1, 1, 126, 126, 1)

In [34]:
s = np.concatenate((s1, s2))
l_max = np.concatenate((l_max1, l_max2))

s, l_max = shuffle(s, l_max)

train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s, l_max, test_size = 0.3, random_state = 6)

In [35]:
funct3 = exp_generator(1, 2)
funct3

0.204219866843413*x**2 - 0.210374622235534*x + 1.59664462678772

In [36]:
s3, l_max3, l_min3 = datalist_generator(funct3, -14, -14, 51, 51, 0.5)

train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s3, l_max3, test_size = 0.3, random_state = 21)

In [37]:
train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [38]:
m_s = np.mean(train_samples)
m_s

3917.4858335951294

In [39]:
sd_s = np.std(train_samples)
sd_s

6564.805077046875

In [40]:
m_l = np.mean(train_labels)
m_l

0.3736419143388127

In [41]:
sd_l = np.std(train_labels)
sd_l

0.6597117196958283

In [42]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 32, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 40, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 32, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [43]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [44]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 125, shuffle = True, verbose = 2)

Epoch 1/125
1147/1147 - 13s - loss: 1.6324 - mean_absolute_error: 0.4287 - val_loss: 0.0244 - val_mean_absolute_error: 0.0864 - 13s/epoch - 12ms/step
Epoch 2/125
1147/1147 - 7s - loss: 0.0128 - mean_absolute_error: 0.0624 - val_loss: 0.0071 - val_mean_absolute_error: 0.0467 - 7s/epoch - 6ms/step
Epoch 3/125
1147/1147 - 7s - loss: 0.0040 - mean_absolute_error: 0.0351 - val_loss: 0.0029 - val_mean_absolute_error: 0.0291 - 7s/epoch - 6ms/step
Epoch 4/125
1147/1147 - 7s - loss: 0.0025 - mean_absolute_error: 0.0260 - val_loss: 0.0020 - val_mean_absolute_error: 0.0217 - 7s/epoch - 6ms/step
Epoch 5/125
1147/1147 - 7s - loss: 0.0018 - mean_absolute_error: 0.0211 - val_loss: 0.0016 - val_mean_absolute_error: 0.0198 - 7s/epoch - 6ms/step
Epoch 6/125
1147/1147 - 7s - loss: 0.0013 - mean_absolute_error: 0.0176 - val_loss: 0.0011 - val_mean_absolute_error: 0.0169 - 7s/epoch - 6ms/step
Epoch 7/125
1147/1147 - 7s - loss: 7.9101e-04 - mean_absolute_error: 0.0146 - val_loss: 5.7405e-04 - val_mean_absol

Epoch 54/125
1147/1147 - 6s - loss: 1.7373e-04 - mean_absolute_error: 0.0076 - val_loss: 1.2046e-04 - val_mean_absolute_error: 0.0047 - 6s/epoch - 5ms/step
Epoch 55/125
1147/1147 - 7s - loss: 1.4724e-04 - mean_absolute_error: 0.0069 - val_loss: 2.8798e-04 - val_mean_absolute_error: 0.0102 - 7s/epoch - 6ms/step
Epoch 56/125
1147/1147 - 6s - loss: 1.4668e-04 - mean_absolute_error: 0.0069 - val_loss: 1.5234e-04 - val_mean_absolute_error: 0.0068 - 6s/epoch - 6ms/step
Epoch 57/125
1147/1147 - 7s - loss: 1.3250e-04 - mean_absolute_error: 0.0066 - val_loss: 1.5730e-04 - val_mean_absolute_error: 0.0072 - 7s/epoch - 6ms/step
Epoch 58/125
1147/1147 - 7s - loss: 1.5625e-04 - mean_absolute_error: 0.0073 - val_loss: 1.5464e-04 - val_mean_absolute_error: 0.0070 - 7s/epoch - 6ms/step
Epoch 59/125
1147/1147 - 7s - loss: 1.3697e-04 - mean_absolute_error: 0.0066 - val_loss: 1.4099e-04 - val_mean_absolute_error: 0.0064 - 7s/epoch - 6ms/step
Epoch 60/125
1147/1147 - 7s - loss: 1.4679e-04 - mean_absolute_e

Epoch 107/125
1147/1147 - 6s - loss: 1.2905e-04 - mean_absolute_error: 0.0064 - val_loss: 1.2706e-04 - val_mean_absolute_error: 0.0054 - 6s/epoch - 6ms/step
Epoch 108/125
1147/1147 - 6s - loss: 1.1828e-04 - mean_absolute_error: 0.0062 - val_loss: 1.1707e-04 - val_mean_absolute_error: 0.0046 - 6s/epoch - 6ms/step
Epoch 109/125
1147/1147 - 6s - loss: 1.3755e-04 - mean_absolute_error: 0.0066 - val_loss: 1.8439e-04 - val_mean_absolute_error: 0.0079 - 6s/epoch - 5ms/step
Epoch 110/125
1147/1147 - 6s - loss: 1.3454e-04 - mean_absolute_error: 0.0065 - val_loss: 1.2844e-04 - val_mean_absolute_error: 0.0058 - 6s/epoch - 5ms/step
Epoch 111/125
1147/1147 - 6s - loss: 1.3956e-04 - mean_absolute_error: 0.0067 - val_loss: 1.1846e-04 - val_mean_absolute_error: 0.0047 - 6s/epoch - 5ms/step
Epoch 112/125
1147/1147 - 7s - loss: 1.2571e-04 - mean_absolute_error: 0.0063 - val_loss: 1.2070e-04 - val_mean_absolute_error: 0.0050 - 7s/epoch - 6ms/step
Epoch 113/125
1147/1147 - 6s - loss: 1.2379e-04 - mean_abs

##### Results are low enough, but unfortunately, with minor oscillations throughout the epochs

In [45]:
model.save('Research_mcomb.h5') # Overwritten once more in a later experiment, since the model failed to perform this time as well.

In [46]:
result1 = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

656/656 - 2s - loss: 9.3061e-05 - mean_absolute_error: 0.0043 - 2s/epoch - 3ms/step


In [ ]:
for i in test_labels:
    print(i)

In [48]:
predictions1 = model.predict(x = test_samples, batch_size = 20, verbose = 2)

656/656 - 2s - 2s/epoch - 3ms/step


In [ ]:
for i in predictions1:
    print(i)

#### Same prediction for different samples that doesn't have any pattern of having the same actual label.

In [50]:
result2 = model.evaluate(train_samples, train_labels, batch_size = 20, verbose = 2)

1529/1529 - 4s - loss: 8.2469e-05 - mean_absolute_error: 0.0043 - 4s/epoch - 3ms/step


In [ ]:
for i in train_labels:
    print(i)

In [52]:
predictions2 = model.predict(x = train_samples, batch_size = 20, verbose = 2)

1529/1529 - 3s - 3s/epoch - 2ms/step


In [ ]:
for i in predictions2:
    print(i)

#### Same behavior as like that for test set.

### So, even after using more data, this architecture still struggles with this combination dataset. Hence, this new architecture is also not suitable for our purpose unfortunately.

### Since changing the architecture didn't work for us, we now look back at the data once more. One thing that was noticeable in the earlier experiments was, when printed out, there was a difference in how the actual labels and the predictions were being expressed. The actual labels were expressed with more precision, having more decimal places in comparison, and were printed out as single number entities. On the other hand, the predictions had lower precision and were printed out as separate lists with a single number. This difference made it seem like there could be a difference in data type of the actual labels and the predictions. So, we decide to change the default float type once to 'float64' and another time to 'float32', so that all the float numbers are processed as having the same data type, and rerun the previous experiment to check the results.

## Changing the default float type

#### We change the default float type of the system, once to 'float32' and then next to 'float64', using the following code. The given code shows the default float type being set to 'float64'. Each time after setting a particular default float type, we run the earlier experiment involving the regular sized dataset of the first combination of samples.

In [2]:
tf.keras.backend.set_floatx('float64')

In [29]:
tf.keras.backend.floatx()

'float64'

#### The results of this experiment are not showed here. However, for both the float types, the outcome was the same. Changing the default float type did not fix our issue. The model still continued to predict same value for different samples that had different actual curvatures (labels).

### So, changing the default float type doesn't fix our problem, which also means that the difference between the actual labels and the predictions is not caused by how the float numbers are being processed in the experiment. In order to make sure that the difference in how the actual labels and the predictions are being expressed is not the cause of our issue at hand, we run another experiment where we set a particular data type for our input dataset when generating it, once to 'float64' and next to 'float32', and then check the performance of our model each time.

## Setting the data type of the input dataset
### Combination of samples used - Sine Wave Surface, Parabolic Cylinder (Quadratic Univariate Polynomial), Circular Paraboloid (z = x^2 + y^2)<br>Architecture - Depth(20 layers in total), Width (40 nodes highest among the layers)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked<br>Using a smaller step size for Parabolic Cylinder samples; Trained for a moderately higher number of epochs

#### The results below are for the experiment where we set the data type of the input dataset to 'float32'.

In [6]:
funct1 = exp_generator(3, 1)
funct1

0.593988879915532*sin(2.01369540968675*x - 4.57047002658068) - 0.607193781823679

In [7]:
s1, l_max1, l_min1 = datalist_generator(funct1, 5, 5, 115, 115, 1)

In [8]:
funct2 = exp_generator(5, 1)
funct2

x**2 + y**2

In [9]:
s2, l_max2, l_min2 = datalist_generator(funct2, 1, 1, 111, 111, 1)

s = np.concatenate((s1, s2))
l_max = np.concatenate((l_max1, l_max2))

s, l_max = shuffle(s, l_max)

train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s, l_max, test_size = 0.3, random_state = 6)

In [10]:
funct3 = exp_generator(1, 2)
funct3

0.204219866843413*x**2 - 0.210374622235534*x + 1.59664462678772

In [11]:
s3, l_max3, l_min3 = datalist_generator(funct3, -12, -12, 43, 43, 0.5)

train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s3, l_max3, test_size = 0.3, random_state = 21)

train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [12]:
m_s = np.mean(train_samples)
m_s

3000.628

In [13]:
sd_s = np.std(train_samples)
sd_s

5072.793

In [14]:
m_l = np.mean(train_labels)
m_l

0.3736061

In [15]:
sd_l = np.std(train_labels)
sd_l

0.65577453

In [16]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 32, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 40, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 32, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [17]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [18]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 100, shuffle = True, verbose = 2)

Epoch 1/100
901/901 - 13s - loss: 1.3831 - mean_absolute_error: 0.4753 - val_loss: 0.1292 - val_mean_absolute_error: 0.2120 - 13s/epoch - 15ms/step
Epoch 2/100
901/901 - 5s - loss: 0.0326 - mean_absolute_error: 0.0983 - val_loss: 0.0126 - val_mean_absolute_error: 0.0668 - 5s/epoch - 6ms/step
Epoch 3/100
901/901 - 5s - loss: 0.0079 - mean_absolute_error: 0.0530 - val_loss: 0.0062 - val_mean_absolute_error: 0.0468 - 5s/epoch - 6ms/step
Epoch 4/100
901/901 - 5s - loss: 0.0040 - mean_absolute_error: 0.0368 - val_loss: 0.0033 - val_mean_absolute_error: 0.0333 - 5s/epoch - 6ms/step
Epoch 5/100
901/901 - 5s - loss: 0.0028 - mean_absolute_error: 0.0302 - val_loss: 0.0022 - val_mean_absolute_error: 0.0271 - 5s/epoch - 6ms/step
Epoch 6/100
901/901 - 6s - loss: 0.0019 - mean_absolute_error: 0.0247 - val_loss: 0.0022 - val_mean_absolute_error: 0.0278 - 6s/epoch - 7ms/step
Epoch 7/100
901/901 - 4s - loss: 0.0013 - mean_absolute_error: 0.0202 - val_loss: 0.0011 - val_mean_absolute_error: 0.0184 - 4s

Epoch 55/100
901/901 - 5s - loss: 1.9308e-04 - mean_absolute_error: 0.0077 - val_loss: 1.6342e-04 - val_mean_absolute_error: 0.0058 - 5s/epoch - 6ms/step
Epoch 56/100
901/901 - 6s - loss: 1.9299e-04 - mean_absolute_error: 0.0078 - val_loss: 4.8667e-04 - val_mean_absolute_error: 0.0146 - 6s/epoch - 6ms/step
Epoch 57/100
901/901 - 5s - loss: 1.9035e-04 - mean_absolute_error: 0.0074 - val_loss: 1.6163e-04 - val_mean_absolute_error: 0.0056 - 5s/epoch - 6ms/step
Epoch 58/100
901/901 - 5s - loss: 2.1725e-04 - mean_absolute_error: 0.0083 - val_loss: 2.8306e-04 - val_mean_absolute_error: 0.0095 - 5s/epoch - 6ms/step
Epoch 59/100
901/901 - 5s - loss: 2.0314e-04 - mean_absolute_error: 0.0079 - val_loss: 2.1765e-04 - val_mean_absolute_error: 0.0083 - 5s/epoch - 6ms/step
Epoch 60/100
901/901 - 5s - loss: 1.6523e-04 - mean_absolute_error: 0.0069 - val_loss: 1.5680e-04 - val_mean_absolute_error: 0.0052 - 5s/epoch - 6ms/step
Epoch 61/100
901/901 - 5s - loss: 1.9359e-04 - mean_absolute_error: 0.0078 -

#### There is constant oscillation in the results across all epochs; one noticable aspect was, when the training loss got down to a decently low value, the validation loss would rise up. However, in general, the results are good enough.

In [19]:
model.save('Research_mcomb.h5') # Since this experiment didn't fix our problem, we again overwrote this in a later experiment.

In [20]:
result1 = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

515/515 - 1s - loss: 1.2812e-04 - mean_absolute_error: 0.0062 - 1s/epoch - 2ms/step


In [21]:
for i in test_labels:
    print(i)

0.55408555
0.098799355
0.0010024335
0.01461402
0.009300716
0.034993693
0.0048024254
0.23223004
0.011123305
0.008518354
0.6905713
0.010795051
0.0090848105
0.039992
0.0048024254
0.3261462
0.009820809
0.13862154
0.53666645
0.009244905
0.1914416
0.2970339
0.56169754
0.7773554
0.0073532206
0.007308573
0.01199585
0.10769543
0.000682654
0.012777271
1.2680225
0.008907912
0.007942952
0.00035719777
0.15570237
0.017965602
0.21282145
0.010183735
0.0004853606
0.10769543
0.0063770562
0.02794812
0.021198766
0.04641133
0.0042146943
0.009407105
0.009434697
0.009819862
0.013366344
0.115401484
0.015311853
0.010757509
0.0073532206
0.012682382
0.5293149
0.2511465
0.008519282
0.1914416
0.019591844
0.8212621
2.1319735
1.1681184
0.15570237
1.7130169
0.25531197
2.1449127
0.008466903
0.06929589
0.13862154
0.0120471
0.00045533836
0.93234134
0.0041722255
0.0076858867
0.008405958
0.016338103
0.013667568
0.011475128
1.345485
0.00031856945
0.019173244
0.00085461745
2.3313358
0.04424114
0.0017124482
0.00033714873
2.0

In [22]:
predictions1 = model.predict(x = test_samples, batch_size = 20, verbose = 2)

515/515 - 2s - 2s/epoch - 3ms/step


In [23]:
for i in predictions1:
    print(i)

[0.5581726]
[0.09354352]
[0.01105673]
[0.01433434]
[0.01105673]
[0.03399868]
[0.01105673]
[0.2417996]
[0.01105673]
[0.01105673]
[0.7011716]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.32814878]
[0.01105673]
[0.13763548]
[0.5365056]
[0.01105673]
[0.1922684]
[0.30508363]
[0.5708352]
[0.7848198]
[0.01105673]
[0.01105673]
[0.01105673]
[0.10540102]
[0.01105673]
[0.01105673]
[1.2788782]
[0.01105673]
[0.01105673]
[0.01105673]
[0.16454649]
[0.01105673]
[0.22068755]
[0.01105673]
[0.01105673]
[0.10540091]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.04013674]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.11519954]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.53772396]
[0.24979042]
[0.01105673]
[0.1922684]
[0.01105673]
[0.8278772]
[2.1286817]
[1.180736]
[0.16454649]
[1.7220232]
[0.25567293]
[2.1521456]
[0.01105673]
[0.01105673]
[0.13763548]
[0.01105673]
[0.01105673]
[0.9244979]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[1

[0.01105673]
[0.05879775]
[0.10293568]
[0.01105673]
[0.01105673]
[0.20873405]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.10129335]
[0.01433434]
[0.05546527]
[0.1922684]
[0.01105673]
[0.01105673]
[0.3785671]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01481558]
[0.01105673]
[0.4747965]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.391295]
[0.22068755]
[0.01433434]
[0.01105673]
[0.01105673]
[0.01105673]
[0.1867296]
[0.01105673]
[0.01105673]
[0.05879775]
[0.01105673]
[0.6447521]
[0.01105673]
[0.01105673]
[0.32814878]
[0.01105673]
[0.10293568]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.17160414]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.5732036]
[0.0265084]
[0.3785671]
[0.3778879]
[0.01105673]
[2.397476]
[0.7011716]
[0.01105673]
[0.14083935]
[0.01105673]
[0.3895744]
[0.01105673]
[0.01105673]
[0.01105673]
[0.52290446]
[0.01105673]
[0.01105673]
[0.01105673]
[2.2075708]
[0.01105673]
[0.01105673]
[0

[0.01105673]
[0.01105673]
[0.01105673]
[0.0604845]
[2.2075706]
[0.01105673]
[0.01105673]
[0.01105673]
[0.08544508]
[0.03259791]
[0.02262565]
[0.10129335]
[0.81529707]
[0.01105673]
[0.01105673]
[0.01105673]
[0.07380449]
[0.01105673]
[0.01105673]
[0.445329]
[0.01105673]
[0.5365057]
[0.01105673]
[1.7532895]
[0.01105673]
[2.392255]
[0.01105673]
[0.01105673]
[0.05546527]
[0.01105673]
[1.09544]
[0.01105673]
[0.80853766]
[1.3479189]
[1.7287601]
[1.373256]
[0.64475197]
[0.22556952]
[0.04106064]
[0.01105673]
[0.4747965]
[0.01105673]
[0.01105673]
[0.35830784]
[2.4063883]
[0.7659026]
[1.5852485]
[1.3431683]
[0.01105673]
[0.63876253]
[0.4747965]
[0.01105673]
[0.01105673]
[0.01105673]
[2.4062183]
[0.01105673]
[0.01105673]
[2.1549463]
[0.01105673]
[0.5365057]
[0.01105673]
[0.01105673]
[2.390151]
[1.3431683]
[1.3479189]
[0.01105673]
[0.01105673]
[0.10293568]
[0.43095857]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.06861712]
[0.01105673]
[0.30508298]
[0.01105673]
[0.32278207]
[0.01105673]
[

[0.24979042]
[0.05879775]
[0.01105673]
[0.01105673]
[0.07632983]
[0.01105673]
[0.29298866]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.10293568]
[2.2020562]
[0.4747965]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[1.5541248]
[0.01105673]
[0.01105673]
[1.1541713]
[0.01105673]
[0.01433434]
[0.01105673]
[0.4747965]
[0.01105673]
[0.4747965]
[0.68602943]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.25567293]
[0.5581726]
[0.6618853]
[0.01105673]
[1.8236216]
[2.4063883]
[0.01105673]
[2.4062183]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01481558]
[1.8236216]
[0.01105673]
[1.1541713]
[0.01105673]
[0.01105673]
[0.9878548]
[0.68602943]
[0.01105673]
[0.01105673]
[0.04186318]
[0.01105673]
[0.01105673]
[0.15754409]
[0.01105673]
[0.01105673]
[0.01105673]
[0.9244979]
[0.01105673]
[0.01105673]
[0.18145448]
[0.27129737]
[0.01105673]
[0.01105673]
[2.3001547]
[0.01105673]
[0.01433434]
[0.68602943]
[0.5732036]
[0.01105673]
[0.0443407]
[0.2

[0.01105673]
[0.01105673]
[0.0265084]
[0.01105673]
[0.32278207]
[0.01105673]
[2.1286817]
[0.6447521]
[0.6618853]
[0.15754409]
[0.01105673]
[0.32278207]
[1.5113453]
[0.01105673]
[0.01105673]
[0.12311494]
[2.4063876]
[0.0604845]
[0.01105673]
[0.01105673]
[2.3326936]
[0.01105673]
[0.01105673]
[0.01105673]
[0.04013674]
[0.01105673]
[0.01105673]
[0.01105673]
[1.2822022]
[0.01105673]
[0.7011716]
[0.30404213]
[0.28316364]
[0.01105673]
[0.3895744]
[0.04578417]
[0.01105673]
[0.04106064]
[0.01105673]
[0.026483]
[0.01105673]
[0.01105673]
[0.07380448]
[0.01105673]
[0.68602943]
[0.01105673]
[0.0265084]
[0.9878548]
[0.01105673]
[0.01105673]
[0.01105673]
[0.18145448]
[0.07380448]
[0.01105673]
[2.3043294]
[0.01105673]
[0.01105673]
[1.3732573]
[0.04654469]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[1.0986502]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[2.4042087]
[0.01481558]
[0.20873405]
[0.9244979]
[0.22556914]
[0.01105673]
[0.01105673]
[0.558

[0.9878548]
[2.3001547]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01481558]
[0.01105673]
[0.06017376]
[0.18145448]
[0.01105673]
[1.5137081]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.29298866]
[0.01105673]
[0.01105673]
[0.01481558]
[2.1549463]
[0.01105673]
[0.68602943]
[0.20873405]
[0.01105673]
[2.2075706]
[0.01105673]
[0.01105673]
[0.01105673]
[0.0443407]
[0.01105673]
[0.01105673]
[0.28316364]
[0.01105673]
[0.01105673]
[0.10433722]
[0.30508363]
[0.17369325]
[0.01105673]
[0.01105673]
[0.01105673]
[0.6618853]
[0.01105673]
[0.01105673]
[0.35269776]
[0.01105673]
[0.01105673]
[0.01105673]
[2.3001547]
[0.01105673]
[0.01105673]
[0.04013691]
[0.01105673]
[0.01105673]
[0.63876253]
[2.4062183]
[0.0260491]
[0.01105673]
[0.04654469]
[0.1571261]
[1.2822022]
[0.01105673]
[0.05546527]
[0.01105673]
[0.02209866]
[0.20873405]
[0.68602943]
[0.01105673]
[0.01105673]
[0.17160414]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[0.01105673]
[1.5809087]
[0.01105673]

#### Same prediction for different samples that doesn't have similar actual curvatures (labels).

In [24]:
result2 = model.evaluate(train_samples, train_labels, batch_size = 20, verbose = 2)

1201/1201 - 3s - loss: 1.4624e-04 - mean_absolute_error: 0.0062 - 3s/epoch - 2ms/step


In [ ]:
for i in train_labels:
    print(i)

In [26]:
predictions2 = model.predict(x = train_samples, batch_size = 20, verbose = 2)

1201/1201 - 3s - 3s/epoch - 2ms/step


In [ ]:
for i in predictions2:
    print(i)

#### Same behavior for the training set as like that for test set.

### Setting the data type of the input set to 'float32' didn't fix our problem. The outcome was the same when we set it up to 'float64' as well. Hence, we can conclude that the difference in how the actual labels and the predictions were being expressed did not cause our model to fail. It was just a difference in how the system was printing them out.

### In all the experiments done so far, one thing that was particularly noticable was that, our models struggled with the samples having a lower curvature value around the range of 10^-4 or less compared to those with higher curvatures. So next, we plan to use scaling as our tool and multiply our dataset with a slightly large scalar and check if that helps in any improvement of our model's performance.

## Checking performance by scaling the dataset
### Combination of samples used - Sine Wave Surface, Parabolic Cylinder (Quadratic Univariate Polynomial), Circular Paraboloid (z = x^2 + y^2)<br>Architecture - Depth(20 layers in total), Width (40 nodes highest among the layers)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked<br>Using a smaller step size for Parabolic Cylinder samples; Trained for a moderately higher number of epochs

In [6]:
tf.keras.backend.set_floatx('float64') # The default float type was set to 'float64' for this experiment

In [7]:
# Function for multiplying input values and curvatures by a constant

def mult_scale(inp, curv):
    l_sample = []
    l_samples = []
    l_labels = []
    
    con = 100       # Constant used for scaling
                    # Used a slightly large constant because the data was already enough larger in value,
                    # so didn't want to get into the NaN problem
    
    for l in inp:
        for item in l:
            item *= con
            l_sample.append(item)
            
        l_samples.append(l_sample)
        l_sample = []
    
    for item in curv:
        item *= con
        l_labels.append(item)
    
    # The input dataset was also set to have the 'float64' data type
    sc_samples = np.array(l_samples, dtype = 'float64')
    sc_labels = np.array(l_labels, dtype = 'float64')
    
    return sc_samples, sc_labels

In [8]:
funct1 = exp_generator(3, 1)
funct1

0.593988879915532*sin(2.01369540968675*x - 4.57047002658068) - 0.607193781823679

In [9]:
s1, l_max1, l_min1 = datalist_generator(funct1, 5, 5, 115, 115, 1)

In [10]:
funct2 = exp_generator(5, 1)
funct2

x**2 + y**2

In [11]:
s2, l_max2, l_min2 = datalist_generator(funct2, 1, 1, 111, 111, 1)

s = np.concatenate((s1, s2))
l_max = np.concatenate((l_max1, l_max2))

s, l_max = shuffle(s, l_max)

train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s, l_max, test_size = 0.3, random_state = 6)

In [12]:
funct3 = exp_generator(1, 2)
funct3

0.204219866843413*x**2 - 0.210374622235534*x + 1.59664462678772

In [13]:
s3, l_max3, l_min3 = datalist_generator(funct3, -12, -12, 43, 43, 0.5)

train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s3, l_max3, test_size = 0.3, random_state = 21)

train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [14]:
train_samples, train_labels = mult_scale(train_samples, train_labels)
test_samples, test_labels = mult_scale(test_samples, test_labels)

In [15]:
m_s = np.mean(train_samples)
m_s

298690.59058297565

In [16]:
sd_s = np.std(train_samples)
sd_s

504351.7822533959

In [17]:
m_l = np.mean(train_labels)
m_l

37.393587546014835

In [18]:
sd_l = np.std(train_labels)
sd_l

65.75003705467151

In [25]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 32, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 40, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 32, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [26]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [27]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 100, shuffle = True, verbose = 2)

Epoch 1/100
901/901 - 7s - loss: 521701.9005 - mean_absolute_error: 189.8047 - val_loss: 2352.5091 - val_mean_absolute_error: 28.9139 - 7s/epoch - 7ms/step
Epoch 2/100
901/901 - 4s - loss: 2095.7060 - mean_absolute_error: 25.7786 - val_loss: 1843.9025 - val_mean_absolute_error: 23.1904 - 4s/epoch - 4ms/step
Epoch 3/100
901/901 - 4s - loss: 1666.8827 - mean_absolute_error: 21.1117 - val_loss: 1497.2349 - val_mean_absolute_error: 19.1246 - 4s/epoch - 5ms/step
Epoch 4/100
901/901 - 4s - loss: 1408.8462 - mean_absolute_error: 18.1656 - val_loss: 1301.8954 - val_mean_absolute_error: 17.4210 - 4s/epoch - 5ms/step
Epoch 5/100
901/901 - 4s - loss: 1224.0168 - mean_absolute_error: 16.6955 - val_loss: 1115.1891 - val_mean_absolute_error: 16.0157 - 4s/epoch - 5ms/step
Epoch 6/100
901/901 - 4s - loss: 1036.3073 - mean_absolute_error: 15.2451 - val_loss: 931.0292 - val_mean_absolute_error: 14.4457 - 4s/epoch - 5ms/step
Epoch 7/100
901/901 - 4s - loss: 836.4689 - mean_absolute_error: 13.7181 - val_l

Epoch 57/100
901/901 - 4s - loss: 3.5823 - mean_absolute_error: 0.9242 - val_loss: 1.5143 - val_mean_absolute_error: 0.7780 - 4s/epoch - 5ms/step
Epoch 58/100
901/901 - 4s - loss: 2.1017 - mean_absolute_error: 0.7891 - val_loss: 1.1963 - val_mean_absolute_error: 0.5801 - 4s/epoch - 5ms/step
Epoch 59/100
901/901 - 4s - loss: 2.1043 - mean_absolute_error: 0.7858 - val_loss: 1.4466 - val_mean_absolute_error: 0.7016 - 4s/epoch - 4ms/step
Epoch 60/100
901/901 - 4s - loss: 1.6917 - mean_absolute_error: 0.7633 - val_loss: 1.5196 - val_mean_absolute_error: 0.6993 - 4s/epoch - 4ms/step
Epoch 61/100
901/901 - 4s - loss: 2.6163 - mean_absolute_error: 0.9252 - val_loss: 2.0080 - val_mean_absolute_error: 0.8660 - 4s/epoch - 5ms/step
Epoch 62/100
901/901 - 4s - loss: 2.2148 - mean_absolute_error: 0.8314 - val_loss: 1.2490 - val_mean_absolute_error: 0.6317 - 4s/epoch - 5ms/step
Epoch 63/100
901/901 - 3s - loss: 2.3382 - mean_absolute_error: 0.8599 - val_loss: 1.1853 - val_mean_absolute_error: 0.6008 

##### Received these results on the 3rd attempt. Loss and error are quite high, and they were in continuous oscillation across the epochs, though the general trend was downward.

In [ ]:
model.save('Research_mcomb.h5') # Overwritten again later, because this experiment also leads to failure

In [28]:
result1 = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

515/515 - 1s - loss: 0.6566 - mean_absolute_error: 0.4811 - 801ms/epoch - 2ms/step


In [29]:
for i in test_labels:
    print(i)

27.549441125015
133.0750190382691
5.70367711658479
25.11464934662333
1.6210947626279844
0.035719778078113315
1.171274200490727
2.27965566122176
43.07769307365432
4.482179210873585
13.862153170143927
0.05928392644099678
1.0261268811991335
3.9992002399200284
1.3253173168819836
29.198858221514435
0.23284651480562193
0.023145539445648913
233.1335715057423
0.022014359038362314
1.049771142032064
0.1555718214597806
92.09225254743552
4.424113970639921
0.17124481616056142
35.09385613655042
179.75340450712787
28.04023085637606
1.2066378732787795
34.542051614909944
4.286479796890474
0.808102245625016
3.4993694735547263
1.2140319604318506
25.11464934662333
1.6103393644569493
90.96141284141153
149.98893957611764
1.32999774399324
2.567269296894028
1.227455940013173
0.8703596533569953
7.625825478517643
0.802230090404781
171.30168624890666
2.324795699392311
43.71167722730851
6.072289559079183
90.96141284141153
0.25981677244649126
0.9296695971951578
45.86817809023663
172.89167798067885
1.21403196043185

In [30]:
predictions1 = model.predict(x = test_samples, batch_size = 20, verbose = 2)

515/515 - 1s - 1s/epoch - 2ms/step


In [31]:
for i in predictions1:
    print(i)

[27.05649943]
[133.04686146]
[4.89538664]
[25.85855706]
[1.9887813]
[0.73205264]
[1.33054422]
[2.35325637]
[42.29609073]
[3.21958845]
[14.82034747]
[0.70080191]
[0.90073605]
[2.60389448]
[1.6307137]
[28.55353951]
[0.70080191]
[0.9777773]
[235.07533757]
[1.01053025]
[0.98960513]
[0.70080191]
[92.11161582]
[5.08379964]
[0.70080191]
[34.12795588]
[179.16839364]
[27.32460758]
[1.40043783]
[34.22398292]
[2.62426982]
[0.70080191]
[4.13070353]
[1.64556432]
[25.85855706]
[1.98549616]
[90.34716342]
[150.64705143]
[1.64163803]
[2.43642019]
[1.44758826]
[0.70080191]
[8.66141397]
[0.70080191]
[171.68398666]
[2.3676718]
[43.69482686]
[4.9842235]
[90.34716342]
[0.70080191]
[0.70080191]
[45.21237022]
[172.84550565]
[1.64556432]
[0.70080191]
[56.29104515]
[2.95693731]
[3.53000459]
[77.44862438]
[0.70080191]
[0.70080191]
[1.67649886]
[0.70080191]
[1.62388175]
[157.55456587]
[2.51563482]
[1.32540898]
[1.74442815]
[2.43211611]
[1.39588767]
[14.21031831]
[231.80636535]
[1.98674577]
[0.70080191]
[133.04686

[108.53671014]
[1.61813137]
[0.70080191]
[32.4585688]
[1.21046749]
[1.07261635]
[0.70080191]
[92.59362936]
[26.46891025]
[231.80636535]
[0.70080191]
[19.67282611]
[0.70080191]
[240.48121374]
[1.03284619]
[1.14738345]
[1.47457621]
[240.02335586]
[179.16839364]
[1.07088583]
[181.46542435]
[1.32540898]
[14.21031831]
[0.70080191]
[0.70080191]
[14.21031831]
[0.70080191]
[2.05961963]
[1.85181988]
[235.07533757]
[1.52833152]
[133.04686146]
[0.70080191]
[0.99236593]
[0.7471292]
[26.46891025]
[38.10861933]
[2.08322446]
[22.80371157]
[0.70080191]
[201.07668953]
[53.03257967]
[0.70080191]
[215.80334728]
[8.66141397]
[32.4585688]
[14.21031831]
[0.70080191]
[79.8327003]
[22.80371157]
[157.57308416]
[29.51523024]
[20.09938109]
[133.48239302]
[17.20116803]
[92.11161582]
[0.70080191]
[65.32159037]
[92.11161582]
[0.70080191]
[0.70080191]
[0.70080191]
[0.70080191]
[0.70080191]
[1.47354626]
[1.77069452]
[0.70080191]
[63.64404295]
[151.22684756]
[0.70080191]
[0.87147473]
[22.80371157]
[9.62771045]
[2.2647

[0.75841354]
[2.05154244]
[0.70080191]
[0.70080191]
[1.45833562]
[0.70080191]
[76.15499325]
[127.81605964]
[174.7859914]
[81.96251728]
[116.97901668]
[0.88280646]
[29.51523024]
[1.88300129]
[34.12795588]
[92.11161582]
[43.69482686]
[2.16160993]
[0.70080191]
[2.34578497]
[5.43995029]
[172.84550565]
[65.32159037]
[0.70080191]
[2.15015788]
[0.70080191]
[0.70080191]
[0.70080191]
[179.16839364]
[13.80807002]
[2.44885652]
[2.16576697]
[0.70080191]
[0.70080191]
[157.57308416]
[199.54496039]
[111.15891351]
[2.41985826]
[9.48960206]
[1.72722015]
[0.70080191]
[27.32460758]
[0.70080191]
[1.64556432]
[0.70080191]
[1.20519296]
[174.7859914]
[0.70080191]
[4.89538664]
[13.80807002]
[67.27802583]
[239.29037471]
[0.70080191]
[0.70080191]
[1.47105787]
[25.85855706]
[34.22398292]
[199.54496039]
[1.01053025]
[192.77360437]
[0.70080191]
[0.70080191]
[0.70080191]
[0.70080191]
[212.96775822]
[1.98674577]
[1.24534758]
[75.55892219]
[1.64556432]
[0.70080191]
[129.7984966]
[0.70080191]
[54.68477253]
[19.3733051

#### The predictions are close enough, but there is still a noticable difference with the actual labels. As the curvature values are higher now, the loss and error are also high. Unfortunately, the problem of predicting same curvature for different samples persists in this experiment as well, but the frequency is lower in comparison.

In [32]:
result2 = model.evaluate(train_samples, train_labels, batch_size = 20, verbose = 2)

1201/1201 - 2s - loss: 0.6830 - mean_absolute_error: 0.4810 - 2s/epoch - 1ms/step


In [ ]:
for i in train_labels:
    print(i)

In [34]:
predictions2 = model.predict(x = train_samples, batch_size = 20, verbose = 2)

1201/1201 - 1s - 1s/epoch - 1ms/step


In [ ]:
for i in predictions2:
    print(i)

#### Same type of behavior like that for the test set.

### So, scaling the dataset also doesn't help us in solving our problem. Hence, we can conclude that the smaller curvatures in the range of 10^-4 aren't the issue.
### If we carefully analyse all our earlier experiments, we would see that our earlier model had the best performance with this particular combination dataset compared to the other 2 sets, and in those experiments we hadn't started using the random_state parameter when splitting the dataset. Another thing to note is that, the earlier model struggled with the other 2 combination datasets, but this latest architecture performed well on them. So, it makes little sense that this architecture would struggle with the first combination dataset, unless there is a change with the earlier experiment, which is the inclusion of the random_state parameter. Hence, using this architecture, we test the performance in the next experiment by changing the argument to the random_state parameter in the train_test_split function from what was being used so far.

## Checking performance by changing random_state parameter of train_test_split function
### Combination of samples used - Sine Wave Surface, Parabolic Cylinder (Quadratic Univariate Polynomial), Circular Paraboloid (z = x^2 + y^2)<br>Architecture - Depth(20 layers in total), Width (40 nodes highest among the layers)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked<br>Using a smaller step size for Parabolic Cylinder samples; Trained for a moderately higher number of epochs

#### This experiment was conducted twice. in the first experiment, we received the results (loss and error) on the 2nd attempt, where the results had oscillations throughout the epochs, but they got down to decent enough low values. However, the predictions were found to having the same issue of being similar for different samples. So we tried out a 2nd experiment, where we checked the predictions when receiving the results on the first attempt, which were comparatively higher in value. We did this to figure out if trying to get low loss and error caused our model to break into predicting similar curvatures for different samples. But in this case, the predictions were even worse.<br>The following shows the outputs for the 2nd experiment.

In [3]:
tf.keras.backend.set_floatx('float64') # Set the default float type to 'float64'

In [8]:
funct1 = exp_generator(3, 1)
funct1

0.593988879915532*sin(2.01369540968675*x - 4.57047002658068) - 0.607193781823679

In [9]:
s1, l_max1, l_min1 = datalist_generator(funct1, 5, 5, 115, 115, 1)

In [10]:
funct2 = exp_generator(5, 1)
funct2

x**2 + y**2

In [11]:
s2, l_max2, l_min2 = datalist_generator(funct2, 1, 1, 111, 111, 1)

s = np.concatenate((s1, s2))
l_max = np.concatenate((l_max1, l_max2))

s, l_max = shuffle(s, l_max)

train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s, l_max, test_size = 0.3, random_state = 4)

# The 'random_state' argument was changed from what we had in previous experiements

In [12]:
funct3 = exp_generator(1, 2)
funct3

0.204219866843413*x**2 - 0.210374622235534*x + 1.59664462678772

In [13]:
s3, l_max3, l_min3 = datalist_generator(funct3, -12, -12, 43, 43, 0.5)

train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s3, l_max3, test_size = 0.3, random_state = 11)

# Similarly, the 'random_state' argument was changed here too

train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [14]:
m_s = np.mean(train_samples)
m_s

3026.956505733026

In [15]:
sd_s = np.std(train_samples)
sd_s

5078.046510230222

In [16]:
m_l = np.mean(train_labels)
m_l

0.36632167764644163

In [17]:
sd_l = np.std(train_labels)
sd_l

0.6505026500557333

In [18]:
model = Sequential([
    Dense(units = 4, input_shape = (9,), activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 32, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 40, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 32, activation = 'relu'),
    Dense(units = 28, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 20, activation = 'relu'),
    Dense(units = 16, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 8, activation = 'relu'),
    Dense(units = 4, activation = 'relu'),
    Dense(units = 1)
])

In [19]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [ ]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 100, shuffle = True, verbose = 2)

##### Received the results at the first try. They got stuck to particular values respectively, which were quite high in value.

In [23]:
model.save('Research_mcomb.h5') # Deleted, as this experiment yielded a very poor performance for our model

In [21]:
result1 = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

515/515 - 1s - loss: 0.4454 - mean_absolute_error: 0.4910 - 1s/epoch - 3ms/step


In [22]:
for i in test_labels:
    print(i)

0.0006826539657393814
0.00965372848011158
0.016384088617973885
0.027660822219612094
0.0011861929153761171
2.4032664081451545
2.331335715057423
0.04116645244171622
0.010183735122315959
0.009593567558827468
0.0002703983746431383
0.008326036793383243
0.009578153011637069
0.010326716458254892
0.18800903613731773
0.008131089083011767
0.36967720494279216
0.05763801828837256
0.010917882893685645
0.00965372848011158
0.014783175110933706
0.0003185694564544265
2.3038528271841914
0.0097962741568593
0.00903498683577966
1.7130168624890667
0.013453151513425528
0.008210454582657532
0.012273301782875102
0.0442411397063992
0.009770989177930747
0.013802297100423858
0.01204011223122523
0.09121879825813574
2.3863473043443904
0.015338848640974578
0.06321395412410141
0.0008546174351114534
0.20842051281282897
1.939929676815565
0.010064493591294528
0.010489045650789983
0.0017124481616056142
0.021217845347030695
0.017299976536922734
0.06441750644817072
0.38275122656342403
0.00998291886466452
0.1693525197521385

In [23]:
predictions1 = model.predict(x = test_samples, batch_size = 20, verbose = 2)

515/515 - 2s - 2s/epoch - 3ms/step


In [24]:
for i in predictions1:
    print(i)

[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]

[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]

[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]

[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]

[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]

[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]
[0.36779113]

#### The model gives the same prediction for all the samples.

In [25]:
result2 = model.evaluate(train_samples, train_labels, batch_size = 20, verbose = 2)

1201/1201 - 3s - loss: 0.4232 - mean_absolute_error: 0.4804 - 3s/epoch - 3ms/step


In [ ]:
for i in train_labels:
    print(i)

In [27]:
predictions2 = model.predict(x = train_samples, batch_size = 20, verbose = 2)

1201/1201 - 3s - 3s/epoch - 2ms/step


In [ ]:
for i in predictions2:
    print(i)

#### Same behavior for the train set too.

### So, we see that changing the argument to the random_state parameter of the train_test_split function didn't help us in improving the performance of our model. Since even after trying out so many things to improve the performance didn't work out at all, it just proves that this architecture is just not suited for our purpose.
### Hence, next we again try out the CNN architecture by employing Transfer Learning with VGG16 model. Last time we had tried this out, we used a dataset that had very low curvature values which were not suited for our research according to our hypothesis. Now that we are using a combination dataset with higher curvature values, we again try out the CNN architecture since VGG16 has been proven to give better performance. We add 4 dense layers with a maximum width of 128 nodes at the top of the VGG16 base and check out the performance on the first combination dataset.

## Architecture - Transfer Learning applied with VGG16 Model - Layers added to the top (4 Dense Hidden Layers, Max Width - 128)<br>Combination of samples used - Sine Wave Surface, Parabolic Cylinder (Quadratic Univariate Polynomial), Circular Paraboloid (z = x^2 + y^2)
### "Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked<br>Using a smaller step size for Parabolic Cylinder samples

In [1]:
import sympy
import numpy as np
import tensorflow as tf
from tensorflow import keras
from random import uniform, seed
from sklearn.utils import shuffle
from tensorflow.keras.models import *
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.metrics import *
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

In [4]:
tf.keras.backend.set_floatx('float64') # Default float type set to 'float64'

In [3]:
x = sympy.Symbol('x')
y = sympy.Symbol('y')

In [4]:
# Function for generating input sample and unsigned labels for VGG16/Resnet50V2 transfer-learned model for a particular point 
# of interest

def data_generator(exp, center_x, center_y, ss):
    l_sample = []
    
    i = center_x - (16*ss)
    j = center_y - (16*ss)
    
    while (i < center_x+(16*ss)+ss):
        while(j < center_y+(16*ss)+ss):
            value = exp.evalf(subs={x: i, y: j})
            l_sample.extend([value, value, value])
            
            j = j + ss
        
        j = center_y - (16*ss)
        i = i + ss
    
    sample = np.array(l_sample, dtype = 'float64')
    sample = sample.reshape((33,33,3), order = 'C')
    l_sample_n = sample.tolist()
    
    fx = exp.diff(x, 1)
    fy = exp.diff(y, 1)
    fxx = exp.diff(x, 2)
    fyy = exp.diff(y, 2)
    fxy = exp.diff(x, y, 1)
    
    v_fx = fx.evalf(subs={x: center_x, y: center_y})
    v_fy = fy.evalf(subs={x: center_x, y: center_y})
    v_fxx = fxx.evalf(subs={x: center_x, y: center_y})
    v_fyy = fyy.evalf(subs={x: center_x, y: center_y})
    v_fxy = fxy.evalf(subs={x: center_x, y: center_y})
    
    K = (v_fxx*v_fyy - v_fxy**2) / (1 + v_fx**2 + v_fy**2)**2
    H = (v_fxx + v_fyy + v_fxx*v_fy**2 + v_fyy*v_fx**2 - 2*v_fx*v_fy*v_fxy) / (2 * (1 + v_fx**2 + v_fy**2)**1.5)
    
    k1 = H + (H**2 - K)**0.5
    k2 = H - (H**2 - K)**0.5
    
    #*********************************************
    # Changes made to create unsigned labels
    
    u_k1 = abs(k1)
    u_k2 = abs(k2)
    
    if(u_k1 < u_k2):
        temp = u_k1
        u_k1 = u_k2
        u_k2 = temp
    
    #*********************************************
    
    #*********************************************
    # Changes made to split labels into two groups
    
    l_label_max = u_k1
    l_label_min = u_k2
    
    return l_sample_n, l_label_max, l_label_min

    #*********************************************

In [5]:
# Function for generating a list of samples and labels for a range of points

def datalist_generator(exp, x_start, y_start, x_end, y_end, ss):
    l_samples = []
    l_labels_max = []
    l_labels_min = []
    
    i = x_start
    j = y_start
    
    while (i < x_end+ss):
        while(j < y_end+ss):
            
            t_sample, t_label_max, t_label_min = data_generator(exp, i, j, ss)
            
            if(t_label_max >= 0.0002):
                # Restriction applied, since lower curvatures caused our model to fail as evident from earlier experiments
                
                l_samples.append(t_sample)
                l_labels_max.extend([t_label_max])
                l_labels_min.append(t_label_min)
            
            j = j + ss
        
        j = y_start
        i = i + ss
        
    samples = np.array(l_samples, dtype = 'float64')
    labels_max = np.array(l_labels_max, dtype = 'float64')
    labels_min = np.array(l_labels_min, dtype = 'float64')
    
    return samples, labels_max, labels_min

In [6]:
# Function for generating expressions

def exp_generator(num, h_deg):
    
    if (num == 1):
        
        seed(num + h_deg**4)
        
        a = uniform(-1.0, 1.0)
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 1.0)
        d = uniform(-0.5, 0.5)
        e = uniform(-3.0, 3.0)
        
        if(h_deg == 3):
            a = 0.0
            b = uniform(-0.5, 0.5)
            c = uniform(-2.0, 2.0)
            d = uniform(-5.0, 0.5)
        elif(h_deg == 2):
            a = 0.0
            b = 0.0
            c = uniform(-0.5, 0.5)
        
        f = a*x**4 + b*x**3 + c*x**2 + d*x + e
        
        return f
    
    elif (num == 2):
        
        seed(num + h_deg)
        
        a = uniform(-5.0, 5.0)
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-5.0, 5.0)
        e = uniform(-5.0, 5.0)
        f = uniform(-5.0, 5.0)
        g = uniform(-5.0, 5.0)
        h = uniform(-5.0, 5.0)
        i = uniform(-5.0, 5.0)
        j = uniform(-5.0, 5.0)
        k = uniform(-5.0, 5.0)
        l = uniform(-5.0, 5.0)
        m = uniform(-5.0, 5.0)
        n = uniform(-5.0, 5.0)
        o = uniform(-5.0, 5.0)
        
        if(h_deg == 3):
            a = 0.0
            b = 0.0
            c = 0.0
            d = 0.0
            e = 0.0
        elif(h_deg == 2):
            a = 0.0
            b = 0.0
            c = 0.0
            d = 0.0
            e = 0.0
            f = 0.0
            g = 0.0
            h = 0.0
            i = 0.0
        
        f = a*x**4 + b*y**4 + c*x**3*y + d*x*y**3 + e*x**2*y**2 + f*x**3 + g*y**3 + h*x**2*y + i*x*y**2 + j*x**2 + k*y**2 + l*x*y + m*x + n*y + o
        
        return f
    
    elif (num == 3):
        
        seed(num**num)
        
        a = uniform(-2.0, 2.0)
        
        while(a == 0):
            a = uniform(-2.0, 2.0)
        
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-1.0, 1.0)
        
        f = a*sympy.sin(b*x-c) + d
        
        return f
    
    elif (num == 4):
        
        seed(num**num)
        
        a = uniform(-2.0, 2.0)
        
        while(a == 0):
            a = uniform(-2.0, 2.0)
        
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-1.0, 1.0)
        
        f = a*sympy.cos(b*x-c) + d
        
        return f
    
    elif (num == 5):
        
        f = x**2 + y**2
        
        return f
        
    elif (num == 6):
        
        seed(num*4)
        
        a = uniform(0.0, 3.0)
        b = uniform(0.0, 3.0)
        
        f = a*x**2 + b*y**2
        
        return f
    
    elif (num == 7):
        
        seed(num**h_deg)
        
        a = uniform(-2.0, 2.0)
        
        while(a == 0):
            a = uniform(-2.0, 2.0)
        
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-2.0, 2.0)
        e = uniform(-1.0, 1.0)
        g = uniform(-5.0, 5.0)
        h = uniform(-3.0, 3.0)
        
        f = a*(sympy.sin(b*x-c))**2 + d*sympy.sin(e*x-g) + h
        
        return f
    
    else:
        f = 0
        return f

##### A smaller-sized dataset was generated for this experiment, as this was the first experiment using CNN architecture on the combination dataset.

In [9]:
funct1 = exp_generator(3, 1)
funct1

0.593988879915532*sin(2.01369540968675*x - 4.57047002658068) - 0.607193781823679

In [10]:
s1, l_max1, l_min1 = datalist_generator(funct1, 16, 16, 47, 47, 1)

In [11]:
s1_i, l_max1_i, l_min1_i = datalist_generator(funct1, 48, 48, 79, 79, 1)

In [12]:
s1_ii, l_max1_ii, l_min1_ii = datalist_generator(funct1, 80, 80, 111, 111, 1)

In [13]:
funct2 = exp_generator(5, 1)
funct2

x**2 + y**2

In [14]:
s2, l_max2, l_min2 = datalist_generator(funct2, 16, 16, 47, 47, 1)

In [15]:
s2_i, l_max2_i, l_min2_i = datalist_generator(funct2, 48, 48, 79, 79, 1)

In [16]:
s2_ii, l_max2_ii, l_min2_ii = datalist_generator(funct2, 80, 80, 111, 111, 1)

In [17]:
s = np.concatenate((s1, s1_i, s1_ii, s2, s2_i, s2_ii))
l_max = np.concatenate((l_max1, l_max1_i, l_max1_ii, l_max2, l_max2_i, l_max2_ii))

s, l_max = shuffle(s, l_max)

train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s, l_max, test_size = 0.3, random_state = 4)

In [18]:
funct3 = exp_generator(1, 2)
funct3

0.204219866843413*x**2 - 0.210374622235534*x + 1.59664462678772

In [19]:
s3, l_max3, l_min3 = datalist_generator(funct3, 0, 0, 15.5, 15.5, 0.5)

In [20]:
s3_i, l_max3_i, l_min3_i = datalist_generator(funct3, 16, 16, 31.5, 31.5, 0.5)

In [21]:
s_i = np.concatenate((s3, s3_i))
l_max_i = np.concatenate((l_max3, l_max3_i))

s_i, l_max_i = shuffle(s_i, l_max_i)

train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s_i, l_max_i, test_size = 0.3, random_state = 11)

In [22]:
train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [26]:
base = VGG16(weights = 'imagenet', include_top = False, input_shape = (33, 33, 3))
base.trainable = False

inp = Input(shape = (33, 33, 3))
p_out = preprocess_input(inp)
temp = base(p_out)

temp = Flatten()(temp)
temp = Dense(units = 128, activation = 'relu')(temp)
temp = Dense(units = 64, activation = 'relu')(temp)
temp = Dense(units = 32, activation = 'relu')(temp)
temp = Dense(units = 8, activation = 'relu')(temp)
out = Dense(units = 1)(temp)

model = Model(inp, out)

In [27]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [28]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 75, shuffle = True, verbose = 2)

Epoch 1/75
215/215 - 134s - loss: 850.0278 - mean_absolute_error: 10.2511 - val_loss: 11.4338 - val_mean_absolute_error: 1.9174 - 134s/epoch - 622ms/step
Epoch 2/75
215/215 - 154s - loss: 7.1908 - mean_absolute_error: 1.4785 - val_loss: 3.1790 - val_mean_absolute_error: 1.0876 - 154s/epoch - 714ms/step
Epoch 3/75
215/215 - 154s - loss: 2.8872 - mean_absolute_error: 1.0123 - val_loss: 1.7474 - val_mean_absolute_error: 0.8586 - 154s/epoch - 714ms/step
Epoch 4/75
215/215 - 152s - loss: 1.7787 - mean_absolute_error: 0.8446 - val_loss: 1.2936 - val_mean_absolute_error: 0.7631 - 152s/epoch - 708ms/step
Epoch 5/75
215/215 - 152s - loss: 1.1980 - mean_absolute_error: 0.7316 - val_loss: 0.9614 - val_mean_absolute_error: 0.6754 - 152s/epoch - 708ms/step
Epoch 6/75
215/215 - 153s - loss: 0.9288 - mean_absolute_error: 0.6631 - val_loss: 0.8515 - val_mean_absolute_error: 0.6311 - 153s/epoch - 712ms/step
Epoch 7/75
215/215 - 153s - loss: 0.8449 - mean_absolute_error: 0.6414 - val_loss: 0.7220 - val_

Epoch 56/75
215/215 - 86s - loss: 3.4891 - mean_absolute_error: 0.8471 - val_loss: 0.3363 - val_mean_absolute_error: 0.4032 - 86s/epoch - 399ms/step
Epoch 57/75
215/215 - 85s - loss: 0.3270 - mean_absolute_error: 0.4097 - val_loss: 0.4220 - val_mean_absolute_error: 0.4728 - 85s/epoch - 398ms/step
Epoch 58/75
215/215 - 86s - loss: 0.3329 - mean_absolute_error: 0.4081 - val_loss: 0.2998 - val_mean_absolute_error: 0.3884 - 86s/epoch - 399ms/step
Epoch 59/75
215/215 - 86s - loss: 0.3301 - mean_absolute_error: 0.4090 - val_loss: 0.4736 - val_mean_absolute_error: 0.4840 - 86s/epoch - 399ms/step
Epoch 60/75
215/215 - 86s - loss: 0.3303 - mean_absolute_error: 0.4090 - val_loss: 0.3209 - val_mean_absolute_error: 0.4083 - 86s/epoch - 398ms/step
Epoch 61/75
215/215 - 86s - loss: 0.3560 - mean_absolute_error: 0.4223 - val_loss: 0.2893 - val_mean_absolute_error: 0.3840 - 86s/epoch - 400ms/step
Epoch 62/75
215/215 - 86s - loss: 0.3756 - mean_absolute_error: 0.4416 - val_loss: 0.2752 - val_mean_absol

##### Obtained on the 2nd attempt. The results are higher, and they oscillated quite enough throughout the epochs, though it can be said that the general trend was downward.

In [ ]:
model.save('Research_cnncomb.h5') # Overwritten, as the experiment was performed again with some changes to the architecture

In [29]:
result1 = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

123/123 - 36s - loss: 0.3303 - mean_absolute_error: 0.4157 - 36s/epoch - 294ms/step


In [30]:
for i in test_labels:
    print(i)

0.04116645244171622
0.03499369473554726
0.02885249003225535
0.010098820766430691
0.0017124481616056142
0.017997884759087503
0.8212621468035939
0.00030132795580947887
0.0005180729107614556
1.141224642306315
1.9897550938814967
0.00108918813280051
0.021830887122399202
0.1476193048926105
0.010796939069782727
2.1982866505352483
0.024774570557832002
2.338842482648719
2.0044595303392625
0.005516623021640118
0.010792536519737252
0.6354018266976316
0.6354018266976316
0.00037886448018202275
0.002910737308188308
0.19144158851282628
0.37542165010302037
0.3509385613655042
0.00022014359038362313
0.008703596533569953
0.007669593494611221
2.157632584649323
0.10417930577530653
0.02300675059582937
0.37542165010302037
1.36030663860646
0.007384499330910623
0.007786582314558242
0.05859359763850934
0.005516623021640118
0.9711488571467721
0.006864750414032364
0.019591843262589434
0.38275122656342403
0.006984260370872139
0.007807438528197859
0.9830487975938474
0.0006356435361348629
0.00037886448018202275
0.00

In [31]:
predictions1 = model.predict(x = test_samples, batch_size = 20, verbose = 2)

123/123 - 36s - 36s/epoch - 296ms/step


In [32]:
for i in predictions1:
    print(i)

[1.04489588]
[0.04866397]
[0.06921636]
[0.75587341]
[0.00695544]
[0.02587434]
[0.97319873]
[0.00702052]
[0.00818419]
[0.97514005]
[0.94920773]
[0.00256822]
[-0.08606065]
[0.85245507]
[0.32711313]
[0.95793869]
[-0.12616018]
[1.06983563]
[1.03111962]
[0.01353751]
[-0.22706136]
[0.90496851]
[0.90496851]
[0.00657689]
[0.01484186]
[0.19515381]
[1.00605766]
[1.05449371]
[0.01348344]
[0.01348152]
[0.88204531]
[1.01398947]
[0.11282643]
[-0.73086156]
[1.00605766]
[0.96890022]
[0.25291]
[1.24202097]
[0.07142864]
[0.01353751]
[0.9761435]
[0.84691379]
[0.11181055]
[0.39867273]
[0.23220437]
[-0.04741799]
[1.03363675]
[0.00305348]
[0.00657689]
[0.01000569]
[0.01348344]
[1.04142259]
[0.93621084]
[1.04709497]
[1.04424217]
[-0.23875744]
[0.01401289]
[-0.44563815]
[0.91727227]
[0.1729602]
[0.85723043]
[1.04424217]
[0.74544576]
[0.28980586]
[0.84813316]
[1.01697885]
[-0.01226468]
[-1.03526866]
[0.85177015]
[0.97514005]
[1.03633642]
[-0.41706041]
[0.11282643]
[0.01000569]
[1.05429392]
[1.02516981]
[0.1919

[1.04089134]
[0.97385547]
[1.03252745]
[1.01415653]
[-0.17865461]
[0.85177015]
[0.05522233]
[0.00831099]
[0.38943502]
[0.95793869]
[0.90496851]
[1.01195029]
[0.07233914]
[1.01410919]
[1.02089594]
[0.54892333]
[1.02516981]
[0.11493346]
[-0.02850346]
[1.02823176]
[-0.17320974]
[0.99403175]
[1.06381457]
[0.23922249]
[0.07142864]
[0.00470202]
[0.99403175]
[1.06616692]
[0.12888133]
[0.00839569]
[1.22151705]
[0.95051717]
[1.02516981]
[1.01415653]
[0.09668754]
[0.09105464]
[0.85723043]
[0.05522233]
[0.07926471]
[0.23551554]
[0.04418239]
[1.06616692]
[-0.21334323]
[1.05728421]
[0.54980645]
[0.00999728]
[0.09105464]
[0.01085636]
[1.08222899]
[-0.861604]
[0.19515381]
[1.14167336]
[0.01463329]
[0.9777306]
[0.00702052]
[0.95417751]
[1.04124787]
[-0.06300076]
[-0.09587191]
[-0.11802762]
[1.04489588]
[1.00589246]
[0.63444298]
[0.90842733]
[1.02665454]
[0.97319873]
[-0.42089231]
[0.52781033]
[0.54900631]
[0.9567129]
[0.60027455]
[0.44625753]
[0.90496851]
[0.01190577]
[0.4021424]
[0.00093469]
[0.23681

#### For a number of samples, the difference of the predictions with the corresponding actual labels was quite high. However, good thing was that there were no similar predictions. Also, some predictions were negative.

In [33]:
result2 = model.evaluate(train_samples, train_labels, batch_size = 20, verbose = 2)

287/287 - 84s - loss: 0.3295 - mean_absolute_error: 0.4160 - 84s/epoch - 291ms/step


In [34]:
for i in train_labels:
    print(i)

0.00108918813280051
0.010873259207376414
2.2948705367927276
0.0009246156780227532
0.00037886448018202275
0.010693963984966167
0.02310479552252581
0.016245136705583516
0.45868178090236633
0.6727361813732983
0.008065237395766924
0.0073109162836549
2.0044595303392625
0.006377056356736636
2.209962943256261
0.011863431264698455
0.021836091148008857
0.25531196263689226
0.007416926308700203
0.011988951278036552
1.4998893957611765
0.34542051614909947
0.10769543304252131
1.5518539456246079
0.011456983806306326
1.7130168624890667
0.007201843331648344
0.0006356435361348629
1.282357521290466
0.0002703983746431383
0.010115340254253188
0.027469521447677638
0.026304406718934935
0.01028947630200612
0.010936148417227058
0.020305876634975753
0.02923214438469955
0.00024355420065950542
0.029917466839755234
0.35650753016850817
0.21282145958703794
0.01155606117996059
0.22769284063567924
2.3940371285725353
0.010756886527707139
0.00019963544798890157
1.5518539456246079
1.939929676815565
0.02192513290610496
0.

In [35]:
predictions2 = model.predict(x = train_samples, batch_size = 20, verbose = 2)

287/287 - 85s - 85s/epoch - 294ms/step


In [36]:
for i in predictions2:
    print(i)

[0.00256822]
[0.56768941]
[1.07608509]
[0.00461169]
[0.00657689]
[0.25772535]
[-0.19127113]
[0.10315089]
[1.0087139]
[0.96366761]
[0.09330474]
[0.51297522]
[1.03111962]
[0.01348486]
[1.05728421]
[0.55573086]
[-0.81107106]
[0.26161019]
[0.84502413]
[0.74356911]
[0.9777306]
[0.85723043]
[1.04550719]
[1.0292842]
[0.88986698]
[0.99403175]
[0.71304768]
[0.00305348]
[1.02089594]
[0.00797781]
[0.48388308]
[-0.01384743]
[-0.32864377]
[-0.01840829]
[0.44484478]
[-0.59218869]
[-0.1032707]
[0.00865737]
[-0.07206188]
[0.85930776]
[0.84741638]
[0.8613812]
[1.02516981]
[1.08222899]
[0.23407138]
[0.01401289]
[1.0292842]
[1.01408208]
[-0.03201013]
[0.95417751]
[-0.00526383]
[-0.4501464]
[0.0070233]
[0.00461169]
[0.11282643]
[1.04550719]
[-0.09129499]
[-0.58903109]
[0.00195025]
[1.00586441]
[0.99065143]
[-0.00657073]
[1.13738717]
[0.98249341]
[1.02089594]
[1.07469643]
[-0.22711664]
[-0.39512026]
[1.06983563]
[0.89906447]
[0.04866397]
[0.19968399]
[1.0119339]
[-0.1817286]
[0.01348486]
[0.93481762]
[1.04

[0.42488138]
[0.01353751]
[0.01188929]
[0.00695544]
[1.01697885]
[0.94949741]
[0.00661742]
[0.00557012]
[1.01648759]
[0.39867273]
[0.00702052]
[-0.21377756]
[0.17107134]
[0.99403175]
[-0.07567362]
[0.55564819]
[1.01408208]
[0.14544766]
[-0.11921012]
[1.03363675]
[0.00999728]
[0.14483223]
[0.00625793]
[0.01353751]
[-0.5067209]
[0.83566147]
[0.41112091]
[1.01697885]
[0.96397936]
[0.87264758]
[0.03275069]
[0.09105464]
[1.06306406]
[1.05429392]
[0.93481762]
[1.02823176]
[0.62271749]
[0.97319873]
[0.94337922]
[0.69924637]
[0.16373786]
[0.39867273]
[0.90496851]
[1.06301752]
[0.42612328]
[0.94949741]
[0.94864242]
[1.06616692]
[0.00093469]
[0.99065143]
[0.00818419]
[0.00732906]
[0.36561982]
[1.05600228]
[0.85930776]
[0.94949741]
[0.19515381]
[1.04709497]
[1.07107835]
[0.85245507]
[0.00270293]
[1.06616692]
[0.00051331]
[0.0070233]
[0.35851965]
[0.00386535]
[0.85723043]
[1.04124787]
[1.03252745]
[0.07142864]
[1.01715369]
[1.08222899]
[0.00469916]
[1.03418825]
[1.0292842]
[0.38813293]
[0.38967816

[0.89826708]
[0.00470202]
[-0.05471715]
[0.40876986]
[0.9567129]
[0.00093469]
[-0.11338797]
[-0.06628193]
[1.02823176]
[0.00332179]
[0.88986698]
[0.87396924]
[0.01348486]
[1.03204459]
[0.01000569]
[0.00557012]
[0.85245507]
[0.00432874]
[0.91727227]
[1.04709497]
[1.01324955]
[1.08222899]
[0.99403175]
[0.95417751]
[1.06301752]
[0.29308928]
[1.02914264]
[1.16814997]
[1.03633642]
[0.87264758]
[0.87264758]
[1.01324955]
[1.04442504]
[1.04836708]
[1.04550719]
[-0.63360816]
[0.00702052]
[0.07070861]
[0.00831099]
[-0.24375258]
[-0.15294042]
[-0.25550112]
[-0.11036631]
[-0.25637109]
[1.17281561]
[0.29448315]
[1.05429392]
[0.67459142]
[1.03418825]
[0.4845282]
[0.00461169]
[0.01463329]
[0.00183081]
[0.01348344]
[1.04550719]
[0.00865737]
[1.03252745]
[0.7265461]
[0.04866397]
[0.96160593]
[0.96366761]
[0.00630679]
[-0.23453436]
[-0.11154419]
[1.06983563]
[0.64630414]
[1.05429392]
[-0.18858322]
[0.93621084]
[-0.10068986]
[0.07448187]
[0.95268466]
[1.07608509]
[0.96890022]
[-0.16746722]
[0.99599825]
[

#### Similar type of predictions as like that for test set.

### Looking at the results of this experiment, we can conclude that, even though this particular architecture did not give a great performance (the difference between the predictions and actual curvatures were high, as a result of which the losses and errors were high too), it could solve the problem we were facing where our previous models were predicting similar curvatures for different samples.
### Since the architecture we just used is still struggling to accurately predict our curvatures, we change the architecture a little by adding another dense hidden layer at the top with 256 nodes, as we had found earlier this particular architecture to perform better than the one we just used.

## Architecture - Transfer Learning applied with VGG16 Model - Layers added to the top (5 Dense Hidden Layers, Max Width - 256)
### Combination of samples used - Sine Wave Surface, Parabolic Cylinder (Quadratic Univariate Polynomial), Circular Paraboloid (z = x^2 + y^2)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked<br>Using a smaller step size for Parabolic Cylinder samples

In [2]:
tf.keras.backend.set_floatx('float64') # Default float type set to 'float64'

In [7]:
funct1 = exp_generator(3, 1)
funct1

0.593988879915532*sin(2.01369540968675*x - 4.57047002658068) - 0.607193781823679

In [8]:
s1, l_max1, l_min1 = datalist_generator(funct1, 16, 16, 47, 47, 1)

In [9]:
s1_i, l_max1_i, l_min1_i = datalist_generator(funct1, 48, 48, 79, 79, 1)

In [10]:
s1_ii, l_max1_ii, l_min1_ii = datalist_generator(funct1, 80, 80, 111, 111, 1)

In [11]:
s1_iii, l_max1_iii, l_min1_iii = datalist_generator(funct1, 112, 112, 143, 143, 1)

In [12]:
funct2 = exp_generator(5, 1)
funct2

x**2 + y**2

In [13]:
s2, l_max2, l_min2 = datalist_generator(funct2, 16, 16, 47, 47, 1)

In [14]:
s2_i, l_max2_i, l_min2_i = datalist_generator(funct2, 48, 48, 79, 79, 1)

In [15]:
s2_ii, l_max2_ii, l_min2_ii = datalist_generator(funct2, 80, 80, 111, 111, 1)

In [16]:
s2_iii, l_max2_iii, l_min2_iii = datalist_generator(funct2, 112, 112, 143, 143, 1)

In [17]:
s = np.concatenate((s1, s1_i, s1_ii, s1_iii, s2, s2_i, s2_ii, s2_iii))
l_max = np.concatenate((l_max1, l_max1_i, l_max1_ii, l_max1_iii, l_max2, l_max2_i, l_max2_ii, l_max2_iii))

s, l_max = shuffle(s, l_max)

train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s, l_max, test_size = 0.3, random_state = 4)

In [18]:
funct3 = exp_generator(1, 2)
funct3

0.204219866843413*x**2 - 0.210374622235534*x + 1.59664462678772

In [19]:
s3, l_max3, l_min3 = datalist_generator(funct3, 0, 0, 15.5, 15.5, 0.5)

In [20]:
s3_i, l_max3_i, l_min3_i = datalist_generator(funct3, 16, 16, 31.5, 31.5, 0.5)

In [21]:
s3_ii, l_max3_ii, l_min3_ii = datalist_generator(funct3, 32, 32, 47.5, 47.5, 0.5)

In [32]:
s3_ii.shape

(0,)

##### All the samples in the 3rd bunch for the Parabolic Cylinder expression had curvature values less than the threshold selected. So these were all discarded

In [35]:
s_i = np.concatenate((s3, s3_i))
l_max_i = np.concatenate((l_max3, l_max3_i))

s_i, l_max_i = shuffle(s_i, l_max_i)

train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s_i, l_max_i, test_size = 0.3, random_state = 11)

In [36]:
train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [52]:
base = VGG16(weights = 'imagenet', include_top = False, input_shape = (33, 33, 3))
base.trainable = False

inp = Input(shape = (33, 33, 3))
p_out = preprocess_input(inp)
temp = base(p_out)

temp = Flatten()(temp)
temp = Dense(units = 256, activation = 'relu')(temp)
temp = Dense(units = 128, activation = 'relu')(temp)
temp = Dense(units = 64, activation = 'relu')(temp)
temp = Dense(units = 32, activation = 'relu')(temp)
temp = Dense(units = 8, activation = 'relu')(temp)
out = Dense(units = 1)(temp)

model = Model(inp, out)

In [53]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [54]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 75, shuffle = True, verbose = 2)

Epoch 1/75
268/268 - 150s - loss: 1723.5085 - mean_absolute_error: 7.7968 - val_loss: 0.5162 - val_mean_absolute_error: 0.3921 - 150s/epoch - 560ms/step
Epoch 2/75
268/268 - 106s - loss: 0.3728 - mean_absolute_error: 0.3497 - val_loss: 0.2905 - val_mean_absolute_error: 0.3371 - 106s/epoch - 396ms/step
Epoch 3/75
268/268 - 107s - loss: 0.2783 - mean_absolute_error: 0.3189 - val_loss: 0.2603 - val_mean_absolute_error: 0.3122 - 107s/epoch - 399ms/step
Epoch 4/75
268/268 - 106s - loss: 0.2538 - mean_absolute_error: 0.2937 - val_loss: 0.2576 - val_mean_absolute_error: 0.2943 - 106s/epoch - 394ms/step
Epoch 5/75
268/268 - 105s - loss: 0.2503 - mean_absolute_error: 0.2867 - val_loss: 0.2571 - val_mean_absolute_error: 0.3030 - 105s/epoch - 392ms/step
Epoch 6/75
268/268 - 104s - loss: 0.2486 - mean_absolute_error: 0.2846 - val_loss: 0.2522 - val_mean_absolute_error: 0.2945 - 104s/epoch - 390ms/step
Epoch 7/75
268/268 - 104s - loss: 0.2485 - mean_absolute_error: 0.2843 - val_loss: 0.2519 - val_m

Epoch 56/75
268/268 - 107s - loss: 0.2375 - mean_absolute_error: 0.2716 - val_loss: 0.2430 - val_mean_absolute_error: 0.2848 - 107s/epoch - 398ms/step
Epoch 57/75
268/268 - 105s - loss: 0.2387 - mean_absolute_error: 0.2730 - val_loss: 0.2451 - val_mean_absolute_error: 0.2767 - 105s/epoch - 394ms/step
Epoch 58/75
268/268 - 106s - loss: 0.2369 - mean_absolute_error: 0.2710 - val_loss: 0.2453 - val_mean_absolute_error: 0.2750 - 106s/epoch - 396ms/step
Epoch 59/75
268/268 - 105s - loss: 0.2332 - mean_absolute_error: 0.2688 - val_loss: 0.2394 - val_mean_absolute_error: 0.2789 - 105s/epoch - 393ms/step
Epoch 60/75
268/268 - 106s - loss: 0.2326 - mean_absolute_error: 0.2683 - val_loss: 0.2389 - val_mean_absolute_error: 0.2780 - 106s/epoch - 395ms/step
Epoch 61/75
268/268 - 106s - loss: 0.2328 - mean_absolute_error: 0.2688 - val_loss: 0.2377 - val_mean_absolute_error: 0.2791 - 106s/epoch - 395ms/step
Epoch 62/75
268/268 - 139s - loss: 0.2392 - mean_absolute_error: 0.2715 - val_loss: 0.2397 - v

##### Obtained on the 2nd attempt. The results didn't improve a lot in comparison to the 1st run. There were continuous minute oscillations throughout, but the results gradually but slowly got down. However, they were still high enough in value.

In [27]:
model.save('Research_cnncomb.h5') # Deleted eventually, as this model gives the problem of predicting similar value too

In [55]:
result1 = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

154/154 - 45s - loss: 0.1999 - mean_absolute_error: 0.2469 - 45s/epoch - 289ms/step


In [56]:
for i in test_labels:
    print(i)

1.34548497389608
0.011077485224401384
0.006011320960619172
0.00942463534950522
2.405157376276527
0.0012949982914153728
0.36967720494279216
0.6727361813732983
0.007422323289873542
2.405157376276527
0.007280047099145069
0.00537686865004448
0.005139825958930979
0.006911153710825969
2.157632584649323
2.1449126730981924
0.007852320269114298
0.40841652661884664
0.005586221705704877
0.0003185694564544265
0.05859359763850934
0.006856999561343181
0.35650753016850817
0.02462389517177059
0.011218089169663914
1.781726235842214
2.4032664081451545
0.044932345660558884
0.0032749387201403994
0.023503401629062073
0.005354392522390884
2.4032664081451545
0.7873259312392566
1.282357521290466
1.781726235842214
0.011494063053769576
0.43711677227308515
0.1611202090731316
0.005176052079254924
0.9096141284141153
0.017830865105689514
0.014138954716731475
0.001555718214597806
1.1279393146786703
0.028997691150759522
0.56169754076809
0.11154062043962387
0.0018907926941605807
0.044821792108735846
0.4307769307365432

In [57]:
predictions1 = model.predict(x = test_samples, batch_size = 20, verbose = 2)

154/154 - 45s - 45s/epoch - 294ms/step


In [58]:
for i in predictions1:
    print(i)

[1.15057909]
[0.00837023]
[0.00837023]
[0.00837023]
[1.02514892]
[0.00837023]
[0.84757088]
[0.92802726]
[0.00837023]
[1.02514892]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[1.04974683]
[1.23219598]
[0.00837023]
[0.42771063]
[0.00837023]
[0.00837023]
[0.06725966]
[0.00837023]
[0.65246002]
[0.00837023]
[0.00837023]
[0.92585742]
[1.2962151]
[0.04718349]
[0.00837023]
[0.00837023]
[0.00837023]
[1.2962151]
[1.07284774]
[1.02419372]
[0.92585742]
[0.00837023]
[0.68560478]
[1.10167409]
[0.00837023]
[1.05780474]
[0.00837023]
[0.00837023]
[0.00837023]
[1.16540143]
[0.00837023]
[0.89828642]
[0.76797113]
[0.00837023]
[0.69155266]
[1.08216413]
[0.00837023]
[0.00837023]
[0.76957561]
[0.00837023]
[0.00837023]
[0.00837023]
[1.25123299]
[0.00837023]
[0.7238091]
[0.42771063]
[0.00837023]
[0.73073017]
[1.16419795]
[0.00837023]
[0.00837023]
[0.75112045]
[0.00837023]
[1.15057909]
[0.00837023]
[1.01963499]
[0.00837023]
[0.00837023]
[1.07284774]
[0.00837023]
[0.00837023]
[1.12040578]
[0.84757088]
[0

[1.04200422]
[0.00837023]
[1.00202536]
[0.00837023]
[0.00837023]
[1.04554953]
[0.00837023]
[0.00837023]
[0.00837023]
[0.95065204]
[0.00837023]
[1.33836729]
[0.00837023]
[0.00837023]
[0.00837023]
[0.75112045]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[0.02649821]
[0.00837023]
[1.05446281]
[1.35591253]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[1.11128952]
[0.00837023]
[0.68560478]
[1.10167409]
[0.00837023]
[1.03447584]
[0.00837023]
[0.76957561]
[0.00837023]
[0.62513431]
[0.00837023]
[0.00837023]
[0.00837023]
[0.62268617]
[0.00837023]
[0.03686506]
[0.00837023]
[0.03686506]
[0.00837023]
[1.30076231]
[1.1439309]
[0.73073017]
[0.00837023]
[0.93352732]
[0.00837023]
[0.00837023]
[0.00837023]
[1.02419372]
[0.00837023]
[0.00837023]
[1.00202536]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[0.95719374]
[0.00837023]
[0.00837023]
[0.92585742]
[0.62513431]
[0.00837023]
[0.63634817]
[0.00837023]
[0.00837023]
[0.00837023]
[0.73073017]
[0.00837023]


[0.00837023]
[0.40604519]
[0.00837023]
[0.00837023]
[1.15057909]
[1.00956989]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[0.01239439]
[0.00837023]
[0.00837023]
[1.23982729]
[0.95412956]
[0.00837023]
[0.00837023]
[0.00837023]
[0.90803483]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[0.73029186]
[0.00837023]
[0.00837023]
[0.78972848]
[0.00837023]
[0.14549367]
[0.00837023]
[0.00837023]
[1.16419795]
[0.00837023]
[1.04736607]
[0.00837023]
[0.00837023]
[0.06725966]
[1.1044918]
[0.00837023]
[0.08359198]
[1.05446281]
[0.00837023]
[0.00837023]
[0.955849]
[0.00837023]
[0.00837023]
[0.6476138]
[0.00837023]
[0.93352732]
[1.11720539]
[0.00837023]
[0.00837023]
[0.00837023]
[1.13958987]
[1.13958987]
[0.00837023]
[1.01963499]
[0.80429422]
[0.00837023]
[0.00837023]
[1.04736607]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[0.7238091]
[0.00837023]
[1.16540143]
[0.00837023]
[1.07284774]
[0.00837023]
[0.00837023]
[0.00837023]
[1.09903433]
[0.8

[0.93118013]
[0.00837023]
[0.00837023]
[0.27501229]
[0.00837023]
[1.07952055]
[0.73029186]
[1.13889567]
[0.00837023]
[0.00837023]
[0.9530044]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[1.04136678]
[0.00837023]
[0.6476138]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[0.04718349]
[0.00837023]
[0.27501229]
[0.00837023]
[0.82418599]
[1.07284774]
[0.74775354]
[0.00837023]
[0.00837023]
[0.6476138]
[1.04495342]
[0.00837023]
[0.02649821]
[1.03447584]
[1.03447584]
[1.04136678]
[1.23982729]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[0.00837023]
[1.06721582]
[1.06721582]
[0.06725966]
[1.04136678]
[0.91444658]
[0.00837023]
[1.1044918]
[1.08769228]
[1.33836729]
[0.00837023]
[0.00837023]
[0.90803483]
[0.00837023]
[1.05415604]
[1.1439309]
[0.00837023]
[0.65980499]
[0.00837023]
[1.09903433]
[0.95976499]
[1.34617063]
[0.00837023]
[0.00837023]
[0.00837023]
[1.08769228]
[0.42771063]
[0.96059833]
[1.25670514]
[0.00837023]
[0.00837023]
[0.0

#### This model is giving the same issue of outputting similar predictions for different samples having different curvatures.

In [59]:
result2 = model.evaluate(train_samples, train_labels, batch_size = 20, verbose = 2)

358/358 - 105s - loss: 0.2059 - mean_absolute_error: 0.2543 - 105s/epoch - 293ms/step


In [ ]:
for i in train_labels:
    print(i)

In [61]:
predictions2 = model.predict(x = train_samples, batch_size = 20, verbose = 2)

358/358 - 105s - 105s/epoch - 292ms/step


In [ ]:
for i in predictions2:
    print(i)

#### Similar type of predictions as like the case of test set.

### Unfortunately, even after restricting samples with very low curvatures and changing the architecture, the same problem creeps up again on our model.<br>So next, we again employ Transfer Learning, but this time with a different CNN architecture called Resnet50V2, which has been found to work well for regression problems. We add 3 dense layers at the top with a maximum width of 256 nodes and check out how the model performs with our first combination dataset.

## Architecture - Transfer Learning applied with Resnet50V2 Model - Layers added to the top (3 Dense Hidden Layers, Max Width - 256)
### Combination of samples used - Sine Wave Surface, Parabolic Cylinder (Quadratic Univariate Polynomial), Circular Paraboloid (z = x^2 + y^2)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked<br>Using a smaller step size for Parabolic Cylinder samples

In [1]:
import sympy
import numpy as np
import tensorflow as tf
from tensorflow import keras
from random import uniform, seed
from sklearn.utils import shuffle
from tensorflow.keras.models import *
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.metrics import *
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.applications.resnet_v2 import preprocess_input

In [2]:
tf.keras.backend.set_floatx('float64') # Default float type set

In [7]:
funct1 = exp_generator(3, 1)
funct1

0.593988879915532*sin(2.01369540968675*x - 4.57047002658068) - 0.607193781823679

In [8]:
s1, l_max1, l_min1 = datalist_generator(funct1, 16, 16, 47, 47, 1)

In [9]:
s1_i, l_max1_i, l_min1_i = datalist_generator(funct1, 48, 48, 79, 79, 1)

In [10]:
s1_ii, l_max1_ii, l_min1_ii = datalist_generator(funct1, 80, 80, 111, 111, 1)

In [11]:
funct2 = exp_generator(5, 1)
funct2

x**2 + y**2

In [12]:
s2, l_max2, l_min2 = datalist_generator(funct2, 16, 16, 47, 47, 1)

In [13]:
s2_i, l_max2_i, l_min2_i = datalist_generator(funct2, 48, 48, 79, 79, 1)

In [14]:
s2_ii, l_max2_ii, l_min2_ii = datalist_generator(funct2, 80, 80, 111, 111, 1)

In [15]:
s = np.concatenate((s1, s1_i, s1_ii, s2, s2_i, s2_ii))
l_max = np.concatenate((l_max1, l_max1_i, l_max1_ii, l_max2, l_max2_i, l_max2_ii))

s, l_max = shuffle(s, l_max)

train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s, l_max, test_size = 0.3, random_state = 4)

In [16]:
funct3 = exp_generator(1, 2)
funct3

0.204219866843413*x**2 - 0.210374622235534*x + 1.59664462678772

In [17]:
s3, l_max3, l_min3 = datalist_generator(funct3, 0, 0, 15.5, 15.5, 0.5)

In [18]:
s3_i, l_max3_i, l_min3_i = datalist_generator(funct3, 16, 16, 31.5, 31.5, 0.5)

In [19]:
s_i = np.concatenate((s3, s3_i))
l_max_i = np.concatenate((l_max3, l_max3_i))

s_i, l_max_i = shuffle(s_i, l_max_i)

train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s_i, l_max_i, test_size = 0.3, random_state = 11)

In [20]:
train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [21]:
base = ResNet50V2(weights = 'imagenet', include_top = False, input_shape = (33, 33, 3))
base.trainable = False

inp = Input(shape = (33, 33, 3))
p_out = preprocess_input(inp)
temp = base(p_out)

temp = Flatten()(temp)
temp = Dense(units = 256, activation = 'relu')(temp)
temp = Dense(units = 32, activation = 'relu')(temp)
temp = Dense(units = 8, activation = 'relu')(temp)
out = Dense(units = 1)(temp)

model = Model(inp, out)

In [22]:
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 33, 33, 3)]       0         
                                                                 
 tf.math.truediv (TFOpLambda  (None, 33, 33, 3)        0         
 )                                                               
                                                                 
 tf.math.subtract (TFOpLambd  (None, 33, 33, 3)        0         
 a)                                                              
                                                                 
 resnet50v2 (Functional)     (None, 2, 2, 2048)        23564800  
                                                                 
 flatten (Flatten)           (None, 8192)              0         
                                                                 
 dense (Dense)               (None, 256)               209740

In [23]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [24]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 50, shuffle = True, verbose = 2)

Epoch 1/50
215/215 - 42s - loss: 18.0399 - mean_absolute_error: 0.7061 - val_loss: 0.5853 - val_mean_absolute_error: 0.3780 - 42s/epoch - 194ms/step
Epoch 2/50
215/215 - 37s - loss: 0.5907 - mean_absolute_error: 0.3799 - val_loss: 0.5814 - val_mean_absolute_error: 0.3749 - 37s/epoch - 171ms/step
Epoch 3/50
215/215 - 37s - loss: 0.5861 - mean_absolute_error: 0.3778 - val_loss: 0.5764 - val_mean_absolute_error: 0.3735 - 37s/epoch - 170ms/step
Epoch 4/50
215/215 - 37s - loss: 0.5806 - mean_absolute_error: 0.3772 - val_loss: 0.5705 - val_mean_absolute_error: 0.3734 - 37s/epoch - 171ms/step
Epoch 5/50
215/215 - 37s - loss: 0.5743 - mean_absolute_error: 0.3779 - val_loss: 0.5639 - val_mean_absolute_error: 0.3744 - 37s/epoch - 172ms/step
Epoch 6/50
215/215 - 37s - loss: 0.5675 - mean_absolute_error: 0.3796 - val_loss: 0.5570 - val_mean_absolute_error: 0.3761 - 37s/epoch - 172ms/step
Epoch 7/50
215/215 - 37s - loss: 0.5602 - mean_absolute_error: 0.3805 - val_loss: 0.5493 - val_mean_absolute_er

##### The results had a slow but gradual descent, but were higher in value compared to previous experiments.

In [25]:
result1 = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

123/123 - 20s - loss: 0.4189 - mean_absolute_error: 0.4445 - 20s/epoch - 160ms/step


In [26]:
for i in test_labels:
    print(i)

0.0037017492307321598
0.010080332737114572
0.0304819971239475
0.01905453351160852
1.5833225832642632
2.157632584649323
0.008703596533569953
0.007612270364311722
0.15165011513707144
0.01028947630200612
0.011407730615371668
0.0070485046575615665
0.009360419977768545
0.001555718214597806
2.331335715057423
0.02086391813081353
0.012273301782875102
1.0974257023109544
0.011735302943237744
0.9711488571467721
0.014427371025549026
0.01133276356148845
0.0007914843774896349
0.01045231134978854
0.00942463534950522
0.018948197750903915
0.8109447896194174
0.6354018266976316
0.20842051281282897
0.11540148396937845
0.27549441125015
0.5293148806682709
0.0004853606053990628
0.007031582081000492
0.8109447896194174
0.0007914843774896349
0.0003371487354588797
0.0002703983746431383
0.6354018266976316
0.017919391725820175
0.00998739885839698
2.0044595303392625
0.020796133942132814
0.30212211254071164
0.007856196462927107
0.018130198127661083
0.006962388938135722
0.011536051335727243
0.5366664560429131
0.00742

In [27]:
predictions1 = model.predict(x = test_samples, batch_size = 20, verbose = 2)

123/123 - 20s - 20s/epoch - 164ms/step


In [28]:
for i in predictions1:
    print(i)

[0.00636105]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.0126901]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.00138036]
[0.47785253]
[0.47785253]
[0.01633164]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[-0.00128201]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[-0.01254181]
[0.47785253]
[0.47785253]
[-0.00128201]
[-0.00253292]
[-0.00108152]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.01090904]
[0.47785253]
[-0.00933905]
[-0.01254181]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.03080798]
[-0.00504179]
[0.47785253]
[0.47785253]
[-0.00814997]
[0.47785253]
[0.47785253]
[0.00034398]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.47785253]
[0.0

[0.47785253]
[0.47785253]
[0.38729177]
[0.47785253]
[-0.00790309]
[0.47785253]
[0.00122502]
[0.00731644]
[0.47785253]
[0.47785253]
[0.01090904]


#### The same problem is taking place again: predictions are the same for majority of the samples, even though their corresponding actual labels are different.

In [29]:
result2 = model.evaluate(train_samples, train_labels, batch_size = 20, verbose = 2)

286/286 - 44s - loss: 0.4056 - mean_absolute_error: 0.4389 - 44s/epoch - 153ms/step


In [ ]:
for i in train_labels:
    print(i)

In [31]:
predictions2 = model.predict(x = train_samples, batch_size = 20, verbose = 2)

286/286 - 43s - 43s/epoch - 149ms/step


In [ ]:
for i in predictions2:
    print(i)

#### Similar type of predictions for train set too. Output deleted for that reason to save memory.

### Unfortunately, the Resnet50V2 architecture fails to perform as well.
### So finally, we decide to apply selective connectivity for the first hidden layer of our architecture with the inputs, instead of making it fully connected. The research we are working on was based on a previous research done by my professor, Dr. Hauenstein, where he developed mathematical models to estimate surface curvatures from range images of those surfaces. These mathematical models utilized the concept of partial derivatives. We decided to replicate the use of that concept in our research by allowing some nodes in the initial layers of our model to consider our inputs, which we visualized as 3x3 structures, only row-wise, some other nodes only column-wise, and the rest of the nodes across the diagonal, and then propagate those information throughout the rest of the network as usual.<br>In order to accomplish this, the way we structured our model was that we applied this selective connectivity only for the first hidden layer of our network. We employed the use of a masking array in our architecture which ensured a few nodes in the first layer were connected to the row-wise elements of our inputs only, by masking off the rest of the elements in our inputs for these nodes using zeros. Similarly, a few other nodes in that layer was connected to the column-wise elements, and the rest of the nodes to the diagonal elements. A total of 8 nodes were used for the first hidden layer. The rest of the network had all dense layers, with our architecture having a total of 12 layers and a maximum width of 28 nodes.<br>We then used this architecture to look at its performance on the first combination dataset.

## Architecture - Applying selective connectivity for the first hidden layer;<br>&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&ensp;&nbsp;Depth(12 layers in total), Width (28 nodes highest among the layers)
### Combination of samples used - Sine Wave Surface, Parabolic Cylinder (Quadratic Univariate Polynomial), Circular Paraboloid (z = x^2 + y^2)<br>"Unsigned" & "Split" Labels<br>Training & Testing from scratch on Max Valued Curvatures<br>Seeded expressions used & predictions checked<br>Using a smaller step size for Parabolic Cylinder samples

In [1]:
import sympy
import numpy as np
import tensorflow as tf
from tensorflow import keras
from random import uniform, seed
from sklearn.utils import shuffle
from tensorflow.keras.models import *
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

In [2]:
x = sympy.Symbol('x')
y = sympy.Symbol('y')

In [3]:
# Function for generating input sample and unsigned labels for a particular point of interest

def data_generator(exp, center_x, center_y, ss):
    l_sample = []
    
    i = center_x - ss
    j = center_y - ss
    
    while (i < center_x+ss+ss):
        while(j < center_y+ss+ss):
            value = exp.evalf(subs={x: i, y: j})
            l_sample.append(value)
            
            j = j + ss
        
        j = center_y - ss
        i = i + ss
    
    fx = exp.diff(x, 1)
    fy = exp.diff(y, 1)
    fxx = exp.diff(x, 2)
    fyy = exp.diff(y, 2)
    fxy = exp.diff(x, y, 1)
    
    v_fx = fx.evalf(subs={x: center_x, y: center_y})
    v_fy = fy.evalf(subs={x: center_x, y: center_y})
    v_fxx = fxx.evalf(subs={x: center_x, y: center_y})
    v_fyy = fyy.evalf(subs={x: center_x, y: center_y})
    v_fxy = fxy.evalf(subs={x: center_x, y: center_y})
    
    K = (v_fxx*v_fyy - v_fxy**2) / (1 + v_fx**2 + v_fy**2)**2
    H = (v_fxx + v_fyy + v_fxx*v_fy**2 + v_fyy*v_fx**2 - 2*v_fx*v_fy*v_fxy) / (2 * (1 + v_fx**2 + v_fy**2)**1.5)
    
    k1 = H + (H**2 - K)**0.5
    k2 = H - (H**2 - K)**0.5
    
    #*********************************************
    # Changes made to create unsigned labels
    
    u_k1 = abs(k1)
    u_k2 = abs(k2)
    
    if(u_k1 < u_k2):
        temp = u_k1
        u_k1 = u_k2
        u_k2 = temp
    
    #*********************************************
    
    #*********************************************
    # Changes made to split labels into two groups
    
    l_label_max = u_k1
    l_label_min = u_k2
    
    return l_sample, l_label_max, l_label_min

    #*********************************************

In [4]:
# Function for generating a list of samples and labels for a range of points

def datalist_generator(exp, x_start, y_start, x_end, y_end, ss):
    l_samples = []
    l_labels_max = []
    l_labels_min = []
    
    count = 0
    
    i = x_start
    j = y_start
    
    while (i < x_end+ss):
        while(j < y_end+ss):
            
            t_sample, t_label_max, t_label_min = data_generator(exp, i, j, ss)
            
            if(t_label_max >= 0.0002):
                
                l_samples.append(t_sample)
                
                l_labels_max.append(t_label_max)
                count += 1
                
                l_labels_min.append(t_label_min)
            
            j = j + ss
        
        j = y_start
        i = i + ss
        
    samples = np.array(l_samples, dtype = 'float32')
    
    # Earlier we had noticed that there was a difference in how the actual labels and the model predictions were printed out.
    # We had conducted a few experiments to figure out if this difference was causing our models to fail. Eventually, we could
    # not find any conclusive evidence that the difference was causing an issue and also failed to find the reason behind the
    # difference. Nevertheless, while conducting this current experiment we were successful in discovering why the system was
    # printing out the elements of those two vectors differently.
    # By studying the details of the output tensors formed by Keras models, we came to know that Keras models create 2D output
    # tensors. As such, when we were printing out the elements in our model's output tensor one at a time, we were looking at
    # 1D vectors with a single element corresponding to our single curvature output. However, the way we had previously set up
    # our data generating function, it was generating 1D NumPy arrays for our actual labels. As such, when we were printing out
    # the actual-label vector element-wise, the system was printing scalars.
    # That is why, we have changed our function here to generate the array for the actual labels as a 2D vector. Another fix
    # could have been was without changing anything in our function, we could have accessed the first index of each element in
    # the output tensor when printing the predictions. In that case, both the actual labels and predictions would have printed 
    # out as scalars.
    
    labels_max = np.array(l_labels_max, dtype = 'float32')
    labels_max = labels_max.reshape((count, 1), order = 'C')
    
    labels_min = np.array(l_labels_min, dtype = 'float32')
    
    return samples, labels_max, labels_min

In [5]:
# Function for generating expressions

def exp_generator(num, h_deg):
    
    if (num == 1):
        
        seed(num + h_deg**4)
        
        a = uniform(-1.0, 1.0)
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 1.0)
        d = uniform(-0.5, 0.5)
        e = uniform(-3.0, 3.0)
        
        if(h_deg == 3):
            a = 0.0
            b = uniform(-0.5, 0.5)
            c = uniform(-2.0, 2.0)
            d = uniform(-5.0, 0.5)
        elif(h_deg == 2):
            a = 0.0
            b = 0.0
            c = uniform(-0.5, 0.5)
        
        f = a*x**4 + b*x**3 + c*x**2 + d*x + e
        
        return f
    
    elif (num == 2):
        
        seed(num + h_deg)
        
        a = uniform(-5.0, 5.0)
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-5.0, 5.0)
        e = uniform(-5.0, 5.0)
        f = uniform(-5.0, 5.0)
        g = uniform(-5.0, 5.0)
        h = uniform(-5.0, 5.0)
        i = uniform(-5.0, 5.0)
        j = uniform(-5.0, 5.0)
        k = uniform(-5.0, 5.0)
        l = uniform(-5.0, 5.0)
        m = uniform(-5.0, 5.0)
        n = uniform(-5.0, 5.0)
        o = uniform(-5.0, 5.0)
        
        if(h_deg == 3):
            a = 0.0
            b = 0.0
            c = 0.0
            d = 0.0
            e = 0.0
        elif(h_deg == 2):
            a = 0.0
            b = 0.0
            c = 0.0
            d = 0.0
            e = 0.0
            f = 0.0
            g = 0.0
            h = 0.0
            i = 0.0
        
        f = a*x**4 + b*y**4 + c*x**3*y + d*x*y**3 + e*x**2*y**2 + f*x**3 + g*y**3 + h*x**2*y + i*x*y**2 + j*x**2 + k*y**2 + l*x*y + m*x + n*y + o
        
        return f
    
    elif (num == 3):
        
        seed(num**num)
        
        a = uniform(-2.0, 2.0)
        
        while(a == 0):
            a = uniform(-2.0, 2.0)
        
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-1.0, 1.0)
        
        f = a*sympy.sin(b*x-c) + d
        
        return f
    
    elif (num == 4):
        
        seed(num**num)
        
        a = uniform(-2.0, 2.0)
        
        while(a == 0):
            a = uniform(-2.0, 2.0)
        
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-1.0, 1.0)
        
        f = a*sympy.cos(b*x-c) + d
        
        return f
    
    elif (num == 5):
        
        f = x**2 + y**2
        
        return f
        
    elif (num == 6):
        
        seed(num*4)
        
        a = uniform(0.0, 3.0)
        b = uniform(0.0, 3.0)
        
        f = a*x**2 + b*y**2
        
        return f
    
    elif (num == 7):
        
        seed(num**h_deg)
        
        a = uniform(-2.0, 2.0)
        
        while(a == 0):
            a = uniform(-2.0, 2.0)
        
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-2.0, 2.0)
        e = uniform(-1.0, 1.0)
        g = uniform(-5.0, 5.0)
        h = uniform(-3.0, 3.0)
        
        f = a*(sympy.sin(b*x-c))**2 + d*sympy.sin(e*x-g) + h
        
        return f
    
    else:
        f = 0
        return f

In [6]:
funct1 = exp_generator(3, 1)
funct1

0.593988879915532*sin(2.01369540968675*x - 4.57047002658068) - 0.607193781823679

##### The architecture we used in this experiment required a batch size of 9 to work without any error. So, we set up the range of coordinates for data generation, test split percentage, and validation split percentage in a way that all our train set, test set, and validation set sizes become multiples of 9.

In [7]:
s1, l_max1, l_min1 = datalist_generator(funct1, 1, 1, 97, 96, 1)

In [8]:
funct2 = exp_generator(5, 1)
funct2

x**2 + y**2

In [9]:
s2, l_max2, l_min2 = datalist_generator(funct2, 6, 6, 102, 101, 1)

s = np.concatenate((s1, s2))
l_max = np.concatenate((l_max1, l_max2))

s, l_max = shuffle(s, l_max)

train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s, l_max, test_size = 0.25, random_state = 4)

In [10]:
funct3 = exp_generator(1, 2)
funct3

0.204219866843413*x**2 - 0.210374622235534*x + 1.59664462678772

In [11]:
s3, l_max3, l_min3 = datalist_generator(funct3, -10, -10, 66, 65, 1)
train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s3, l_max3, test_size = 0.25, random_state = 11)

train_samples = np.concatenate((train_samples1, train_samples2))
train_labels = np.concatenate((train_labels1, train_labels2))
test_samples = np.concatenate((test_samples1, test_samples2))
test_labels = np.concatenate((test_labels1, test_labels2))

train_samples, train_labels = shuffle(train_samples, train_labels)
test_samples, test_labels = shuffle(test_samples, test_labels)

In [12]:
train_samples.shape

(16362, 9)

In [13]:
test_labels.shape

(5454, 1)

In [25]:
inp = Input(shape = (9,))

temp = Dense(units = 8, activation = 'relu')(inp)

# The following is the masking array we used to accomplish the selective connectivity for the first hidden layer of our model.
# The rows corresponded to the 9 attributes of each sample and the columns corresponded to the 8 units in the 1st hiddent layer.

mask = np.array([[1, 0, 0, 1, 0, 0, 1, 0],
                 [1, 0, 0, 0, 1, 0, 0, 0],
                 [1, 0, 0, 0, 0, 1, 0, 1],
                 [0, 1, 0, 1, 0, 0, 0, 0],
                 [0, 1, 0, 0, 1, 0, 1, 1],
                 [0, 1, 0, 0, 0, 1, 0, 0],
                 [0, 0, 1, 1, 0, 0, 0, 1],
                 [0, 0, 1, 0, 1, 0, 0, 0],
                 [0, 0, 1, 0, 0, 1, 1, 0]], dtype = 'float32')

mask_layer = Multiply()([temp, mask])

temp = Dense(units = 12, activation = 'relu')(mask_layer)
temp = Dense(units = 16, activation = 'relu')(temp)
temp = Dense(units = 20, activation = 'relu')(temp)
temp = Dense(units = 24, activation = 'relu')(temp)
temp = Dense(units = 28, activation = 'relu')(temp)
temp = Dense(units = 24, activation = 'relu')(temp)
temp = Dense(units = 20, activation = 'relu')(temp)
temp = Dense(units = 16, activation = 'relu')(temp)
temp = Dense(units = 12, activation = 'relu')(temp)
temp = Dense(units = 8, activation = 'relu')(temp)

out = Dense(units = 1)(temp)

model = Model(inp, out)

In [26]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [27]:
model.fit(train_samples, train_labels, validation_split = 0.33, batch_size = 9, epochs = 75, shuffle = True, verbose = 2)

Epoch 1/75
1218/1218 - 3s - loss: 698.7891 - mean_absolute_error: 6.1825 - val_loss: 0.4433 - val_mean_absolute_error: 0.4036 - 3s/epoch - 3ms/step
Epoch 2/75
1218/1218 - 3s - loss: 0.3562 - mean_absolute_error: 0.4035 - val_loss: 0.3429 - val_mean_absolute_error: 0.4200 - 3s/epoch - 3ms/step
Epoch 3/75
1218/1218 - 3s - loss: 0.3230 - mean_absolute_error: 0.3948 - val_loss: 0.3081 - val_mean_absolute_error: 0.3875 - 3s/epoch - 3ms/step
Epoch 4/75
1218/1218 - 3s - loss: 0.2799 - mean_absolute_error: 0.3575 - val_loss: 0.2617 - val_mean_absolute_error: 0.3432 - 3s/epoch - 3ms/step
Epoch 5/75
1218/1218 - 4s - loss: 0.2427 - mean_absolute_error: 0.3264 - val_loss: 0.2241 - val_mean_absolute_error: 0.3118 - 4s/epoch - 3ms/step
Epoch 6/75
1218/1218 - 3s - loss: 0.2126 - mean_absolute_error: 0.3042 - val_loss: 0.1886 - val_mean_absolute_error: 0.2903 - 3s/epoch - 3ms/step
Epoch 7/75
1218/1218 - 4s - loss: 0.1943 - mean_absolute_error: 0.2901 - val_loss: 0.1874 - val_mean_absolute_error: 0.294

Epoch 57/75
1218/1218 - 4s - loss: 0.0912 - mean_absolute_error: 0.1212 - val_loss: 0.0932 - val_mean_absolute_error: 0.1237 - 4s/epoch - 3ms/step
Epoch 58/75
1218/1218 - 3s - loss: 0.0940 - mean_absolute_error: 0.1213 - val_loss: 0.1026 - val_mean_absolute_error: 0.1587 - 3s/epoch - 2ms/step
Epoch 59/75
1218/1218 - 3s - loss: 0.0934 - mean_absolute_error: 0.1211 - val_loss: 0.0924 - val_mean_absolute_error: 0.1237 - 3s/epoch - 3ms/step
Epoch 60/75
1218/1218 - 4s - loss: 0.0899 - mean_absolute_error: 0.1184 - val_loss: 0.0920 - val_mean_absolute_error: 0.1178 - 4s/epoch - 3ms/step
Epoch 61/75
1218/1218 - 3s - loss: 0.0938 - mean_absolute_error: 0.1221 - val_loss: 0.0910 - val_mean_absolute_error: 0.1169 - 3s/epoch - 3ms/step
Epoch 62/75
1218/1218 - 4s - loss: 0.0981 - mean_absolute_error: 0.1257 - val_loss: 0.0915 - val_mean_absolute_error: 0.1149 - 4s/epoch - 3ms/step
Epoch 63/75
1218/1218 - 4s - loss: 0.0963 - mean_absolute_error: 0.1222 - val_loss: 0.0925 - val_mean_absolute_error: 

##### Received on the 2nd try (the 1st attempt gave that same issue of predicting the same value). There is constant oscillation across the epochs, but the general trend is downward. The results are comparatively better than the first try, and in general, can be considered to be in a decent range in comparison to other experiments.

In [28]:
result1 = model.evaluate(test_samples, test_labels, batch_size = 9, verbose = 2)

606/606 - 1s - loss: 0.0870 - mean_absolute_error: 0.1130 - 890ms/epoch - 1ms/step


In [29]:
for i in test_labels:
    print(i)

[0.01506658]
[0.11154062]
[0.67273617]
[2.1319735]
[0.01145698]
[0.00755384]
[0.00291074]
[0.01065383]
[0.01214085]
[0.01311003]
[0.001295]
[0.01564746]
[0.01128647]
[1.1546179]
[2.1319735]
[0.0003572]
[0.02216075]
[0.7773554]
[0.00815376]
[0.00059284]
[0.5693898]
[0.00232847]
[1.5675682]
[0.00883185]
[0.01083808]
[0.02477457]
[0.06072289]
[0.01531365]
[0.2511465]
[0.76748747]
[1.296788]
[0.27062622]
[0.00880784]
[0.38122064]
[0.42449966]
[0.23223004]
[0.01320801]
[0.01090424]
[0.01188016]
[0.01143305]
[0.02088211]
[1.1681184]
[0.01178327]
[0.00875033]
[0.0159187]
[0.43077692]
[0.00988344]
[0.00291074]
[0.00068265]
[2.0044596]
[1.7817262]
[0.02728391]
[0.21282145]
[0.00787298]
[0.00984314]
[0.93234134]
[0.8212621]
[0.01802973]
[1.0845044]
[0.64387584]
[1.1681184]
[0.00958255]
[0.00232847]
[1.813304]
[0.01038222]
[1.7817262]
[0.01275646]
[0.63540184]
[0.22769284]
[0.03713267]
[1.2680225]
[0.0063204]
[0.9830488]
[0.4586818]
[0.37542164]
[0.5293149]
[0.001295]
[0.00783529]
[0.11154062]
[1

[0.00824751]
[0.00079148]
[2.4032664]
[0.02667615]
[0.01038894]
[0.93234134]
[0.00955801]
[0.90961415]
[0.11154062]
[0.00930072]
[0.01995118]
[0.00020956]
[0.01068297]
[1.7289168]
[0.14120427]
[0.25531197]
[0.02880457]
[2.1863558]
[0.15165012]
[0.00850233]
[0.02728391]
[0.20842052]
[0.01760561]
[0.00759929]
[2.1449127]
[0.8212621]
[0.5693898]
[1.7817262]
[0.02104797]
[0.35093856]
[0.00791132]
[0.8212621]
[0.01216508]
[0.00743289]
[0.27062622]
[0.001295]
[0.02014047]
[2.3903913]
[2.2099628]
[0.08185455]
[0.06072289]
[0.04217308]
[0.56169754]
[0.00092462]
[0.7773554]
[0.9209225]
[0.01133058]
[0.21282145]
[0.04116645]
[0.01039399]
[0.00031857]
[0.36967722]
[0.00783529]
[2.0190277]
[0.2970339]
[0.00826326]
[1.797534]
[0.01799789]
[2.1863558]
[0.04116645]
[0.00834341]
[0.0296402]
[0.00232847]
[1.1546179]
[0.00291074]
[0.0912188]
[0.01020076]
[0.5220395]
[2.3313358]
[0.2368005]
[0.00020956]
[0.00783529]
[0.00971962]
[0.4719917]
[0.01101481]
[0.01848922]
[0.03530572]
[0.01665684]
[0.06094277]

In [30]:
predictions1 = model.predict(x = test_samples, batch_size = 9, verbose = 2)

606/606 - 1s - 924ms/epoch - 2ms/step


In [31]:
for i in predictions1:
    print(i)

[0.01892437]
[0.6671372]
[0.695339]
[2.1517923]
[0.09080283]
[0.6671372]
[0.03861909]
[0.01580121]
[0.01786645]
[0.01795568]
[0.00394699]
[0.01684953]
[0.0129846]
[1.1865592]
[2.1059268]
[0.00765771]
[0.02378303]
[0.7866475]
[0.00045969]
[0.00755937]
[0.594896]
[0.02474697]
[1.5778986]
[0.6671372]
[0.01278123]
[0.02422648]
[0.08459212]
[0.0188338]
[0.13374595]
[0.80234146]
[1.3123852]
[0.24224715]
[0.6671372]
[0.667137]
[0.45761693]
[0.18842866]
[0.0180933]
[0.01498635]
[0.01339163]
[0.01315824]
[0.02003261]
[1.2297165]
[0.01521973]
[0.01127698]
[0.01852612]
[0.4646796]
[0.01933135]
[0.01081284]
[0.00685389]
[2.0204618]
[1.7945302]
[0.02490101]
[0.25384024]
[0.01599325]
[0.01541533]
[0.6671372]
[0.8331163]
[0.02461231]
[1.0742639]
[0.7331433]
[0.667137]
[0.01280232]
[0.00725206]
[0.6671372]
[0.01478095]
[1.8113445]
[0.01720864]
[0.6388911]
[0.2722636]
[0.02993785]
[1.247199]
[0.008021]
[0.98611647]
[0.51090777]
[0.35847914]
[0.5289562]
[0.00918056]
[0.12899682]
[0.12385477]
[0.667137]


[1.329258]
[0.9461379]
[0.6671372]
[0.667137]
[0.667137]
[0.7105299]
[0.01599179]
[0.01828773]
[0.01099934]
[0.64464676]
[0.00667547]
[1.5553567]
[2.1746104]
[0.667137]
[1.3085591]
[0.23615302]
[1.4976082]
[2.2603037]
[0.35721725]
[0.00667547]
[0.05419037]
[0.667137]
[0.02245539]
[0.11144377]
[0.0018207]
[0.6671372]
[0.71053004]
[0.01214076]
[0.01003673]
[0.6671372]
[0.00662014]
[0.5386949]
[0.09662122]
[0.81523687]
[0.01584198]
[0.01193459]
[2.0227916]
[0.17021887]
[0.89854383]
[0.00646964]
[0.01731642]
[0.7684936]
[0.02028663]
[0.01660509]
[2.310238]
[0.01439521]
[2.3008964]
[0.6671372]
[0.15084429]
[0.01845427]
[0.09562989]
[0.6671372]
[0.00394699]
[0.01781595]
[0.00798636]
[0.01746686]
[2.4031217]
[0.01655376]
[1.7011187]
[0.00484652]
[0.01550491]
[0.01749732]
[0.01006766]
[0.29931116]
[0.33174235]
[0.6671372]
[0.00648937]
[0.01735775]
[0.01659681]
[0.01712057]
[0.01014055]
[0.01900063]
[0.07252503]
[0.0065978]
[1.5129092]
[0.0067648]
[0.12184235]
[0.01623894]
[-0.00861429]
[2.3289

#### The same issue still persists. However, the good thing is that, in this case, it isn't as prominent. For the samples where the model didn't break, the predictions were quite close to the actual labels.

In [32]:
result2 = model.evaluate(train_samples, train_labels, batch_size = 9, verbose = 2)

1818/1818 - 3s - loss: 0.0911 - mean_absolute_error: 0.1169 - 3s/epoch - 1ms/step


In [33]:
for i in train_labels:
    print(i)

[0.01019546]
[0.01145698]
[0.02082768]
[0.00108919]
[1.1681184]
[1.2823576]
[0.01148193]
[0.23223004]
[1.797534]
[1.9399297]
[0.00986996]
[0.00755988]
[0.01143156]
[0.22769284]
[2.0044596]
[0.53666645]
[0.01573243]
[1.2823576]
[0.53666645]
[0.5693898]
[2.3903913]
[0.01216508]
[1.5833225]
[0.01327534]
[0.90961415]
[0.01326366]
[0.56169754]
[0.01113226]
[0.00028531]
[1.4843951]
[0.0151009]
[0.00079148]
[0.01671958]
[0.01596123]
[0.0098527]
[0.01119066]
[1.1681184]
[0.01388588]
[0.01045231]
[0.01686158]
[2.1449127]
[0.22769284]
[0.00916766]
[0.01137227]
[0.01015438]
[0.0082221]
[0.01333897]
[0.2754944]
[0.01151844]
[0.00068265]
[0.01438254]
[0.16935252]
[0.01832396]
[0.00830763]
[0.00932941]
[1.0974257]
[0.01115093]
[2.3940372]
[1.296788]
[2.3940372]
[1.7289168]
[0.2368005]
[0.64387584]
[2.1319735]
[0.02611536]
[0.01680435]
[0.90961415]
[0.42449966]
[0.01754049]
[0.01746808]
[0.09879936]
[0.00023146]
[0.01605972]
[0.76748747]
[0.06072289]
[0.00877016]
[0.2511465]
[0.01470389]
[0.2804023]


[0.04424114]
[2.3903913]
[0.00861853]
[1.0845044]
[0.0151216]
[0.00926671]
[0.04266924]
[0.01470389]
[2.2948706]
[0.01422596]
[0.00920994]
[0.01071297]
[0.42449966]
[0.53666645]
[0.0121328]
[0.00965373]
[0.01799789]
[0.6905713]
[1.4998894]
[0.00812545]
[0.00895529]
[0.00901367]
[0.00731953]
[2.3388424]
[2.345992]
[0.25531197]
[0.01850504]
[0.01218946]
[0.00746992]
[0.00023146]
[0.01081587]
[0.27062622]
[0.23223004]
[0.4719917]
[0.0644175]
[0.27062622]
[2.3038528]
[0.00881502]
[0.00900563]
[0.00941836]
[0.0096095]
[2.2099628]
[0.17350438]
[0.01019228]
[0.09500241]
[0.00849895]
[0.01706413]
[0.00025651]
[0.06072289]
[0.9209225]
[0.01146306]
[0.00232847]
[0.04482179]
[1.5675682]
[1.5675682]
[0.02014047]
[0.00476336]
[0.01651765]
[0.42449966]
[2.4051573]
[0.01078688]
[0.02192513]
[2.1863558]
[0.00938719]
[0.01319191]
[2.3903913]
[0.00051807]
[0.00155572]
[0.0085233]
[0.3454205]
[0.01145698]
[0.3454205]
[0.93234134]
[0.00020956]
[0.03447764]
[0.00827004]
[0.37542164]
[0.04493235]
[0.0762582

[0.01019758]
[0.01703686]
[0.09500241]
[0.20842052]
[0.465303]
[0.0194248]
[0.04558423]
[0.00947399]
[0.01756211]
[0.5693898]
[0.21282145]
[1.2823576]
[0.00031857]
[0.06072289]
[0.02061418]
[0.01244929]
[0.01154912]
[0.01136786]
[1.2680225]
[0.01145698]
[0.0101112]
[0.21282145]
[0.00045534]
[0.01391947]
[0.01074818]
[1.296788]
[2.3940372]
[0.00025651]
[0.00983886]
[0.01397236]
[0.03376279]
[0.2804023]
[0.00970817]
[0.20842052]
[0.00840062]
[2.3388424]
[0.4586818]
[0.2970339]
[0.23223004]
[0.2511465]
[0.0097302]
[0.01441837]
[2.1319735]
[0.14120427]
[0.43711677]
[0.00045534]
[0.01145698]
[0.01188772]
[0.01287539]
[0.2511465]
[0.2804023]
[0.01154066]
[0.01488028]
[1.1412246]
[0.9830488]
[0.0107327]
[0.00747994]
[0.02728391]
[0.00887208]
[0.00794446]
[1.1546179]
[0.43711677]
[0.03466355]
[0.63540184]
[0.03660647]
[0.11154062]
[1.7130169]
[0.00855976]
[0.1476193]
[2.0044596]
[0.67273617]
[0.5220395]
[0.01668463]
[1.813304]
[0.02728391]
[0.20842052]
[0.3454205]
[0.00943848]
[0.0063204]
[0.0

[0.00025651]
[0.00023146]
[2.345992]
[0.16935252]
[0.01362184]
[1.7289168]
[0.0167925]
[0.00108919]
[2.3903913]
[0.01526362]
[2.345992]
[2.3863473]
[0.01296925]
[1.296788]
[2.3903913]
[2.3313358]
[1.7130169]
[0.02415255]
[0.0164616]
[0.00809345]
[0.38553214]
[1.296788]
[0.42449966]
[0.05703677]
[0.01145698]
[0.02404062]
[0.01746009]
[2.3313358]
[0.67273617]
[0.01642603]
[0.04482179]
[0.00783529]
[0.01968271]
[0.42449966]
[0.53666645]
[0.00931483]
[0.01550947]
[0.00789061]
[0.04206101]
[0.00739924]
[0.2804023]
[0.00889416]
[0.01280343]
[0.04424114]
[0.42449966]
[0.01588052]
[0.13862154]
[0.00232847]
[0.01168393]
[1.0845044]
[0.00480243]
[1.2680225]
[2.1449127]
[0.03185861]
[0.01693881]
[0.01054723]
[0.36967722]
[0.00868736]
[0.01854956]
[0.13862154]
[0.00045534]
[1.7817262]
[0.05614346]
[0.22769284]
[0.01696561]
[0.00023146]
[0.56169754]
[0.00751556]
[0.2804023]
[0.01112399]
[0.00028531]
[0.01351691]
[0.4719917]
[0.0095432]
[0.01439594]
[0.00783529]
[1.2823576]
[0.35093856]
[0.2804023]


[1.813304]
[0.01088872]
[0.00020956]
[0.2511465]
[1.0845044]
[0.01532443]
[1.5675682]
[0.01571494]
[0.0912188]
[0.37542164]
[0.01283185]
[0.001295]
[0.01481969]
[0.37542164]
[0.01026505]
[0.00155572]
[0.00934202]
[0.01621734]
[0.00783529]
[0.01128432]
[1.296788]
[0.00092462]
[0.00476336]
[0.01620243]
[1.4843951]
[0.01777997]
[0.21282145]
[0.56169754]
[2.3863473]
[0.00291074]
[0.02598279]
[0.22769284]
[1.1546179]
[0.5220395]
[0.01501216]
[0.01122233]
[0.04998438]
[1.4843951]
[1.3603066]
[2.3313358]
[0.01107749]
[0.0110585]
[0.07759248]
[0.04424114]
[0.00982602]
[0.0128541]
[0.00859955]
[2.1863558]
[0.13862154]
[2.1449127]
[0.00031857]
[1.1412246]
[0.00025651]
[0.01391947]
[0.01145698]
[0.03399611]
[1.1412246]
[2.1449127]
[0.01799789]
[0.43077692]
[0.2368005]
[0.04380029]
[0.01277727]
[0.00783303]
[0.16522466]
[0.00887313]
[0.00876207]
[0.42449966]
[0.00874196]
[0.01533885]
[0.0102098]
[0.00919903]
[0.01596733]
[1.9399297]
[0.00232847]
[1.7289168]
[0.02104797]
[0.00092462]
[0.00935059]
[

In [34]:
predictions2 = model.predict(x = train_samples, batch_size = 9, verbose = 2)

1818/1818 - 2s - 2s/epoch - 1ms/step


In [35]:
for i in predictions2:
    print(i)

[0.01813461]
[0.6671372]
[0.01898479]
[0.00650485]
[1.1686699]
[1.2653862]
[0.01358399]
[0.31883222]
[1.7774473]
[0.6671372]
[0.01624514]
[0.00216205]
[0.01471979]
[0.14458273]
[2.0204618]
[0.5224756]
[0.02055074]
[1.3085591]
[0.6082568]
[0.6185289]
[2.3051028]
[0.01609033]
[1.5553567]
[0.6671372]
[0.667137]
[0.01933686]
[0.60239446]
[0.00910677]
[0.01291396]
[1.5506923]
[0.02117816]
[0.0115758]
[0.6671372]
[0.02254149]
[0.01450987]
[0.01758625]
[0.6671372]
[0.01639594]
[0.01145281]
[0.02423647]
[2.1603851]
[0.2722636]
[0.00460313]
[0.01631679]
[0.01726641]
[-0.00391637]
[0.01618786]
[0.42730957]
[0.01483263]
[0.01181125]
[0.6671372]
[0.21708487]
[0.0215676]
[0.01604058]
[-0.00043915]
[0.6671372]
[0.01228566]
[2.4031217]
[1.3123852]
[2.4031217]
[0.667137]
[0.2769184]
[0.68662226]
[0.6671372]
[0.01880567]
[0.01768847]
[0.9268933]
[0.4166677]
[0.6671372]
[0.02468731]
[0.12361115]
[0.00972649]
[0.01926298]
[0.6671372]
[0.15729193]
[0.00159221]
[0.1311034]
[0.6671372]
[0.26949412]
[0.12333

[0.5234991]
[0.3928823]
[2.3545191]
[1.7013876]
[0.2645082]
[0.667137]
[0.01908626]
[0.12527522]
[0.6671372]
[0.5234991]
[0.6023944]
[1.7216657]
[2.4236856]
[0.7778124]
[0.12447524]
[0.01762776]
[0.0094634]
[0.01892878]
[0.01552482]
[0.00909291]
[0.01638943]
[0.11980398]
[0.12027683]
[0.01647169]
[0.667137]
[0.97065175]
[0.01064886]
[0.01460983]
[0.80234146]
[0.0096636]
[1.3761396]
[2.4201334]
[2.197057]
[0.02171156]
[0.00725206]
[0.6691678]
[0.01839831]
[0.2396243]
[0.00992428]
[0.25887436]
[0.6671372]
[0.00875114]
[1.1374617]
[0.00806647]
[1.6054568]
[1.344089]
[0.01302533]
[0.67569786]
[0.1320942]
[0.6671372]
[0.00473384]
[0.00765695]
[0.97065175]
[0.00024237]
[1.1611673]
[0.6023944]
[0.02664755]
[1.1058027]
[0.6671372]
[-0.00910335]
[0.01302828]
[0.0094634]
[0.00686956]
[0.00400401]
[0.18120873]
[-0.00219496]
[0.12448098]
[0.6671372]
[0.01976855]
[0.01764025]
[2.1674364]
[0.0040033]
[0.01894678]
[0.01140897]
[0.02190911]
[0.00563051]
[0.6671372]
[0.667137]
[0.667137]
[0.01753922]
[

[0.02103613]
[0.01841344]
[0.0062557]
[0.6671372]
[0.01437412]
[0.02662781]
[0.00777808]
[0.6671372]
[0.00900987]
[0.02489001]
[0.11231434]
[0.9444894]
[0.01860106]
[0.5157167]
[0.2805019]
[0.300947]
[0.6671372]
[0.01701578]
[0.268032]
[0.0169598]
[0.0117485]
[0.01626535]
[2.254025]
[0.50687945]
[2.3440387]
[0.6671372]
[2.1746104]
[0.01786523]
[0.01653101]
[0.4804386]
[0.6671372]
[0.01321171]
[0.0281699]
[2.3440387]
[0.1540313]
[0.07252502]
[0.12211083]
[1.7925458]
[0.01191254]
[0.00652916]
[0.01942517]
[0.0165737]
[0.00867237]
[0.6671372]
[0.13085534]
[0.12410627]
[0.01836677]
[0.6671372]
[0.01754786]
[0.71053004]
[0.6528299]
[0.13508539]
[0.6671372]
[0.00648937]
[0.01773815]
[0.01761134]
[0.00961783]
[0.01585783]
[0.01712887]
[0.65043175]
[0.01038952]
[0.6671372]
[0.00661168]
[0.01778378]
[0.01998676]
[0.6671372]
[0.6671372]
[0.0096703]
[1.329258]
[0.17021887]
[0.6671372]
[0.667137]
[0.0065699]
[0.01838233]
[0.01129519]
[0.01548488]
[0.00964291]
[0.00561139]
[1.1058027]
[0.10129458]


[1.9356704]
[0.01719518]
[0.01947145]
[0.6671372]
[0.01398145]
[0.01295647]
[0.00699426]
[0.6671372]
[0.02474696]
[0.02023803]
[0.5817946]
[0.03300346]
[0.52614975]
[0.00810249]
[2.3220494]
[0.01082607]
[0.6671372]
[0.667137]
[0.667137]
[0.00392541]
[0.01177157]
[0.01708774]
[0.01471421]
[2.335851]
[0.00594951]
[0.6671372]
[0.07829981]
[0.667137]
[0.0170541]
[2.141039]
[0.01488896]
[0.00573041]
[0.01798037]
[0.17021887]
[0.23408006]
[0.05787201]
[0.12410627]
[0.0173756]
[0.00686956]
[0.01759703]
[0.18790886]
[-0.01062979]
[0.01745625]
[0.6671372]
[0.0051052]
[0.01100238]
[0.11848921]
[0.6671372]
[0.6671372]
[0.01684994]
[1.5553567]
[0.14585374]
[0.6671372]
[0.01663512]
[0.7866475]
[0.9517225]
[0.01005037]
[0.51090777]
[2.0275652]
[0.2805019]
[0.00917138]
[0.6671372]
[0.02715342]
[0.0069268]
[0.01771761]
[0.81523687]
[0.01582338]
[0.01325323]
[0.03031768]
[0.4599551]
[2.335851]
[0.50547355]
[0.01521601]
[0.01817395]
[0.01230432]
[0.00529314]
[0.0160116]
[0.00394188]
[0.20406403]
[2.1603

#### Same behavior of the model as like that for test set.

### Even though the problem of getting same prediction for different samples showed up with this architecture too, its performance still looked promising, as the frequency of similar predictions was low and the difference with the actual labels was almost negligent for the predictions without the problem. However, the way we set up our architecture with the mask, it required a batch size of 9 and 9 only to work without any runtime error, which indicated that our mask was probably not working the way we intended it to, masking off particular samples from the batch for each of the nodes instead of masking specific attributes of each sample. So next, we decide to figure out a way to make sure that the mask gets applied specifically on each of the samples, either separately during data preprocessing or within the architecture during training as like filters in CNN architectures. Unfortunately, my Master's program came to a close and so, I couldn't continue the research.

### Now, if this research is to be continued, there are a number of ways this can be accomplished. Obviously, the simplest way is to keep continuing the same flow of work: try to find a way of applying the mask on each sample and then see if the results are fruitful.<br>However, we have been continuously struggling to make our models work following this workflow: using this dataset, whatever architecture we use, all would eventually break into predicting similar value. It could be checked whether changing the dataset helps in solving that issue. If we look back at our previous experiments, we will see that working with univariate polynomial samples was not suitable for our research, as these samples had low curvature values. So, we eventually started using a combination dataset, where we had a higher percentage of samples with high-valued curvatures. After trying out different architectures of regular MLP, we settled on an architecture with a depth of 20 layers and a width of 40 nodes. However, using this architecture, we experimented on a combination dataset that had a balanced number of high and low-valued curvatures. Thereafter, we couldn't make our models work even after using complex architectures like CNNs. Hence, as a continuation of this research, what could be done next is conducting experiments using the regular MLP architecture just mentioned on a combination dataset where the percentage of samples with high-valued curvatures remains more as like the earlier experiments we performed and see if suitable results can be obtained. For these experiments, the random_state parameter of the train_test_split function could also be changed to some other values than the ones used so far, just to make sure that the earlier used combination of dataset split do not cause the model to break again.<br>In case this experiment setup still fail to perform on the first combination dataset, like it did for us in our experiments when we used this architecture, we could try changing the order in which the experiments are performed on the 3 different combination datasets. Instead of first conducting the experiment on the first dataset, we could start with either the 2nd or 3rd one, move on to the next datasets, and see if there's any improvement in performance.<br>One thing to be noted is that in our later experiments, even after discarding samples with very low-valued curvatures, we still faced the same problem where our model predicted single curvature value, given we still had some univariate polynomial samples with lower-valued curvatures above a certain threshold. So, the current experiment setup mentioned to be used as a continuation of our research could also fail ultimately like our previous experiments. Since we had discovered that our models perform excellently with high curvature values, the final and easier way to get an outcome from our research could be to entirely discard the univariate polynomial expressions and any other expressions with low-valued curvatures, and work on expressions that give higher curvature values like the ones we used. If it is found that the architecture used performs with an acceptable accuracy after being trained on enough samples of a number of different expressions with high curvatures (of which there is a high chance), we can finalize that neural network as the result of our experiment which predicts maximum-valued curvature from specific types of function values with restrictions on certain types of functions that have low curvatures. Once this research is completed, we can move on to designing another neural network in a similar way that will predict the minimum-valued curvature for the same type of functions, since we had been working on the maximum-valued curvatures so far. Finally, we can conduct the research of developing a classification model that will predict the sign of the curvatures estimated, as we have been working on unsigned curvatures thus far.